# NK13 - Analysis I - scVI figures
- Load modules and data
- Plot 1: UMAP with scANVI labels from CD8 umap annotations
- Plot 2: Bargraphs per L3_ann_overview
- Plot 3: UMAP of virus-specific clones
- Plot 4: Report CSF and PB-biased scores
- Plots X: Report individual markers

In [ ]:
import anndata as ad
import scanpy as sc
import scirpy as ir
import pandas as pd
import numpy as np
import seaborn as sb
import muon as mu
import scvi
import torch

# from scipy.stats import binomtest
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests
from scipy.stats import wilcoxon
import scipy.stats as stats
from scipy.stats import zscore
from sklearn.linear_model import LogisticRegression
# import string

# Helps to define plotting area

import matplotlib.pyplot as plt
import seaborn.objects as so
from scipy.stats import kruskal
import scikit_posthocs as sp
from skbio.diversity.alpha import gini_index, shannon, simpson
# from matplotlib import rcParams 
# from matplotlib.pyplot import rc_context
# import matplotlib.gridspec as gridspec
# import matplotlib.colors as mcolors
# from matplotlib import colors
# # from matplotlib_venn import venn2
# from scipy.stats import spearmanr
# import matplotlib.patches as mpatches
# from pathlib import Path

## For making text editable in .pdfs once exported use cairo backend instead of agg
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42

# set seed
np.random.seed(44)

#Set seaborn theme
sb.set_style("ticks")
# custom_params = {"axes.spines.right": False, "axes.spines.top": False}
# sb.set_theme(style="ticks", rc=custom_params)

#display options
pd.options.display.max_rows = 999
pd.options.display.max_columns = 999

In [ ]:
scvi.settings.seed = 0
print("Last run with scvi-tools version:", scvi.__version__)

In [ ]:
torch.set_float32_matmul_precision("high")

# Colors

In [ ]:
cluster_color_dict = {

    # -------------------- TN --------------------
    "0  TN (IFNlo BACH2int)":  "#dbe9f6",
    "1  TN (IFNint BACH2int)": "#fde0c5",
    "2  TN (IFNhi BACH2hi)":   "#e2d9f3",
    "6  TN (IFNlo BACH2hi)":   "#D7BCA6",

    # -------------------- TCM --------------------
    "11 TCM (IFNhi)": "#74c476",

    # -------------------- TSCM --------------------
    "15 TSCM (IFNhi)": "#a1d99b",

    # -------------------- TEM --------------------
    "5  TEM (IFNlo)": "#3182bd",
    "8  TEM (IFNhi)": "#003FFF",

    # -------------------- TEM/TCM --------------------
    "7  TEM/TCM (IFNint)": "#9ecae1",

    # -------------------- TE --------------------
    "13 TE (IFNlo)": "#756bb1",
    "9  TE (TYROBPhi KIRhi)": "#C00000",
    "10 TE (GNLYhi)": "#843c39",
    "12 TE (CX3CR1hi IFNhi)": "#9c9ede",
    "14 TE (EOMEShi IFNhi)": "#54278f",

    # -------------------- TCSF --------------------
    "3  TCSF (GZMKhi HLA-DRhi)": "#d6616b",
    "4  TCSF (CD69hi IL7Rhi)": "#fdae6b",
    "22 TCSF (GZMKint ITGA4hi)": "#fd8d3c",
    "19 TCSF (CXCR6hi XCL1hi)": "#8b0000",
    "16 TCSF (TCF7hi ITGA4int)": "#637939",

    # -------------------- TA --------------------
    "17 TA (TCF7hi NFKBhi)": "#f768a1",
    "20 TA (CD69hi TGFB1hi)": "#c51b8a",
    "21 TA (NR4A1hi JUN/FOShi)": "#F4E3ED",
    "23 TA (BATF3hi CXCR6hi)": "#FF1B76",   
    "25 TA (CD3Elo Ribolo)": "#C7C7C7",

    # -------------------- TCF7hi IKZF2hi --------------------
    "24 TCF7hi IKZF2hi": "#e2d9f3",

    # -------------------- Cytotoxic gamma delta --------------------
    "18 Cytotoxic gamma delta": "#8c6d31",
}

# Loading data multimer data

In [ ]:
mdata = mu.read(filename="../../04_data_objects/06_preprocessed_v2/2026_6_multimer_dataset_preprocessed_annotated_intgr.h5mu")
len(mdata)

In [ ]:
#Group annotation
patient_list = [ "Pt11", "Pt12", "Pt14", "Pt17", "Pt20"]
control_list = ["Pt1","Pt8", "Pt18", "Pt19", "Pt21"]

#Match the lists defined above to define "RRMS" and "Control"
mdata["gex"].obs["catpat"] = pd.Categorical(np.where(mdata["gex"].obs["donor"].isin(patient_list), "MS", np.where(mdata["gex"].obs["donor"].isin(control_list), "NIC", "IC")))


# How many epitope-specific T cells did we annotate with multimers?

In [ ]:
# How many cells in total?
mdata["gex"].obs["Virus"].value_counts().sum()

In [ ]:
# How many clonotypes in total
mdata["gex"].obs[["clone_handle", "Virus"]].drop_duplicates().value_counts().sum()

In [ ]:
# How many antigen-specific responses in total
mdata["gex"].obs[["handle", "donor"]].drop_duplicates().value_counts().sum()

In [ ]:
# How many antigen-specific responses in total
mdata["gex"].obs[["handle"]].drop_duplicates().value_counts().sum()

# Plot 1: UMAP with scANVI labels from CD8 umap annotations

In [ ]:
#Make a dictionary for annotations using predicted labels from scANVI to annotate your multimer data
anvi_ann = pd.read_csv("../../03_annotation/11_multimer_experiment_ann/1_LabelTransferObs_from_scANVI.csv")
anvi_dict = anvi_ann.set_index("Unnamed: 0").to_dict()

obs_df = mdata["gex"].obs.copy()

## map the dictionary unto the sc-seq object
mdata["gex"].obs["C_scANVI"] = mdata["gex"].obs.index.map(anvi_dict["C_scANVI"])
mdata["gex"].obs["L3_ann_overview_scanvi"] = mdata["gex"].obs.index.map(anvi_dict["L3_ann_overview_scanvi"])

## Convert to category
mdata["gex"].obs["L3_ann_overview_scanvi"] = (
    mdata["gex"].obs["L3_ann_overview_scanvi"].astype("category")
)

mdata["gex"].obs[["C_scANVI", "L3_ann_overview_scanvi"]].dtypes

In [ ]:
mdata["gex"].obs["L3_ann_overview_scanvi"].value_counts()

In [ ]:
color_dict = {
    "TA": "#D24FA5",
    "TCM": "#9ECAE1",
    "TE": "#C00000",
    "TEM": "#3182BD",
    "TEMRA": "#4C3696", 
    "GDT cells": "#8C6D31",
    "TN" : "#D7BCA6", 
    "NK-like" : "#279E68"
}

cats = mdata["gex"].obs["L3_ann_overview_scanvi"].cat.categories

mdata["gex"].uns["L3_ann_overview_scanvi_colors"] = [
    color_dict[c] for c in cats
]

sc.settings.set_figure_params(figsize=(5, 4))
fig = sc.pl.umap(mdata["gex"], color="L3_ann_overview_scanvi", legend_fontweight= 'bold', frameon= False , size=25, vmin=0, legend_fontoutline=2, return_fig=True, show=False)
plt.tight_layout()
fig.savefig("../../05_plots/09_multimers/14_scANVI_OverviewLabels_on_multimer_intergration.pdf",  format="pdf", dpi=400)
plt.show

# Plot 2: Bargraphs per L3_ann_overview

In [ ]:
colors = {
    "GDT cells" : "#8C6D31", 
    "NK-like" : "#279E68",
    "TA" : "#D24FA5",
    "TN" : "#D7BCA6", 
    "TCM": "#9ECAE1", 
    "TEM" : "#3182BD", 
    "TEMRA" : "#4C3696", 
    "TE" : "#C00000"
}

In [ ]:
obs_df = mdata["gex"].obs.copy()
# obs_df = obs_df[~obs_df["donor"].isin(["Pt4", "Pt5"])]

## dropna
obs_df = obs_df.dropna(subset="Virus")

# Remove unused categories
for col in obs_df.select_dtypes(include="category").columns:
    obs_df[col] = (
        obs_df[col].cat.remove_unused_categories()
    )

#Select columns
obs_df = obs_df[["clone_handle", "Virus", "L3_ann_overview_scanvi"]]

#Count cells
count_df = obs_df.groupby(["Virus", "L3_ann_overview_scanvi"]).size().reset_index(name="count")

#sum 
count_df["total_cells"] = count_df.groupby("Virus")["count"].transform("sum")

#percentage
count_df["percent_cells"] = 100*count_df["count"] / count_df["total_cells"]

# Pivot to wide format
plot_df = (
    count_df.pivot(
        index="Virus",
        columns="L3_ann_overview_scanvi",
        values="percent_cells"
    )
    .fillna(0)
)

# Optional: specify virus order
virus_order = ["CMV", "NWV", "HMPV", "FLU-A", "HHV-1", "SARS-COV-2", "HAdV-C", "EBV", "HHV-2", "VZV"]
plot_df = plot_df.reindex([v for v in virus_order if v in plot_df.index])

#Stacked bargraphs
plot_df = plot_df[[c for c in cats if c in plot_df.columns]]

fig, ax = plt.subplots(figsize=(6, 3))

plot_df.plot(
    kind="bar",
    stacked=True,
    color=[colors[c] for c in plot_df.columns],
    width=0.8,
    edgecolor="none",
    ax=ax,
)

# Axis formatting
ax.set_xlabel("")
ax.set_ylabel("% of cells")
ax.set_ylim(0, 100)
sb.despine()
ax.tick_params(axis="x", rotation=90)
ax.grid(False)

# Legend
ax.legend(
    title="Cell state",
    title_fontsize=10,
    fontsize=9,
    frameon=False,
    ncol=1,                 # Increase if many cell states
    loc="upper left",
    bbox_to_anchor=(1.02, 1),
    borderaxespad=0,
    handlelength=1.2,
    handleheight=0.8,
    labelspacing=0.4,
    columnspacing=1.2,
)

plt.tight_layout()
plt.savefig("../../05_plots/09_multimers/39_Preferred_pheno_L3overview.pdf", format="pdf", dpi=400)
plt.show()

In [ ]:
#Count unique clones
obs_df[["clone_handle", "Virus"]].drop_duplicates()[["Virus"]].value_counts().reset_index()


In [ ]:
#Count cells
obs_df[["Virus"]].value_counts().reset_index()


# Plot 3: UMAP of virus-specific clones

In [ ]:
sc.settings.set_figure_params(figsize=(4, 4))
fig = sc.pl.umap(mdata["gex"], color="Virus", groups=["EBV"], legend_fontweight= 'bold', legend_loc = "on data", frameon= False , size=40, vmin=0, legend_fontoutline=2, return_fig=True, show=False)
plt.tight_layout()
fig.savefig("../../05_plots/09_multimers/6_EBV_specificity_umap.pdf",  format="pdf", dpi=400)
plt.show

In [ ]:
sc.settings.set_figure_params(figsize=(4, 4))
fig = sc.pl.umap(mdata["gex"], color="Virus", groups=["CMV"], legend_fontweight= 'bold', legend_loc = "on data", frameon= False , size=40, vmin=0, legend_fontoutline=2, return_fig=True, show=False)
plt.tight_layout()
fig.savefig("../../05_plots/09_multimers/6_CMV_specificity_umap.pdf",  format="pdf", dpi=400)
plt.show

In [ ]:
sc.settings.set_figure_params(figsize=(4, 4))
fig = sc.pl.umap(mdata["gex"], color="Virus", groups=["FLU-A", "HHV-6B"], legend_fontweight= 'bold', legend_loc = "on data", frameon= False , size=80, vmin=0, legend_fontoutline=2, return_fig=True, show=False)
plt.tight_layout()
fig.savefig("../../05_plots/09_multimers/6_FLU_specificity_umap.pdf",  format="pdf", dpi=400)
plt.show

In [ ]:
sc.settings.set_figure_params(figsize=(4, 4))
fig = sc.pl.umap(mdata["gex"], color="Virus", groups=["HAdV-C", "HHV-1", "HHV-2", "HMPV"], legend_loc = "on data", legend_fontweight= 'bold', frameon= False , size=80, vmin=0, legend_fontoutline=2, return_fig=True, show=False)
plt.tight_layout()
fig.savefig("../../05_plots/09_multimers/6_Other1_specificity_umap.pdf",  format="pdf", dpi=400)
plt.show

In [ ]:
sc.settings.set_figure_params(figsize=(4, 4))
fig = sc.pl.umap(mdata["gex"], color="Virus", groups=["NWV", "SARS-COV-2", "VZV"], legend_loc = "on data", legend_fontweight= 'bold', frameon= False , size=80, vmin=0, legend_fontoutline=2, return_fig=True, show=False)
plt.tight_layout()
fig.savefig("../../05_plots/09_multimers/6_Other2_specificity_umap.pdf",  format="pdf", dpi=400)
plt.show

# Plot 4: Report CSF and PB-biased scores

In [ ]:
sc.settings.set_figure_params(figsize=(4, 4))
fig = sc.pl.umap(mdata["gex"], color="CSF-biased score",  legend_fontweight= 'bold', legend_loc = "on data", cmap="RdBu_r", frameon= False , size=40, vmin=0, legend_fontoutline=2, return_fig=True, show=False)
plt.tight_layout()
fig.savefig("../../05_plots/09_multimers/30_CSFbiased_umap.pdf",  format="pdf", dpi=400)
plt.show

In [ ]:
sc.settings.set_figure_params(figsize=(4, 4))
fig = sc.pl.umap(mdata["gex"], color="PB-biased score",  legend_fontweight= 'bold', legend_loc = "on data", cmap="RdBu_r", frameon= False , size=40, vmin=0, legend_fontoutline=2, return_fig=True, show=False)
plt.tight_layout()
fig.savefig("../../05_plots/09_multimers/30_PBbiased_umap.pdf",  format="pdf", dpi=400)
plt.show

# Plot 5: CSF and PB-biased scores per clone

## CSF-biased score per clone

In [ ]:
## Extract score values per cell and plot as a box plot per specificity
obs_df = mdata["gex"].obs[["Virus", "CSF-biased score", "clone_handle"]].copy()
obs_df = obs_df.dropna(subset="Virus")
obs_df = obs_df.groupby(["clone_handle", "Virus"], observed=True).mean("CSF-biased score").reset_index()

#plotting order: 
order = (
    obs_df
    .groupby("Virus")["CSF-biased score"]
    .median()
    .sort_values()
    .index
)

# Create the figure object
fig, ax = plt.subplots(figsize=(4,4))

# Stripplot (background dots)
sb.stripplot(
    data=obs_df,
    x="Virus",
    y="CSF-biased score",
    color="black",
    jitter=0.25,
    size=5,
    zorder=1,
    order=order,
    ax=ax
)

# Boxplot (foreground)
sb.boxplot(
    data=obs_df,
    x="Virus",
    y="CSF-biased score",
    order=order,
    fliersize=0,
    width=0.5,
    color="white",
    linecolor="#4D4D4D", 
    linewidth=0.5,
    zorder=2,
    ax=ax
)

plt.ylabel("Mean CSF-biased score\nper clone")
plt.xlabel("Virus")
plt.xticks(rotation=90)
sb.despine()

plt.tight_layout()
fig.savefig("../../05_plots/09_multimers/7_mean_clone_CSF_biased_score.pdf", bbox_inches='tight')
plt.show()

In [ ]:
import scipy.stats as stats
import scikit_posthocs as sp

## Extract score values per cell and plot as a box plot per specificity
obs_df = mdata["gex"].obs[["Virus", "CSF-biased score", "clone_handle"]].copy()
obs_df = obs_df.dropna(subset="Virus")
obs_df = obs_df.groupby(["clone_handle", "Virus"], observed=True).mean("PB-biased score").reset_index()

groups = [
    group["CSF-biased score"].values
    for _, group in obs_df.groupby("Virus", observed=True)
]

H, p = stats.kruskal(*groups)

print(f"Kruskal-Wallis H = {H:.3f}, p = {p:.3e}")

In [ ]:
dunn = sp.posthoc_dunn(
    obs_df,
    val_col="CSF-biased score",
    group_col="Virus",
    p_adjust="fdr_bh",   # or "holm", "bonferroni"
)

dunn = dunn.loc[virus_order, virus_order]

dunn_long = (
    dunn.stack()
    .reset_index()
    .rename(columns={
        "level_0": "Virus_1",
        "level_1": "Virus_2",
        0: "adjusted_p"
    })
)

# Remove self-comparisons
dunn_long = dunn_long[dunn_long["Virus_1"] != dunn_long["Virus_2"]]

# Remove duplicate comparisons
dunn_long["comparison"] = dunn_long.apply(
    lambda x: tuple(sorted((x["Virus_1"], x["Virus_2"]))),
    axis=1,
)

dunn_long = (
    dunn_long
    .drop_duplicates("comparison")
    .drop(columns="comparison")
    .sort_values("adjusted_p")
)

dunn_long.head(10)

## PB-biased score per clone

In [ ]:
## Extract score values per cell and plot as a box plot per specificity
obs_df = mdata["gex"].obs[["Virus", "PB-biased score", "clone_handle"]].copy()
obs_df = obs_df.dropna(subset="Virus")
obs_df = obs_df.groupby(["clone_handle", "Virus"], observed=True).mean("PB-biased score").reset_index()

# Create the figure object
fig, ax = plt.subplots(figsize=(4,4))

# Stripplot (background dots)
sb.stripplot(
    data=obs_df,
    x="Virus",
    y="PB-biased score",
    color="black",
    jitter=0.25,
    size=5,
    zorder=1,
    order=order,
    ax=ax
)

# Boxplot (foreground)
sb.boxplot(
    data=obs_df,
    x="Virus",
    y="PB-biased score",
    order=order,
    fliersize=0,
    width=0.5,
    color="white",
    linecolor="#4D4D4D", 
    linewidth=0.5,
    zorder=2,
    ax=ax
)

plt.ylabel("Mean PB-biased score\nper clone")
plt.xlabel("Virus")
plt.xticks(rotation=90)
sb.despine()

plt.tight_layout()
fig.savefig("../../05_plots/09_multimers/7_mean_clone_PB_biased_score.pdf", bbox_inches='tight')
plt.show()

In [ ]:
import scipy.stats as stats
import scikit_posthocs as sp

## Extract score values per cell and plot as a box plot per specificity
obs_df = mdata["gex"].obs[["Virus", "PB-biased score", "clone_handle"]].copy()
obs_df = obs_df.dropna(subset="Virus")
obs_df = obs_df.groupby(["clone_handle", "Virus"], observed=True).mean("PB-biased score").reset_index()

groups = [
    group["PB-biased score"].values
    for _, group in obs_df.groupby("Virus", observed=True)
]

H, p = stats.kruskal(*groups)

print(f"Kruskal-Wallis H = {H:.3f}, p = {p:.3e}")

In [ ]:
dunn = sp.posthoc_dunn(
    obs_df,
    val_col="PB-biased score",
    group_col="Virus",
    p_adjust="fdr_bh",   # or "holm", "bonferroni"
)

dunn = dunn.loc[virus_order, virus_order]

dunn_long = (
    dunn.stack()
    .reset_index()
    .rename(columns={
        "level_0": "Virus_1",
        "level_1": "Virus_2",
        0: "adjusted_p"
    })
)

# Remove self-comparisons
dunn_long = dunn_long[dunn_long["Virus_1"] != dunn_long["Virus_2"]]

# Remove duplicate comparisons
dunn_long["comparison"] = dunn_long.apply(
    lambda x: tuple(sorted((x["Virus_1"], x["Virus_2"]))),
    axis=1,
)

dunn_long = (
    dunn_long
    .drop_duplicates("comparison")
    .drop(columns="comparison")
    .sort_values("adjusted_p")
    .assign(
    significance=lambda x: pd.cut(
        x["adjusted_p"],
        bins=[-float("inf"), 0.0001, 0.001, 0.01, 0.05, float("inf")],
        labels=["****","***", "**", "*", "ns"]
    ))
)

dunn_long.head(10)

# Plots 6-7: Report individual markers

## GZMK

In [ ]:
sc.settings.set_figure_params(figsize=(4, 4))
fig = sc.pl.umap(mdata["gex"], color="GZMK",  legend_fontweight= 'bold', legend_loc = "on data", cmap="RdBu_r", frameon= False , size=40, vmin=0, legend_fontoutline=2, return_fig=True, show=False)
plt.tight_layout()
fig.savefig("../../05_plots/09_multimers/30_GZMK_umap.pdf",  format="pdf", dpi=400)
plt.show

In [ ]:
# Work with clones of interest (those with an annotated antigen-specificity)
mtemp = mdata.copy()
print(len(mtemp["gex"]))
mu.pp.filter_obs(mtemp["gex"], "Virus", lambda x: (~x.isna()))
print(len(mtemp["gex"]))

# Remove unused categories
for col in mtemp["gex"].obs.select_dtypes(include="category").columns:
    mtemp["gex"].obs[col] = (
        mtemp["gex"].obs[col].cat.remove_unused_categories()
    )

## Transfer GZMK to .obs
# Get index of GZMK
gene_idx = mtemp["gex"].var_names.get_loc("GZMK")

# Extract expression (handle sparse matrices safely)
gzmk_expr = mtemp["gex"].X[:, gene_idx]

# Convert to dense if needed
if not isinstance(gzmk_expr, np.ndarray):
    gzmk_expr = gzmk_expr.toarray().flatten()
else:
    gzmk_expr = gzmk_expr.flatten()

# Add to obs
mtemp["gex"].obs["GZMK_norm"] = gzmk_expr

## Extract score values per cell and plot as a box plot per specificity
obs_df = mtemp["gex"].obs[["Virus", "clone_handle", "GZMK_norm"]].dropna().copy()
obs_df = (
    obs_df
    .groupby(["Virus", "clone_handle"])["GZMK_norm"]
    .mean()
    .reset_index()
)
obs_df = obs_df[~obs_df["GZMK_norm"].isna()]

#plotting order: 
order =  ["CMV", "NWV", "HMPV", "FLU-A", "HHV-1", "SARS-COV-2", "HAdV-C", "EBV", "HHV-2", "VZV"]

fig, ax = plt.subplots(figsize=(4,4))

# Boxplot
sb.boxplot(
    data=obs_df,
    x="Virus",
    y="GZMK_norm",
    width=0.5,
    color="white",
    linecolor="#4D4D4D", 
    linewidth=0.5,
    fliersize=0,
    zorder=2,
    order=order,
    ax=ax
)

# Set transparency of the boxes
for patch in ax.patches:
    patch.set_alpha(0.90)

# Stripplot
sb.stripplot(
    data=obs_df,
    x="Virus",
    y="GZMK_norm",
    jitter=0.25,
    size=4,
    alpha=0.7,
    color="black",
    zorder=1,
    order=order,
    ax=ax
)



plt.ylabel("Mean GZMK expr.\n")
plt.xlabel("Virus")
plt.xticks(rotation=90)
sb.despine()
plt.grid(False)

plt.tight_layout()
fig.savefig("../../05_plots/09_multimers/31_GZMK_virus_boxplots.pdf", format="pdf", dpi=400)
plt.show()

In [ ]:
import scipy.stats as stats
import scikit_posthocs as sp

groups = [
    group["GZMK_norm"].values
    for _, group in obs_df.groupby("Virus", observed=True)
]

H, p = stats.kruskal(*groups)

print(f"Kruskal-Wallis H = {H:.3f}, p = {p:.3e}")

In [ ]:
dunn = sp.posthoc_dunn(
    obs_df,
    val_col="GZMK_norm",
    group_col="Virus",
    p_adjust="fdr_bh",   # or "holm", "bonferroni"
)

dunn = dunn.loc[order, order]

dunn_long = (
    dunn.stack()
    .reset_index()
    .rename(columns={
        "level_0": "Virus_1",
        "level_1": "Virus_2",
        0: "adjusted_p"
    })
)

# Remove self-comparisons
dunn_long = dunn_long[dunn_long["Virus_1"] != dunn_long["Virus_2"]]

# Remove duplicate comparisons
dunn_long["comparison"] = dunn_long.apply(
    lambda x: tuple(sorted((x["Virus_1"], x["Virus_2"]))),
    axis=1,
)

dunn_long = (
    dunn_long
    .drop_duplicates("comparison")
    .drop(columns="comparison")
    .sort_values("adjusted_p")
)

dunn_long.head(10)

In [ ]:
## Transfer GZMK to .obs
# Get index of GZMK
gene_idx = mtemp["gex"].var_names.get_loc("GZMK")

# Extract expression (handle sparse matrices safely)
gzmk_expr = mtemp["gex"].X[:, gene_idx]

# Convert to dense if needed
if not isinstance(gzmk_expr, np.ndarray):
    gzmk_expr = gzmk_expr.toarray().flatten()
else:
    gzmk_expr = gzmk_expr.flatten()

# Add to obs
mtemp["gex"].obs["gene_norm"] = gzmk_expr

## Extract score values per cell and plot as a box plot per specificity
obs_df = mtemp["gex"].obs[["Virus", "clone_handle", "catpat", "gene_norm"]].dropna().copy()
obs_df = (
    obs_df
    .groupby(["Virus", "catpat", "clone_handle"], observed=True)["gene_norm"]
    .mean()
    .reset_index()
)


#plotting order: 
# order =  ["CMV", "NWV", "HMPV", "HHV-6B", "FLU-A", "HHV-1", "SARS-COV-2", "HAdV-C", "EBV", "HHV-2", "VZV"]
order =  ["CMV", "FLU-A", "SARS-COV-2", "EBV"]

obs_df = obs_df[~obs_df["gene_norm"].isna()]
obs_df = obs_df[obs_df["Virus"].isin(order)]

# Remove unused categories
for col in obs_df.select_dtypes(include="category").columns:
    obs_df[col] = (
        obs_df[col].cat.remove_unused_categories()
    )


# Define manual colors
palette = {
    "NIC": "#FFFFFF",   # blue
    "MS": "#000000"       # red
}

#Define categories for catpat to ensure the correct order
obs_df["catpat"] = pd.Categorical(obs_df["catpat"], ordered=True, categories=["NIC", "MS"])


fig, ax = plt.subplots(figsize=(4,4))

# Boxplot
sb.boxplot(
    data=obs_df,
    x="Virus",
    y="gene_norm",
    hue="catpat",
    palette=["white", "white"],   # white boxes for both groups
    linecolor="#4D4D4D", 
    linewidth=0.5,
    fliersize=0,
    zorder=2,
    order=order,
    ax=ax
)

# Set transparency of the boxes
for patch in ax.patches:
    patch.set_alpha(0.90)

# Stripplot
sb.stripplot(
    data=obs_df,
    x="Virus",
    y="gene_norm",
    hue="catpat",
    edgecolor="black",
    linewidth=0.5,
    palette=palette,
    dodge=True,
    jitter=0.25,
    size=5,
    alpha=0.8,
    zorder=1,
    order=order,
    ax=ax
)

# Remove duplicate legends and create one clean legend
handles, labels = ax.get_legend_handles_labels()
ax.legend(
    handles[:2],
    labels[:2],
    title="Group",
    loc="center left",
    bbox_to_anchor=(1.02, 0.5),
    frameon=False,
)

plt.ylabel("GZMK")
plt.xlabel("Virus")
plt.xticks(rotation=90)
sb.despine()
plt.grid(False)

plt.tight_layout()
fig.savefig("../../05_plots/09_multimers/31_GZMK_virus_boxplots_NICvMS.pdf", format="pdf", dpi=400)
plt.show()

In [ ]:
results = []

for virus, sub in obs_df.groupby("Virus", observed=True):

    control = sub.loc[sub["catpat"] == "NIC", "gene_norm"]
    rrms = sub.loc[sub["catpat"] == "MS", "gene_norm"]

    # Skip if either group is empty
    if len(control) == 0 or len(rrms) == 0:
        continue

    stat, p = mannwhitneyu(
        control,
        rrms,
        alternative="two-sided",
        method="exact"
    )

    results.append({
        "Virus": virus,
        "n_Control": len(control),
        "n_RRMS": len(rrms),
        "U": stat,
        "p": p,
    })

results_df = (
    pd.DataFrame(results)
    .sort_values("Virus")
    .reset_index(drop=True)
)

results_df

## CXCR3

In [ ]:
sc.settings.set_figure_params(figsize=(4, 4))
fig = sc.pl.umap(mdata["gex"], color="CXCR3",  legend_fontweight= 'bold', legend_loc = "on data", cmap="RdBu_r", frameon= False , size=40, vmin=0, legend_fontoutline=2, return_fig=True, show=False)
plt.tight_layout()
fig.savefig("../../05_plots/09_multimers/30_CXCR3_umap.pdf",  format="pdf", dpi=400)
plt.show

In [ ]:
## Transfer GZMK to .obs
# Get index of gene of interest
gene_interest = "CXCR3"
gene_idx = mtemp["gex"].var_names.get_loc(gene_interest)

# Extract expression (handle sparse matrices safely)
gzmk_expr = mtemp["gex"].X[:, gene_idx]

# Convert to dense if needed
if not isinstance(gzmk_expr, np.ndarray):
    gzmk_expr = gzmk_expr.toarray().flatten()
else:
    gzmk_expr = gzmk_expr.flatten()

# Add to obs
mtemp["gex"].obs["gene_norm"] = gzmk_expr

## Extract score values per cell and plot as a box plot per specificity
obs_df = mtemp["gex"].obs[["Virus", "clone_handle", "gene_norm"]].dropna().copy()
obs_df = (
    obs_df
    .groupby(["Virus", "clone_handle"])["gene_norm"]
    .mean()
    .reset_index()
)
obs_df = obs_df[~obs_df["gene_norm"].isna()]

#plotting order: 
order =  ["CMV", "NWV", "HMPV", "FLU-A", "HHV-1", "SARS-COV-2", "HAdV-C", "EBV", "HHV-2", "VZV"]

fig, ax = plt.subplots(figsize=(4,4))

# Boxplot
sb.boxplot(
    data=obs_df,
    x="Virus",
    y="gene_norm",
    width=0.5,
    color="white",
    linecolor="#4D4D4D", 
    linewidth=0.5,
    fliersize=0,
    zorder=2,
    order=order,
    ax=ax
)

# Set transparency of the boxes
for patch in ax.patches:
    patch.set_alpha(0.90)

# Stripplot
sb.stripplot(
    data=obs_df,
    x="Virus",
    y="gene_norm",
    jitter=0.25,
    size=4,
    alpha=0.7,
    color="black",
    zorder=1,
    order=order,
    ax=ax
)



plt.ylabel(f"Mean {gene_interest} expr.\n")
plt.xlabel("Virus")
plt.xticks(rotation=90)
sb.despine()
plt.grid(False)

plt.tight_layout()
fig.savefig(f"../../05_plots/09_multimers/31_{gene_interest}_virus_boxplots.pdf", format="pdf", dpi=400)
plt.show()

In [ ]:
import scipy.stats as stats
import scikit_posthocs as sp

groups = [
    group["gene_norm"].values
    for _, group in obs_df.groupby("gene_norm", observed=True)
]

H, p = stats.kruskal(*groups)

print(f"Kruskal-Wallis H = {H:.3f}, p = {p:.3e}")

In [ ]:
dunn = sp.posthoc_dunn(
    obs_df,
    val_col="gene_norm",
    group_col="Virus",
    p_adjust="fdr_bh",   # or "holm", "bonferroni"
)

dunn = dunn.loc[order, order]

dunn_long = (
    dunn.stack()
    .reset_index()
    .rename(columns={
        "level_0": "Virus_1",
        "level_1": "Virus_2",
        0: "adjusted_p"
    })
)

# Remove self-comparisons
dunn_long = dunn_long[dunn_long["Virus_1"] != dunn_long["Virus_2"]]

# Remove duplicate comparisons
dunn_long["comparison"] = dunn_long.apply(
    lambda x: tuple(sorted((x["Virus_1"], x["Virus_2"]))),
    axis=1,
)

dunn_long = (
    dunn_long
    .drop_duplicates("comparison")
    .drop(columns="comparison")
    .sort_values("adjusted_p")
)

dunn_long.head(10)

## GPR183

In [ ]:
## Transfer GZMK to .obs
# Get index of gene of interest
gene_interest = "DUSP2"
gene_idx = mtemp["gex"].var_names.get_loc(gene_interest)

# Extract expression (handle sparse matrices safely)
gzmk_expr = mtemp["gex"].X[:, gene_idx]

# Convert to dense if needed
if not isinstance(gzmk_expr, np.ndarray):
    gzmk_expr = gzmk_expr.toarray().flatten()
else:
    gzmk_expr = gzmk_expr.flatten()

# Add to obs
mtemp["gex"].obs["gene_norm"] = gzmk_expr

## Extract score values per cell and plot as a box plot per specificity
obs_df = mtemp["gex"].obs[["Virus", "clone_handle", "gene_norm"]].dropna().copy()
obs_df = (
    obs_df
    .groupby(["Virus", "clone_handle"])["gene_norm"]
    .mean()
    .reset_index()
)
obs_df = obs_df[~obs_df["gene_norm"].isna()]

#plotting order: 
order =  ["CMV", "NWV", "HMPV", "FLU-A", "HHV-1", "SARS-COV-2", "HAdV-C", "EBV", "HHV-2", "VZV"]

fig, ax = plt.subplots(figsize=(4,4))

# Boxplot
sb.boxplot(
    data=obs_df,
    x="Virus",
    y="gene_norm",
    width=0.5,
    color="white",
    linecolor="#4D4D4D", 
    linewidth=0.5,
    fliersize=0,
    zorder=2,
    order=order,
    ax=ax
)

# Set transparency of the boxes
for patch in ax.patches:
    patch.set_alpha(0.90)

# Stripplot
sb.stripplot(
    data=obs_df,
    x="Virus",
    y="gene_norm",
    jitter=0.25,
    size=4,
    alpha=0.7,
    color="black",
    zorder=1,
    order=order,
    ax=ax
)



plt.ylabel(f"Mean {gene_interest} expr.\n")
plt.xlabel("Virus")
plt.xticks(rotation=90)
sb.despine()
plt.grid(False)

plt.tight_layout()
fig.savefig(f"../../05_plots/09_multimers/31_{gene_interest}_virus_boxplots.pdf", format="pdf", dpi=400)
plt.show()

# Plot 8: TE and TEM amongst each clone

## TEM statistics

In [ ]:
mdata["gex"].obs["Virus"].value_counts()

In [ ]:
obs_df = mdata["gex"].obs.copy()

# Focus on multimer annotated cells
obs_df = obs_df[~obs_df["Virus"].isna()].copy()
# obs_df = obs_df[~obs_df["catpat"].isin(["IC"])]

# Remove unused categories
for col in obs_df.select_dtypes(include="category").columns:
    obs_df[col] = (
        obs_df[col].cat.remove_unused_categories()
    )

# Categories 
template_df = obs_df.groupby(["clone_handle", "L3_ann_overview_scanvi"]).size().reset_index(name="n_cells")

# Add virus information
vir_ann = obs_df[["clone_handle", "Virus"]].drop_duplicates()
plot_df = template_df.merge(vir_ann, how="left")

# Total cells per clone_handle and Virus
plot_df["total_cells"] = (
    plot_df
    .groupby(["clone_handle", "Virus"], observed=True)["n_cells"]
    .transform("sum")
)

# Percentage of cells in each cluster
plot_df["percent_cells"] = 100 * plot_df["n_cells"] / plot_df["total_cells"]

# Focus on TEM
plot_df = plot_df[plot_df["L3_ann_overview_scanvi"] ==  "TEM"]

# Optional: specify the order of viruses
virus_order = ["CMV", "NWV", "HMPV", "HHV-6B", "FLU-A", "HHV-1", "SARS-COV-2", "HAdV-C", "EBV", "HHV-2", "VZV"]

# Keep only viruses that are present
virus_order = [v for v in virus_order if v in plot_df["Virus"].unique()]

fig, ax = plt.subplots(figsize=(4, 4))

# Boxplots
sb.boxplot(
    data=plot_df,
    x="Virus",
    y="percent_cells",
    order=virus_order,
    fliersize=0,
    width=0.5,
    color="white",
    linecolor="#4D4D4D", 
    linewidth=0.5,
    zorder=2,
    ax=ax,
)

# Set transparency of the boxes
for patch in ax.patches:
    patch.set_alpha(0.90)
    
# Individual clones
sb.stripplot(
    data=plot_df,
    x="Virus",
    y="percent_cells",
    order=virus_order,
    color="black",
    jitter=0.25,
    size=5,
    zorder=1,
    ax=ax,
)

ax.set_xticklabels(ax.get_xticklabels(), rotation=90, ha="right")
ax.set_ylabel("% of cells in TEM")
ax.set_ylim(-10, 110)

sb.despine()
ax.grid(False)

plt.tight_layout()
fig.savefig("../../05_plots/09_multimers/7_percentage_in_L3_overview.pdf", bbox_inches='tight')
plt.show()

In [ ]:
import scipy.stats as stats
import scikit_posthocs as sp

groups = [
    group["percent_cells"].values
    for _, group in plot_df.groupby("Virus", observed=True)
]

H, p = stats.kruskal(*groups)

print(f"Kruskal-Wallis H = {H:.3f}, p = {p:.3e}")

In [ ]:
dunn = sp.posthoc_dunn(
    plot_df,
    val_col="percent_cells",
    group_col="Virus",
    p_adjust="fdr_bh",   # or "holm", "bonferroni"
)

dunn = dunn.loc[virus_order, virus_order]

dunn_long = (
    dunn.stack()
    .reset_index()
    .rename(columns={
        "level_0": "Virus_1",
        "level_1": "Virus_2",
        0: "adjusted_p"
    })
)

# Remove self-comparisons
dunn_long = dunn_long[dunn_long["Virus_1"] != dunn_long["Virus_2"]]

# Remove duplicate comparisons
dunn_long["comparison"] = dunn_long.apply(
    lambda x: tuple(sorted((x["Virus_1"], x["Virus_2"]))),
    axis=1,
)

dunn_long = (
    dunn_long
    .drop_duplicates("comparison")
    .drop(columns="comparison")
    .sort_values("adjusted_p")
)

dunn_long.head(10)

## TCM statistics

In [ ]:
mdata["gex"].obs["Virus"].value_counts()

In [ ]:
obs_df = mdata["gex"].obs.copy()

# Focus on multimer annotated cells
obs_df = obs_df[~obs_df["Virus"].isna()].copy()

# Remove unused categories
for col in obs_df.select_dtypes(include="category").columns:
    obs_df[col] = (
        obs_df[col].cat.remove_unused_categories()
    )

# Categories 
template_df = obs_df.groupby(["clone_handle", "L3_ann_overview_scanvi"]).size().reset_index(name="n_cells")

# Add virus information
vir_ann = obs_df[["clone_handle", "Virus"]].drop_duplicates()
plot_df = template_df.merge(vir_ann, how="left")

# Total cells per clone_handle and Virus
plot_df["total_cells"] = (
    plot_df
    .groupby(["clone_handle", "Virus"], observed=True)["n_cells"]
    .transform("sum")
)

# Percentage of cells in each cluster
plot_df["percent_cells"] = 100 * plot_df["n_cells"] / plot_df["total_cells"]

# Focus on TEM
plot_df = plot_df[plot_df["L3_ann_overview_scanvi"] ==  "TCM"]

# Optional: specify the order of viruses
virus_order = ["CMV", "NWV", "HMPV", "HHV-6B", "FLU-A", "HHV-1", "SARS-COV-2", "HAdV-C", "EBV", "HHV-2", "VZV"]

# Keep only viruses that are present
virus_order = [v for v in virus_order if v in plot_df["Virus"].unique()]

fig, ax = plt.subplots(figsize=(4, 4))

# Boxplots
sb.boxplot(
    data=plot_df,
    x="Virus",
    y="percent_cells",
    order=virus_order,
    fliersize=0,
    width=0.5,
    color="white",
    linecolor="#4D4D4D", 
    linewidth=0.5,
    zorder=2,
    ax=ax,
)

# Set transparency of the boxes
for patch in ax.patches:
    patch.set_alpha(0.90)
    
# Individual clones
sb.stripplot(
    data=plot_df,
    x="Virus",
    y="percent_cells",
    order=virus_order,
    color="black",
    jitter=0.25,
    size=5,
    zorder=1,
    ax=ax,
)

# # Violin plots
# sb.violinplot(
#     data=plot_df,
#     x="Virus",
#     y="percent_cells",
#     order=virus_order,
#     inner=None,
#     cut=0,
#     color="white",
#     linewidth=0.5,
#     linecolor="#4D4D4D",
#     width=0.8,
#     density_norm="width",   # all violins have the same maximum width
#     ax=ax,
# )

# # Make violins slightly transparent
# for violin in ax.collections:
#     violin.set_alpha(0.9)

# # Individual clones
# sb.stripplot(
#     data=plot_df,
#     x="Virus",
#     y="percent_cells",
#     order=virus_order,
#     color="black",
#     jitter=0.1,
#     alpha=0.5,
#     size=5,
#     zorder=3,
#     ax=ax,
# )

ax.set_xticklabels(ax.get_xticklabels(), rotation=90, ha="right")
ax.set_ylabel("% of cells in TCM")
ax.set_ylim(-10, 110)

sb.despine()
ax.grid(False)

plt.tight_layout()
fig.savefig("../../05_plots/09_multimers/7_percentage_TCM_in_L3_overview.pdf", bbox_inches='tight')
plt.show()

In [ ]:
import scipy.stats as stats
import scikit_posthocs as sp

groups = [
    group["percent_cells"].values
    for _, group in plot_df.groupby("Virus", observed=True)
]

H, p = stats.kruskal(*groups)

print(f"Kruskal-Wallis H = {H:.3f}, p = {p:.3e}")

In [ ]:
dunn = sp.posthoc_dunn(
    plot_df,
    val_col="percent_cells",
    group_col="Virus",
    p_adjust="fdr_bh",   # or "holm", "bonferroni"
)

dunn = dunn.loc[virus_order, virus_order]

dunn_long = (
    dunn.stack()
    .reset_index()
    .rename(columns={
        "level_0": "Virus_1",
        "level_1": "Virus_2",
        0: "adjusted_p"
    })
)

# Remove self-comparisons
dunn_long = dunn_long[dunn_long["Virus_1"] != dunn_long["Virus_2"]]

# Remove duplicate comparisons
dunn_long["comparison"] = dunn_long.apply(
    lambda x: tuple(sorted((x["Virus_1"], x["Virus_2"]))),
    axis=1,
)

dunn_long = (
    dunn_long
    .drop_duplicates("comparison")
    .drop(columns="comparison")
    .sort_values("adjusted_p")
)

dunn_long.head(10)

## TE statistics

In [ ]:
mdata["gex"].obs["Virus"].value_counts()

In [ ]:
obs_df = mdata["gex"].obs.copy()

# Focus on multimer annotated cells
obs_df = obs_df[~obs_df["Virus"].isna()].copy()

# Remove unused categories
for col in obs_df.select_dtypes(include="category").columns:
    obs_df[col] = (
        obs_df[col].cat.remove_unused_categories()
    )

# Categories 
template_df = obs_df.groupby(["clone_handle", "L3_ann_overview_scanvi"]).size().reset_index(name="n_cells")

# Add virus information
vir_ann = obs_df[["clone_handle", "Virus"]].drop_duplicates()
plot_df = template_df.merge(vir_ann, how="left")

# Total cells per clone_handle and Virus
plot_df["total_cells"] = (
    plot_df
    .groupby(["clone_handle", "Virus"], observed=True)["n_cells"]
    .transform("sum")
)

# Percentage of cells in each cluster
plot_df["percent_cells"] = 100 * plot_df["n_cells"] / plot_df["total_cells"]

# Focus on TEM
plot_df = plot_df[plot_df["L3_ann_overview_scanvi"] ==  "TE"]

# Optional: specify the order of viruses
virus_order = ["CMV", "NWV", "HMPV", "HHV-6B", "FLU-A", "HHV-1", "SARS-COV-2", "HAdV-C", "EBV", "HHV-2", "VZV"]

# Keep only viruses that are present
virus_order = [v for v in virus_order if v in plot_df["Virus"].unique()]

fig, ax = plt.subplots(figsize=(4, 4))

# Boxplots
sb.boxplot(
    data=plot_df,
    x="Virus",
    y="percent_cells",
    order=virus_order,
    fliersize=0,
    width=0.5,
    color="white",
    linecolor="#4D4D4D", 
    linewidth=0.5,
    zorder=2,
    ax=ax,
)

# Set transparency of the boxes
for patch in ax.patches:
    patch.set_alpha(0.90)
    
# Individual clones
sb.stripplot(
    data=plot_df,
    x="Virus",
    y="percent_cells",
    order=virus_order,
    color="black",
    jitter=0.25,
    size=5,
    zorder=1,
    ax=ax,
)

# # Violin plots
# sb.violinplot(
#     data=plot_df,
#     x="Virus",
#     y="percent_cells",
#     order=virus_order,
#     inner=None,
#     cut=0,
#     color="white",
#     linewidth=0.5,
#     linecolor="#4D4D4D",
#     width=0.8,
#     density_norm="width",   # all violins have the same maximum width
#     ax=ax,
# )

# # Make violins slightly transparent
# for violin in ax.collections:
#     violin.set_alpha(0.9)

# # Individual clones
# sb.stripplot(
#     data=plot_df,
#     x="Virus",
#     y="percent_cells",
#     order=virus_order,
#     color="black",
#     jitter=0.1,
#     alpha=0.5,
#     size=5,
#     zorder=3,
#     ax=ax,
# )

ax.set_xticklabels(ax.get_xticklabels(), rotation=90, ha="right")
ax.set_ylabel("% of cells in TE")
ax.set_ylim(-10, 110)

sb.despine()
ax.grid(False)

plt.tight_layout()
fig.savefig("../../05_plots/09_multimers/7_percentage_TE_in_L3_overview.pdf", bbox_inches='tight')
plt.show()

In [ ]:
# import scipy.stats as stats
import scikit_posthocs as sp

groups = [
    group["percent_cells"].values
    for _, group in plot_df.groupby("Virus", observed=True)
]

H, p = stats.kruskal(*groups)

print(f"Kruskal-Wallis H = {H:.3f}, p = {p:.3e}")

In [ ]:
dunn = sp.posthoc_dunn(
    plot_df,
    val_col="percent_cells",
    group_col="Virus",
    p_adjust="fdr_bh",   # or "holm", "bonferroni"
)

dunn = dunn.loc[virus_order, virus_order]

dunn_long = (
    dunn.stack()
    .reset_index()
    .rename(columns={
        "level_0": "Virus_1",
        "level_1": "Virus_2",
        0: "adjusted_p"
    })
)

# Remove self-comparisons
dunn_long = dunn_long[dunn_long["Virus_1"] != dunn_long["Virus_2"]]

# Remove duplicate comparisons
dunn_long["comparison"] = dunn_long.apply(
    lambda x: tuple(sorted((x["Virus_1"], x["Virus_2"]))),
    axis=1,
)

dunn_long = (
    dunn_long
    .drop_duplicates("comparison")
    .drop(columns="comparison")
    .sort_values("adjusted_p")
    .assign(
        significance=lambda x: pd.cut(
        x["adjusted_p"],
        bins=[-float("inf"), 0.0001, 0.001, 0.01, 0.05, float("inf")],
        labels=["****","***", "**", "*", "ns"]
    ))
)

dunn_long.head(10)

# Plots 9 - X: Counts for writing and supplemental figures

## How many clonotypes do we have for each virus?

In [ ]:
obs_df = mdata["gex"].obs.copy()
obs_df = obs_df.dropna(subset="Virus")
df = obs_df[["handle", "clone_handle", "Virus"]].value_counts().reset_index(name = "counts")
df = df[["Virus"]].value_counts().reset_index(name="counts")

# Order viruses by counts
order = df.sort_values("counts", ascending=False)["Virus"]

# Initialize the matplotlib figure
f, ax = plt.subplots(figsize=(4, 4))

# Plot
sb.set_color_codes("pastel")
sb.barplot(x="Virus", y="counts", data=df, order=order)

# Add count labels on top of bars
for bar in ax.patches:
    height = bar.get_height()
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        height,
        f"{int(height)}",
        ha="center",
        va="bottom",
        fontsize=11
    )


# Add a legend and informative axis label
ax.legend(ncol=1, loc="lower right", frameon=True)
ax.set(ylabel="Number of clonotypes")
plt.xticks(rotation=90)
sb.despine()
plt.grid(False)

plt.tight_layout()
plt.savefig("../../05_plots/09_multimers/28_Clones_discovered_for_each_virus.pdf", format="pdf", dpi=400)
plt.show()

## How many antigen-specific cells did we capture?

In [ ]:
obs_df = mdata["gex"].obs.copy()
obs_df = obs_df.dropna(subset="Virus")
# df = obs_df[["handle", "clone_handle", "Virus"]].value_counts().reset_index(name = "counts")
df = obs_df[["Virus"]].value_counts().reset_index(name="counts")

# Order viruses by counts
order = df.sort_values("counts", ascending=False)["Virus"]

# Initialize the matplotlib figure
f, ax = plt.subplots(figsize=(4, 4))

# Plot
sb.set_color_codes("pastel")
sb.barplot(x="Virus", y="counts", data=df, order=order)

# Add count labels on top of bars
for bar in ax.patches:
    height = bar.get_height()
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        height,
        f"{int(height)}",
        ha="center",
        va="bottom",
        fontsize=11
    )


# Add a legend and informative axis label
ax.legend(ncol=1, loc="lower right", frameon=True)
ax.set(ylabel="Number of clonotypes")
plt.xticks(rotation=90)
sb.despine()
plt.grid(False)

plt.tight_layout()
plt.savefig("../../05_plots/09_multimers/28_cells_discovered_for_each_virus.pdf", format="pdf", dpi=400)
plt.show()

## How many clonotypes do we have for each antigen?

In [ ]:
obs_df = mdata["gex"].obs.copy()
obs_df = obs_df.dropna(subset="Virus")
df = obs_df[["handle", "clone_handle", "Antigen"]].value_counts().reset_index(name = "counts")
df = df[["Antigen"]].value_counts().reset_index(name="counts")

# Order viruses by counts
order = df.sort_values("counts", ascending=False)["Antigen"]

# Initialize the matplotlib figure
f, ax = plt.subplots(figsize=(10, 4))

# Plot
sb.set_color_codes("pastel")
sb.barplot(x="Antigen", y="counts", data=df, order=order)

# Add count labels on top of bars
for bar in ax.patches:
    height = bar.get_height()
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        height,
        f"{int(height)}",
        ha="center",
        va="bottom",
        fontsize=11
    )


# Add a legend and informative axis label
ax.legend(ncol=1, loc="lower right", frameon=True)
ax.set(ylabel="Number of clonotypes")
plt.xticks(rotation=90)
sb.despine()

## How many independent polyclonal responses do we have for each virus?

In [ ]:
obs_df = mdata["gex"].obs.copy()
df = obs_df[["handle", "donor", "Virus"]].value_counts().reset_index(name = "counts")
df = df.dropna(subset="handle")
df = df[["Virus"]].value_counts().reset_index(name="counts")

# Order viruses by counts
order = df.sort_values("counts", ascending=False)["Virus"]

# Initialize the matplotlib figure
f, ax = plt.subplots(figsize=(4, 4))

# Plot
sb.set_color_codes("pastel")
sb.barplot(x="Virus", y="counts", data=df, order=order)

# Add count labels on top of bars
for bar in ax.patches:
    height = bar.get_height()
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        height,
        f"{int(height)}",
        ha="center",
        va="bottom",
        fontsize=11
    )


# Add a legend and informative axis label
ax.legend(ncol=1, loc="lower right", frameon=True)
ax.set(ylabel="Number of polyclonal immune responses")
plt.xticks(rotation=90)
sb.despine()

plt.tight_layout()
f.savefig("../../05_plots/09_multimers/8_number_of_independent_immune_responses.pdf", format="pdf", dpi=400)
plt.show()

## How many responses do we have for each multimer?

In [ ]:
obs_df = mdata["gex"].obs.copy()
df = obs_df[["handle", "clone_handle"]].value_counts().reset_index(name = "counts")
df = df[["handle"]].value_counts().reset_index(name="counts")

#How many in total
print("We have ", sum(df["counts"]), " responses in total")

# Order viruses by counts
order = df.sort_values("counts", ascending=False)["handle"]

# Initialize the matplotlib figure
f, ax = plt.subplots(figsize=(12, 8))

# Plot
sb.set_color_codes("pastel")
sb.barplot(x="handle", y="counts", data=df, order=order)

# Add count labels on top of bars
for bar in ax.patches:
    height = bar.get_height()
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        height,
        f"{int(height)}",
        ha="center",
        va="bottom",
        fontsize=11
    )


# Add a legend and informative axis label
ax.legend(ncol=1, loc="lower right", frameon=True)
ax.set(ylabel="Number of clones")
plt.xticks(rotation=90)
sb.despine()

plt.tight_layout()
f.savefig("../../05_plots/09_multimers/10_number_clones_per_multimer.pdf", format="pdf", dpi=400)
plt.show()

In [ ]:
#Here with the antigen too

obs_df = mdata["gex"].obs.copy()
obs_df["ag_x_handle"] = obs_df["Antigen"].astype(str) + "_" + obs_df["handle"].astype(str) 
obs_df = obs_df.dropna(subset="handle")
df = obs_df[["ag_x_handle", "clone_handle"]].value_counts().reset_index(name = "counts")
df = df[["ag_x_handle"]].value_counts().reset_index(name="counts")

#How many in total
print("We have ", sum(df["counts"]), " responses in total")

# Order viruses by counts
order = df.sort_values("counts", ascending=False)["ag_x_handle"]

# Initialize the matplotlib figure
f, ax = plt.subplots(figsize=(12, 8))

# Plot
sb.set_color_codes("pastel")
sb.barplot(x="ag_x_handle", y="counts", data=df, order=order)

# Add count labels on top of bars
for bar in ax.patches:
    height = bar.get_height()
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        height,
        f"{int(height)}",
        ha="center",
        va="bottom",
        fontsize=11
    )


# Add a legend and informative axis label
ax.legend(ncol=1, loc="lower right", frameon=True)
ax.set(ylabel="Number of clones")
plt.xticks(rotation=90)
sb.despine()

plt.tight_layout()
f.savefig("../../05_plots/09_multimers/10_number_clones_per_multimer_aghandle.pdf", format="pdf", dpi=400)
plt.show()

## How many multimers did we stain for?

In [ ]:
mult_ann = pd.read_excel("../../03_annotation/11_multimer_experiment_ann/FeatureBarcodCorrected.xlsx").dropna(subset="HLA")
mult_ann = mult_ann.groupby("Virus").size().reset_index(name="counts")

#How many in total
print("We collected a total of ", sum(mult_ann["counts"]), " multimers")

# Order viruses by counts
order = mult_ann.sort_values("counts", ascending=False)["Virus"]

# Initialize the matplotlib figure
f, ax = plt.subplots(figsize=(5, 4))

# Plot
sb.set_color_codes("pastel")
sb.barplot(x="Virus", y="counts", data=mult_ann, order=order)

# Add count labels on top of bars
for bar in ax.patches:
    height = bar.get_height()
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        height,
        f"{int(height)}",
        ha="center",
        va="bottom",
        fontsize=11
    )


# Add a legend and informative axis label
ax.legend(ncol=1, loc="lower right", frameon=True)
ax.set(ylabel="Number of multimers\nin the staining panel")
plt.xticks(rotation=90)
sb.despine()

plt.tight_layout()
f.savefig("../../05_plots/09_multimers/9_Number_multimers_per_virus_origin.pdf", format="pdf", dpi=400)
plt.show()

# Plot 10: Antigen disaggregated comparisons

## CSF-biased score

In [ ]:
## Among EBV specificities, are there antigens with different kinetics?

## Extract score values per cell and plot as a box plot per specificity
obs_df = mdata["gex"].obs[["Virus", "CSF-biased score", "clone_handle", "Antigen"]].copy()

# Short list antigens based on how many clones we observed. Only include antigens with more than 5 independent responses
temp_ag_count = obs_df[obs_df["Virus"].isin(["EBV", "CMV", "SARS-COV-2", "FLU-A"])][["Antigen", "clone_handle"]].dropna().drop_duplicates()[["Antigen"]].value_counts().reset_index()
ag_shortlist = temp_ag_count[temp_ag_count["count"] > 5]["Antigen"].tolist()

#Filter to include only the most promising antigens
obs_df = obs_df[obs_df["Antigen"].isin(ag_shortlist)]
obs_df = obs_df.dropna(subset="Antigen")

#Calculate mean CSF-biased scores per clone
obs_df = obs_df.groupby(["clone_handle", "Antigen", "Virus"], observed=True).mean("CSF-biased score").reset_index()

# Goal: Calculate percentile ranks for each clone and display the cummulative % sum of clones belonging to each Antigen category on the y-axis going from the top percentile to the lowest percentile on the x-axis. 
#Sort the dataframe first:
obs_df = obs_df.sort_values("CSF-biased score", ascending=False)

# Calculate percentile rank for each clone's CSF-biased score
# percentile_rank gives the percentage of scores <= current score
obs_df["percentile_rank"] = obs_df["CSF-biased score"].rank(pct=True) * 100

# # Compute cumulative % of clones per Virus group along the ranked list
ag_groups = obs_df["Antigen"].unique()
ag_counts = obs_df["Antigen"].value_counts()  # total clones per virus

fig, ax = plt.subplots(figsize=(3, 3))

for ag in ag_groups:
    # Boolean mask: is each ranked clone from this virus?
    is_ag = (obs_df["Antigen"] == ag).astype(int)
    
    # Cumulative sum normalized by total clones of that virus → cumulative %
    cumulative_pct = is_ag.cumsum() / ag_counts[ag] * 100
    
    ax.plot(obs_df["percentile_rank"], cumulative_pct, label=ag)

ax.set_xlabel("CSF-biased Score Percentile (High → Low)")
ax.set_ylabel("Proportion of clones")
ax.set_title("For antigens with more than 5 independent clones")
ax.invert_xaxis()  # x-axis goes from top percentile (100) to lowest (0)
ax.legend()
plt.grid(False)

plt.tight_layout()
plt.savefig("../../05_plots/09_multimers/18_cummulative_over_clones_and_antigens.pdf", format="pdf", dpi=400)
plt.show()

In [ ]:
### obs_df from before
obs_df["Antigen"] = obs_df["Antigen"].cat.remove_unused_categories()

#plotting order: 
order = (
    obs_df
    .groupby("Antigen")["CSF-biased score"]
    .median()
    .sort_values()
    .index
)

csf_order = order.copy()

# Create the figure object
fig, ax = plt.subplots(figsize=(3.55,3))

# Stripplot (background dots)
sb.stripplot(
    data=obs_df,
    x="Antigen",
    y="CSF-biased score",
    jitter=0.25,
    size=4,
    alpha=0.7,
    color="black",
    zorder=1,
    order=order,
    ax=ax
)

# Boxplot (foreground)
sb.boxplot(
    data=obs_df,
    x="Antigen",
    y="CSF-biased score",
    width=0.5,
    color="white",
    fliersize=0,
    boxprops=dict(alpha=.5),
    linewidth=1,
    zorder=2,
    order=order,
    ax=ax
)

plt.ylabel("Mean CSF-biased score\nper clone")
plt.xlabel("Antigen")
plt.xticks(rotation=90)
sb.despine()

plt.tight_layout()
fig.savefig("../../05_plots/09_multimers/12_mean_clone_CSF_biased_score_per_antigen.pdf", bbox_inches='tight')
plt.show()

# Statistics ---------------------------------------------------------------
dunn = sp.posthoc_dunn(
    obs_df,
    val_col="CSF-biased score",
    group_col="Antigen",
    p_adjust="fdr_bh",
)

# Convert pairwise matrix to long format
dunn_long = (
    dunn
    .rename_axis(index="Virus_1", columns="Virus_2")
    .stack()
    .rename("adjusted_p")
    .reset_index()
)

# Remove self-comparisons
dunn_long = dunn_long[
    dunn_long["Virus_1"] != dunn_long["Virus_2"]
].copy()

# Remove duplicate comparisons
dunn_long["comparison"] = dunn_long.apply(
    lambda row: tuple(sorted([row["Virus_1"], row["Virus_2"]])),
    axis=1,
)

dunn_long = (
    dunn_long
    .drop_duplicates(subset="comparison")
    .drop(columns="comparison")
    .sort_values("adjusted_p")
    .reset_index(drop=True)
)

dunn_long["significance"] = pd.cut(
    dunn_long["adjusted_p"],
    bins=[-float("inf"), 0.0001, 0.001, 0.01, 0.05, float("inf")],
    labels=["****", "***", "**", "*", "ns"],
)

dunn_long.head(20)

In [ ]:
# Clean data
df_clean = obs_df.dropna(subset=['Antigen', 'CSF-biased score'])

# Define the order
order_ag = order

# Perform Dunn's test with FDR correction
dunn_results = sp.posthoc_dunn(
    df_clean, 
    val_col='CSF-biased score', 
    group_col='Antigen', 
    p_adjust='fdr_bh'
)

# Reindex both rows and columns to match your desired order
dunn_results = dunn_results.reindex(index=order_ag, columns=order_ag)

# Convert p-values to -log10 for heatmap coloring
heatmap_data = -np.log10(dunn_results)
heatmap_data = heatmap_data.clip(upper=10)  # clip extreme values

# Function to convert p-value to stars
def pval_to_stars(p):
    if p <= 0.0001:
        return "****"
    elif p <= 0.001:
        return "***"
    elif p <= 0.01:
        return "**"
    elif p <= 0.05:
        return "*"
    else:
        return "ns"

# Apply function to create annotation matrix
stars_df = dunn_results.applymap(pval_to_stars)

stars_df

## PB-biased score

In [ ]:
## Among EBV specificities, are there antigens with different kinetics?

## Extract score values per cell and plot as a box plot per specificity
obs_df = mdata["gex"].obs[["Virus", "PB-biased score", "clone_handle", "Antigen"]].copy()

# Short list antigens based on how many clones we observed. Only include antigens with more than 5 independent responses
temp_ag_count = obs_df[obs_df["Virus"].isin(["EBV", "CMV", "SARS-COV-2", "FLU-A"])][["Antigen", "clone_handle"]].dropna().drop_duplicates()[["Antigen"]].value_counts().reset_index()
ag_shortlist = temp_ag_count[temp_ag_count["count"] > 5]["Antigen"].tolist()

#Filter to include only the most promising antigens
obs_df = obs_df[obs_df["Antigen"].isin(ag_shortlist)]
obs_df = obs_df.dropna(subset="Antigen")

#Calculate mean PB-biased scores per clone
obs_df = obs_df.groupby(["clone_handle", "Antigen", "Virus"], observed=True).mean("PB-biased score").reset_index()

# Goal: Calculate percentile ranks for each clone and display the cummulative % sum of clones belonging to each Antigen category on the y-axis going from the top percentile to the lowest percentile on the x-axis. 
#Sort the dataframe first:
obs_df = obs_df.sort_values("PB-biased score", ascending=False)

# Calculate percentile rank for each clone's PB-biased score
# percentile_rank gives the percentage of scores <= current score
obs_df["percentile_rank"] = obs_df["PB-biased score"].rank(pct=True) * 100

# # Compute cumulative % of clones per Virus group along the ranked list
ag_groups = obs_df["Antigen"].unique()
ag_counts = obs_df["Antigen"].value_counts()  # total clones per virus

fig, ax = plt.subplots(figsize=(3, 3))

for ag in ag_groups:
    # Boolean mask: is each ranked clone from this virus?
    is_ag = (obs_df["Antigen"] == ag).astype(int)
    
    # Cumulative sum normalized by total clones of that virus → cumulative %
    cumulative_pct = is_ag.cumsum() / ag_counts[ag] * 100
    
    ax.plot(obs_df["percentile_rank"], cumulative_pct, label=ag)

ax.set_xlabel("PB-biased Score Percentile (High → Low)")
ax.set_ylabel("Proportion of clones")
ax.set_title("For antigens with more than 5 independent clones")
ax.invert_xaxis()  # x-axis goes from top percentile (100) to lowest (0)
ax.legend()
plt.grid(False)

plt.tight_layout()
plt.savefig("../../05_plots/09_multimers/18_cummulative_over_clones_and_antigens.pdf", format="pdf", dpi=400)
plt.show()

In [ ]:
### obs_df from before
obs_df["Antigen"] = obs_df["Antigen"].cat.remove_unused_categories()

#plotting order: 
# order = (
#     obs_df
#     .groupby("Antigen")["PB-biased score"]
#     .median()
#     .sort_values()
#     .index
# )

# Create the figure object
fig, ax = plt.subplots(figsize=(3.55,3))

# Stripplot (background dots)
sb.stripplot(
    data=obs_df,
    x="Antigen",
    y="PB-biased score",
    jitter=0.25,
    size=4,
    alpha=0.7,
    color="black",
    zorder=1,
    order=csf_order,
    ax=ax
)

# Boxplot (foreground)
sb.boxplot(
    data=obs_df,
    x="Antigen",
    y="PB-biased score",
    width=0.5,
    color="white",
    fliersize=0,
    boxprops=dict(alpha=.5),
    linewidth=1,
    zorder=2,
    order=csf_order,
    ax=ax
)

plt.ylabel("Mean PB-biased score\nper clone")
plt.xlabel("Antigen")
plt.xticks(rotation=90)
sb.despine()

plt.tight_layout()
fig.savefig("../../05_plots/09_multimers/12_mean_clone_PB_biased_score_per_antigen.pdf", bbox_inches='tight')
plt.show()

# Statistics ---------------------------------------------------------------
dunn = sp.posthoc_dunn(
    obs_df,
    val_col="PB-biased score",
    group_col="Antigen",
    p_adjust="fdr_bh",
)

# Convert pairwise matrix to long format
dunn_long = (
    dunn
    .rename_axis(index="Virus_1", columns="Virus_2")
    .stack()
    .rename("adjusted_p")
    .reset_index()
)

# Remove self-comparisons
dunn_long = dunn_long[
    dunn_long["Virus_1"] != dunn_long["Virus_2"]
].copy()

# Remove duplicate comparisons
dunn_long["comparison"] = dunn_long.apply(
    lambda row: tuple(sorted([row["Virus_1"], row["Virus_2"]])),
    axis=1,
)

dunn_long = (
    dunn_long
    .drop_duplicates(subset="comparison")
    .drop(columns="comparison")
    .sort_values("adjusted_p")
    .reset_index(drop=True)
)

dunn_long["significance"] = pd.cut(
    dunn_long["adjusted_p"],
    bins=[-float("inf"), 0.0001, 0.001, 0.01, 0.05, float("inf")],
    labels=["****", "***", "**", "*", "ns"],
)

dunn_long.head(20)

In [ ]:
# Clean data
df_clean = obs_df.dropna(subset=['Antigen', 'PB-biased score'])

# Define the order
order_ag = csf_order

# Perform Dunn's test with FDR correction
dunn_results = sp.posthoc_dunn(
    df_clean, 
    val_col='PB-biased score', 
    group_col='Antigen', 
    p_adjust='fdr_bh'
)

# Reindex both rows and columns to match your desired order
dunn_results = dunn_results.reindex(index=order_ag, columns=order_ag)

# Convert p-values to -log10 for heatmap coloring
heatmap_data = -np.log10(dunn_results)
heatmap_data = heatmap_data.clip(upper=10)  # clip extreme values

# Function to convert p-value to stars
def pval_to_stars(p):
    if p <= 0.0001:
        return "****"
    elif p <= 0.001:
        return "***"
    elif p <= 0.01:
        return "**"
    elif p <= 0.05:
        return "*"
    else:
        return "ns"

# Apply function to create annotation matrix
stars_df = dunn_results.applymap(pval_to_stars)

stars_df

## GZMK

In [ ]:
# Work with clones of interest (those with an annotated antigen-specificity)
mtemp = mdata.copy()
print(len(mtemp["gex"]))
mu.pp.filter_obs(mtemp["gex"], "Antigen", lambda x: (~x.isna()))
# mu.pp.filter_obs(mtemp["gex"], "donor", lambda x: (~x.isin(["Pt4", "Pt5"])))
print(len(mtemp["gex"]))

# Remove unused categories
for col in mtemp["gex"].obs.select_dtypes(include="category").columns:
    mtemp["gex"].obs[col] = (
        mtemp["gex"].obs[col].cat.remove_unused_categories()
    )

## Transfer GZMK to .obs
# Get index of GZMK
gene_idx = mtemp["gex"].var_names.get_loc("GZMK")

# Extract expression (handle sparse matrices safely)
gzmk_expr = mtemp["gex"].X[:, gene_idx]

# Convert to dense if needed
if not isinstance(gzmk_expr, np.ndarray):
    gzmk_expr = gzmk_expr.toarray().flatten()
else:
    gzmk_expr = gzmk_expr.flatten()

# Add to obs
mtemp["gex"].obs["GZMK_norm"] = gzmk_expr

## Extract score values per cell and plot as a box plot per specificity
obs_df = mtemp["gex"].obs[["Virus", "GZMK_norm", "clone_handle", "Antigen"]].copy()

# Short list antigens based on how many clones we observed. Only include antigens with more than 5 independent responses
temp_ag_count = obs_df[obs_df["Virus"].isin(["EBV", "CMV", "SARS-COV-2", "FLU-A"])][["Antigen", "clone_handle"]].dropna().drop_duplicates()[["Antigen"]].value_counts().reset_index()
ag_shortlist = temp_ag_count[temp_ag_count["count"] > 5]["Antigen"].tolist()

#Filter to include only the most promising antigens
obs_df = obs_df[obs_df["Antigen"].isin(ag_shortlist)]
obs_df = obs_df.dropna(subset="Antigen")

#Calculate mean GZMK_norms per clone
obs_df = obs_df.groupby(["clone_handle", "Antigen", "Virus"], observed=True).mean("GZMK_norm").reset_index()

# Goal: Calculate percentile ranks for each clone and display the cummulative % sum of clones belonging to each Antigen category on the y-axis going from the top percentile to the lowest percentile on the x-axis. 
#Sort the dataframe first:
obs_df = obs_df.sort_values("GZMK_norm", ascending=False)

# Calculate percentile rank for each clone's GZMK_norm
# percentile_rank gives the percentage of scores <= current score
obs_df["percentile_rank"] = obs_df["GZMK_norm"].rank(pct=True) * 100

# # Compute cumulative % of clones per Virus group along the ranked list
ag_groups = obs_df["Antigen"].unique()
ag_counts = obs_df["Antigen"].value_counts()  # total clones per virus

fig, ax = plt.subplots(figsize=(3, 3))

for ag in ag_groups:
    # Boolean mask: is each ranked clone from this virus?
    is_ag = (obs_df["Antigen"] == ag).astype(int)
    
    # Cumulative sum normalized by total clones of that virus → cumulative %
    cumulative_pct = is_ag.cumsum() / ag_counts[ag] * 100
    
    ax.plot(obs_df["percentile_rank"], cumulative_pct, label=ag)

ax.set_xlabel("GZMK_norm Percentile (High → Low)")
ax.set_ylabel("Proportion of clones")
ax.set_title("For antigens with more than 5 independent clones")
ax.invert_xaxis()  # x-axis goes from top percentile (100) to lowest (0)
ax.legend()
plt.grid(False)

plt.tight_layout()
plt.savefig("../../05_plots/09_multimers/18_cummulative_over_clones_and_antigens.pdf", format="pdf", dpi=400)
plt.show()

In [ ]:
## Extract score values per cell and plot as a box plot per specificity
obs_df = mtemp["gex"].obs[["Antigen", "Virus", "clone_handle", "GZMK_norm"]].dropna().copy()

# Short list antigens based on how many clones we observed. Only include antigens with more than 5 independent responses
temp_ag_count = obs_df[obs_df["Virus"].isin(["EBV", "CMV", "SARS-COV-2", "FLU-A"])][["Antigen", "clone_handle"]].dropna().drop_duplicates()[["Antigen"]].value_counts().reset_index()
ag_shortlist = temp_ag_count[temp_ag_count["count"] > 5]["Antigen"].tolist()

#Filter to include only the most promising antigens
obs_df = obs_df[obs_df["Antigen"].isin(ag_shortlist)]
obs_df = obs_df.dropna(subset="Antigen")

obs_df = (
    obs_df
    .groupby(["Antigen", "clone_handle"])["GZMK_norm"]
    .mean()
    .reset_index()
)
obs_df = obs_df[~obs_df["GZMK_norm"].isna()]

#plotting order: 
order =  csf_order

fig, ax = plt.subplots(figsize=(3.55,3))

# Boxplot
sb.boxplot(
    data=obs_df,
    x="Antigen",
    y="GZMK_norm",
    width=0.5,
    color="white",
    linecolor="#4D4D4D", 
    linewidth=0.5,
    fliersize=0,
    zorder=2,
    order=order,
    ax=ax
)

# Set transparency of the boxes
for patch in ax.patches:
    patch.set_alpha(0.90)

# Stripplot
sb.stripplot(
    data=obs_df,
    x="Antigen",
    y="GZMK_norm",
    jitter=0.25,
    size=4,
    alpha=0.7,
    color="black",
    zorder=1,
    order=order,
    ax=ax
)



plt.ylabel("Mean GZMK expr.\n")
plt.xlabel("Antigen")
plt.xticks(rotation=90)
sb.despine()
plt.grid(False)

plt.tight_layout()
fig.savefig("../../05_plots/09_multimers/31_GZMK_virus_boxplots.pdf", format="pdf", dpi=400)
plt.show()

In [ ]:
import scipy.stats as stats
import scikit_posthocs as sp

groups = [
    group["GZMK_norm"].values
    for _, group in obs_df.groupby("Antigen", observed=True)
]

H, p = stats.kruskal(*groups)

print(f"Kruskal-Wallis H = {H:.3f}, p = {p:.3e}")

In [ ]:
dunn = sp.posthoc_dunn(
    obs_df,
    val_col="GZMK_norm",
    group_col="Antigen",
    p_adjust="fdr_bh",   # or "holm", "bonferroni"
)

dunn = dunn.loc[order, order]

# Convert pairwise matrix to long format
dunn_long = (
    dunn
    .rename_axis(index="Virus_1", columns="Virus_2")
    .stack()
    .rename("adjusted_p")
    .reset_index()
)

# Remove self-comparisons
dunn_long = dunn_long[dunn_long["Virus_1"] != dunn_long["Virus_2"]]

# Remove duplicate comparisons
dunn_long["comparison"] = dunn_long.apply(
    lambda x: tuple(sorted((x["Virus_1"], x["Virus_2"]))),
    axis=1,
)

dunn_long = (
    dunn_long
    .drop_duplicates("comparison")
    .drop(columns="comparison")
    .sort_values("adjusted_p")
)

dunn_long["significance"] = pd.cut(
    dunn_long["adjusted_p"],
    bins=[-float("inf"), 0.0001, 0.001, 0.01, 0.05, float("inf")],
    labels=["****", "***", "**", "*", "ns"],
)

dunn_long.head(10)

In [ ]:
# Clean data
df_clean = obs_df.dropna(subset=['Antigen', 'GZMK_norm'])

# Define the order
order_ag = order

# Perform Dunn's test with FDR correction
dunn_results = sp.posthoc_dunn(
    df_clean, 
    val_col='GZMK_norm', 
    group_col='Antigen', 
    p_adjust='fdr_bh'
)

# Reindex both rows and columns to match your desired order
dunn_results = dunn_results.reindex(index=order_ag, columns=order_ag)

# Convert p-values to -log10 for heatmap coloring
heatmap_data = -np.log10(dunn_results)
heatmap_data = heatmap_data.clip(upper=10)  # clip extreme values

# Function to convert p-value to stars
def pval_to_stars(p):
    if p <= 0.0001:
        return "****"
    elif p <= 0.001:
        return "***"
    elif p <= 0.01:
        return "**"
    elif p <= 0.05:
        return "*"
    else:
        return "ns"

# Apply function to create annotation matrix
stars_df = dunn_results.applymap(pval_to_stars)

stars_df

## CXCR3

In [ ]:
# Work with clones of interest (those with an annotated antigen-specificity)
mtemp = mdata.copy()
print(len(mtemp["gex"]))
mu.pp.filter_obs(mtemp["gex"], "Antigen", lambda x: (~x.isna()))
# mu.pp.filter_obs(mtemp["gex"], "donor", lambda x: (~x.isin(["Pt4", "Pt5"])))
print(len(mtemp["gex"]))

# Remove unused categories
for col in mtemp["gex"].obs.select_dtypes(include="category").columns:
    mtemp["gex"].obs[col] = (
        mtemp["gex"].obs[col].cat.remove_unused_categories()
    )

## Transfer GZMK to .obs
# Get index of GZMK
gene_idx = mtemp["gex"].var_names.get_loc("CXCR3")

# Extract expression (handle sparse matrices safely)
gzmk_expr = mtemp["gex"].X[:, gene_idx]

# Convert to dense if needed
if not isinstance(gzmk_expr, np.ndarray):
    gzmk_expr = gzmk_expr.toarray().flatten()
else:
    gzmk_expr = gzmk_expr.flatten()

# Add to obs
mtemp["gex"].obs["GZMK_norm"] = gzmk_expr

## Extract score values per cell and plot as a box plot per specificity
obs_df = mtemp["gex"].obs[["Virus", "GZMK_norm", "clone_handle", "Antigen"]].copy()

# Short list antigens based on how many clones we observed. Only include antigens with more than 5 independent responses
temp_ag_count = obs_df[obs_df["Virus"].isin(["EBV", "CMV", "SARS-COV-2", "FLU-A"])][["Antigen", "clone_handle"]].dropna().drop_duplicates()[["Antigen"]].value_counts().reset_index()
ag_shortlist = temp_ag_count[temp_ag_count["count"] > 5]["Antigen"].tolist()

#Filter to include only the most promising antigens
obs_df = obs_df[obs_df["Antigen"].isin(ag_shortlist)]
obs_df = obs_df.dropna(subset="Antigen")

#Calculate mean GZMK_norms per clone
obs_df = obs_df.groupby(["clone_handle", "Antigen", "Virus"], observed=True).mean("GZMK_norm").reset_index()

# Goal: Calculate percentile ranks for each clone and display the cummulative % sum of clones belonging to each Antigen category on the y-axis going from the top percentile to the lowest percentile on the x-axis. 
#Sort the dataframe first:
obs_df = obs_df.sort_values("GZMK_norm", ascending=False)

# Calculate percentile rank for each clone's GZMK_norm
# percentile_rank gives the percentage of scores <= current score
obs_df["percentile_rank"] = obs_df["GZMK_norm"].rank(pct=True) * 100

# # Compute cumulative % of clones per Virus group along the ranked list
ag_groups = obs_df["Antigen"].unique()
ag_counts = obs_df["Antigen"].value_counts()  # total clones per virus

fig, ax = plt.subplots(figsize=(3, 3))

for ag in ag_groups:
    # Boolean mask: is each ranked clone from this virus?
    is_ag = (obs_df["Antigen"] == ag).astype(int)
    
    # Cumulative sum normalized by total clones of that virus → cumulative %
    cumulative_pct = is_ag.cumsum() / ag_counts[ag] * 100
    
    ax.plot(obs_df["percentile_rank"], cumulative_pct, label=ag)

ax.set_xlabel("CXCR3_norm Percentile (High → Low)")
ax.set_ylabel("Proportion of clones")
ax.set_title("For antigens with more than 5 independent clones")
ax.invert_xaxis()  # x-axis goes from top percentile (100) to lowest (0)
ax.legend()
plt.grid(False)

plt.tight_layout()
plt.savefig("../../05_plots/09_multimers/18_cummulative_over_clones_and_antigens.pdf", format="pdf", dpi=400)
plt.show()

In [ ]:
## Extract score values per cell and plot as a box plot per specificity
obs_df = mtemp["gex"].obs[["Antigen", "Virus", "clone_handle", "GZMK_norm"]].dropna().copy()

# Short list antigens based on how many clones we observed. Only include antigens with more than 5 independent responses
temp_ag_count = obs_df[obs_df["Virus"].isin(["EBV", "CMV", "SARS-COV-2", "FLU-A"])][["Antigen", "clone_handle"]].dropna().drop_duplicates()[["Antigen"]].value_counts().reset_index()
ag_shortlist = temp_ag_count[temp_ag_count["count"] > 5]["Antigen"].tolist()

#Filter to include only the most promising antigens
obs_df = obs_df[obs_df["Antigen"].isin(ag_shortlist)]
obs_df = obs_df.dropna(subset="Antigen")

obs_df = (
    obs_df
    .groupby(["Antigen", "clone_handle"])["GZMK_norm"]
    .mean()
    .reset_index()
)
obs_df = obs_df[~obs_df["GZMK_norm"].isna()]

#plotting order: 
order =  csf_order

fig, ax = plt.subplots(figsize=(3.55,3))

# Boxplot
sb.boxplot(
    data=obs_df,
    x="Antigen",
    y="GZMK_norm",
    width=0.5,
    color="white",
    linecolor="#4D4D4D", 
    linewidth=0.5,
    fliersize=0,
    zorder=2,
    order=order,
    ax=ax
)

# Set transparency of the boxes
for patch in ax.patches:
    patch.set_alpha(0.90)

# Stripplot
sb.stripplot(
    data=obs_df,
    x="Antigen",
    y="GZMK_norm",
    jitter=0.25,
    size=4,
    alpha=0.7,
    color="black",
    zorder=1,
    order=order,
    ax=ax
)



plt.ylabel("Mean CXCR3 expr.\n")
plt.xlabel("Antigen")
plt.xticks(rotation=90)
plt.ylim(-0.1, 1.7)
sb.despine()
plt.grid(False)

plt.tight_layout()
fig.savefig("../../05_plots/09_multimers/31_CXCR3_virus_boxplots.pdf", format="pdf", dpi=400)
plt.show()

In [ ]:
import scipy.stats as stats
import scikit_posthocs as sp

groups = [
    group["GZMK_norm"].values
    for _, group in obs_df.groupby("Antigen", observed=True)
]

H, p = stats.kruskal(*groups)

print(f"Kruskal-Wallis H = {H:.3f}, p = {p:.3e}")

In [ ]:
dunn = sp.posthoc_dunn(
    obs_df,
    val_col="GZMK_norm",
    group_col="Antigen",
    p_adjust="fdr_bh",   # or "holm", "bonferroni"
)

dunn = dunn.loc[order, order]

# Convert pairwise matrix to long format
dunn_long = (
    dunn
    .rename_axis(index="Virus_1", columns="Virus_2")
    .stack()
    .rename("adjusted_p")
    .reset_index()
)

# Remove self-comparisons
dunn_long = dunn_long[dunn_long["Virus_1"] != dunn_long["Virus_2"]]

# Remove duplicate comparisons
dunn_long["comparison"] = dunn_long.apply(
    lambda x: tuple(sorted((x["Virus_1"], x["Virus_2"]))),
    axis=1,
)

dunn_long = (
    dunn_long
    .drop_duplicates("comparison")
    .drop(columns="comparison")
    .sort_values("adjusted_p")
)

dunn_long["significance"] = pd.cut(
    dunn_long["adjusted_p"],
    bins=[-float("inf"), 0.0001, 0.001, 0.01, 0.05, float("inf")],
    labels=["****", "***", "**", "*", "ns"],
)

dunn_long.head(10)

In [ ]:
# Clean data
df_clean = obs_df.dropna(subset=['Antigen', 'GZMK_norm'])

# Define the order
order_ag = order

# Perform Dunn's test with FDR correction
dunn_results = sp.posthoc_dunn(
    df_clean, 
    val_col='GZMK_norm', 
    group_col='Antigen', 
    p_adjust='fdr_bh'
)

# Reindex both rows and columns to match your desired order
dunn_results = dunn_results.reindex(index=order_ag, columns=order_ag)

# Convert p-values to -log10 for heatmap coloring
heatmap_data = -np.log10(dunn_results)
heatmap_data = heatmap_data.clip(upper=10)  # clip extreme values

# Function to convert p-value to stars
def pval_to_stars(p):
    if p <= 0.0001:
        return "****"
    elif p <= 0.001:
        return "***"
    elif p <= 0.01:
        return "**"
    elif p <= 0.05:
        return "*"
    else:
        return "ns"

# Apply function to create annotation matrix
stars_df = dunn_results.applymap(pval_to_stars)

stars_df

# Quantify % frequencies of CD8 T cells

We will use a principle from SRH lab to extrapolate the estimated frequency from sorted cells. We will take the % frequency of antigen-specific T cells observed in the read-based data and extrapolate by using the observed % of multimer-pool+ CD8 T cells

In [ ]:
## Extract Virus and handle per cell
obs_df = mdata["gex"].obs[["donor","Virus", "handle","catsort_A2", "catsort_virus",  "total_agspecific_cells", "catagspecifc"]].copy()

## Remove all non-specific cells
obs_df = obs_df[obs_df["catagspecifc"] == "agspecific"]

## Calculate multimer frequency among antigen-specific cells for each donor
obs_df["multimer_cellcout"] = obs_df.groupby(["donor", "handle"], observed=True).transform("size")
obs_df["mult_frac_agsp"]= obs_df["multimer_cellcout"] / obs_df["total_agspecific_cells"]

## Load the total freq. of multimer+ cells
# excel_df = pd.read_excel("../../03_annotation/11_multimer_experiment_ann/2026_03 - FlowFreq_FromSortGates.xlsx")
excel_df = pd.read_excel("../../03_annotation/11_multimer_experiment_ann/2026_03 - FlowFreq_FromRegate.xlsx")
excel_df = pd.melt(excel_df, id_vars=["donor", "catsort_A2"], var_name="catsort_virus", value_vars=["EBV_sort_gate_freq", "Non_EBV_sort_gate_freq"], value_name="sort_freq")

#Remove  "_gate_freq" from EBV_sort and Non_EBV_sort column. 
excel_df["catsort_virus"] = excel_df["catsort_virus"].str.replace("_gate_freq", "", regex=False)

## Remove hidden whitespace
excel_df["catsort_virus"] = excel_df["catsort_virus"].str.strip()
excel_df["catsort_A2"] = excel_df["catsort_A2"].str.strip()

## Merge the two data frames
obs_df = obs_df.merge(excel_df, how="left", on=["donor", "catsort_A2", "catsort_virus"])

## Take the fraction (mult_frac_agspc) of sort_freq to extrapolate the estimated frequency
obs_df["Est_freq"] = obs_df["sort_freq"]*obs_df["mult_frac_agsp"]

## Drop na - these likely belongs to FRC responses and other excluded CMV-responses
obs_df = obs_df.dropna(subset="mult_frac_agsp").drop_duplicates()

## Catpat
# Group annotation
patient_list = ["Pt11", "Pt12", "Pt14", "Pt17", "Pt20"]
control_list = ["Pt1", "Pt8", "Pt18", "Pt19", "Pt21"]

# Define RRMS vs Control
obs_df["catpat"] = pd.Categorical(
    np.where(
        obs_df["donor"].isin(patient_list), "RRMS",
        np.where(obs_df["donor"].isin(control_list), "Control", "IC")
    ),
    categories=["Control", "IC", "RRMS"],
    ordered=True
)


palette = {
    "RRMS": "#000000",
    "IC" : "grey", 
    "Control": "#FFFFFF",
}

#Plot Est_freq
# Create the figure object
fig, ax = plt.subplots(figsize=(10,4))

# Boxplot (foreground)
sb.boxplot(
    data=obs_df,
    x="Virus",
    y="Est_freq",
    hue="catpat",
    width=0.75,
    color="white",
    showcaps=True,
    boxprops={'facecolor': 'white'},
    showfliers=False,
    palette=palette,
    ax=ax
)

# Stripplot (background dots)
sb.stripplot(
    data=obs_df,
    x="Virus",
    y="Est_freq",
    hue="catpat",
    dodge=True,
    s=5,
    jitter=True,
    alpha=0.8,
    linewidth=0.5,
    edgecolor="black",
    palette=palette,
    ax=ax
)

ax.set_yscale("log")
plt.ylabel("Estimated frequency (%)")
plt.xlabel("Virus")
plt.xticks(rotation=90)
plt.title("Each dot is one multimer+ population")

# Remove duplicate legend and move the legend
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles[3:6], labels[3:6], title="Group")
sb.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))

#Despine
sb.despine()

plt.tight_layout()
# fig.savefig("../../05_plots/09_multimers/7_mean_clone_CSF_biased_score.pdf", bbox_inches='tight')
plt.show()

# # Kruskal–Wallis global test
# groups = [
#     g["Est_freq"].values
#     for _, g in obs_df.groupby("catpat")
# ]

# H, p_kw = stats.kruskal(*groups)
# print("Kruskal-Wallis p =", p_kw)

# # Dunn's post-hoc test
# dunn_df = sp.posthoc_dunn(
#     obs_df,
#     val_col="Est_freq",
#     group_col="catpat",
#     p_adjust="fdr_bh"
# )

# print(dunn_df)

In [ ]:
## Extract Virus and handle per cell
obs_df = mdata["gex"].obs[["donor", "Virus", "handle","catsort_A2", "catsort_virus",  "total_agspecific_cells", "catagspecifc"]].copy()

## Remove all non-specific cells
obs_df = obs_df[obs_df["catagspecifc"] == "agspecific"]

## Calculate multimer frequency among antigen-specific cells for each donor
obs_df["multimer_cellcout"] = obs_df.groupby(["donor", "handle"], observed=True).transform("size")
obs_df["mult_frac_agsp"]= obs_df["multimer_cellcout"] / obs_df["total_agspecific_cells"]

## Load the total freq. of multimer+ cells
# excel_df = pd.read_excel("../../03_annotation/11_multimer_experiment_ann/2026_03 - FlowFreq_FromSortGates.xlsx")
excel_df = pd.read_excel("../../03_annotation/11_multimer_experiment_ann/2026_03 - FlowFreq_FromRegate.xlsx")
excel_df = pd.melt(excel_df, id_vars=["donor", "catsort_A2"], var_name="catsort_virus", value_vars=["EBV_sort_gate_freq", "Non_EBV_sort_gate_freq"], value_name="sort_freq")

#Remove  "_gate_freq" from EBV_sort and Non_EBV_sort column. 
excel_df["catsort_virus"] = excel_df["catsort_virus"].str.replace("_gate_freq", "", regex=False)

## Remove hidden whitespace
excel_df["catsort_virus"] = excel_df["catsort_virus"].str.strip()
excel_df["catsort_A2"] = excel_df["catsort_A2"].str.strip()

## Merge the two data frames
obs_df = obs_df.merge(excel_df, how="left", on=["donor", "catsort_A2", "catsort_virus"])

## Take the fraction (mult_frac_agspc) of sort_freq to extrapolate the estimated frequency
obs_df["Est_freq"] = obs_df["sort_freq"]*obs_df["mult_frac_agsp"]

## Drop na - these likely belongs to FRC responses and other excluded CMV-responses
obs_df = obs_df.dropna(subset="mult_frac_agsp").drop_duplicates()

# #Filter on EBV
# obs_df = obs_df[obs_df["Virus"] == "EBV"]
# obs_df["Virus"] = obs_df["Virus"].cat.remove_unused_categories()

## Catpat
# Group annotation
patient_list = ["Pt11", "Pt12", "Pt14", "Pt17", "Pt20"]
control_list = ["Pt1", "Pt8", "Pt18", "Pt19", "Pt21"]

# Define RRMS vs Control
obs_df["catpat"] = pd.Categorical(
    np.where(
        obs_df["donor"].isin(patient_list), "RRMS",
        np.where(obs_df["donor"].isin(control_list), "Control", "IC")
    ),
    categories=["Control", "IC", "RRMS"],
    ordered=True
)

# #Remove ICs to be consistent
# obs_df = obs_df[obs_df["catpat"] != "IC"]
# obs_df["catpat"] = obs_df["catpat"].cat.remove_unused_categories()

palette = {
    "RRMS": "#000000",
    "IC" : "grey", 
    "Control": "#FFFFFF",
}

#Plot Est_freq
# Create the figure object
fig, ax = plt.subplots(figsize=(6,4))

# Boxplot (foreground)
sb.boxplot(
    data=obs_df,
    x="Virus",
    y="Est_freq",
    hue="catpat",
    width=0.75,
    color="white",
    showcaps=True,
    boxprops={'facecolor': 'white'},
    showfliers=False,
    palette=palette,
    ax=ax
)

# Stripplot (background dots)
sb.stripplot(
    data=obs_df,
    x="Virus",
    y="Est_freq",
    hue="catpat",
    dodge=True,
    s=5,
    jitter=True,
    alpha=0.8,
    linewidth=0.5,
    edgecolor="black",
    palette=palette,
    ax=ax
)

ax.set_yscale("log")
plt.ylabel("Estimated frequency (%)")
plt.xlabel("Virus")
plt.xticks(rotation=90)
plt.title("Each dot is one multimer+ population")

# Remove duplicate legend and move the legend
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles[2:4], labels[2:4], title="Group")
sb.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))

#Despine
sb.despine()

plt.tight_layout()
fig.savefig("../../05_plots/09_multimers/27_EstFreq_CtrlMS.pdf", bbox_inches='tight')
plt.show()

# # Kruskal–Wallis global test
# groups = [
#     g["Est_freq"].values
#     for _, g in obs_df.groupby("catpat")
# ]

# H, p_kw = stats.kruskal(*groups)
# print("Kruskal-Wallis p =", p_kw)

# # Dunn's post-hoc test
# dunn_df = sp.posthoc_dunn(
#     obs_df,
#     val_col="Est_freq",
#     group_col="catpat",
#     p_adjust="fdr_bh"
# )

# print(dunn_df)

# Extract values
ctrl_vals = obs_df.loc[obs_df["catpat"] == "Control", "Est_freq"]
rrms_vals = obs_df.loc[obs_df["catpat"] == "RRMS", "Est_freq"]

# Mann–Whitney test
stat, pval = mannwhitneyu(ctrl_vals, rrms_vals, alternative="two-sided")

print("Mann–Whitney p-value:", pval)

## Show % in CSF based on matched clones only

In [ ]:
# Derive a list of clones and antigen specificities
Transfer_df = mdata["gex"].obs[["clone_handle", "handle", "Virus", "HLA", "Antigen", "Sequence"]].dropna(subset="Sequence").copy()


In [ ]:
sdata = mu.read(filename="..\\..\\04_data_objects\\01_preprocessed_scseq_scanpy\\2025_12 - with EBV collected\\2025_12_CD8_annotated.h5mu")

DEG_biased = pd.read_csv("../../06_csv_outputs/DEG/CSF_permisiveness_score.csv")["names"].drop_duplicates()
print(len(DEG_biased))

# Cacluate a score
sc.tl.score_genes(sdata["gex"], 
                      DEG_biased,
                      score_name='CSF-biased score')

DEG_biased = pd.read_csv("../../06_csv_outputs/DEG/PB_resident_score.csv")["names"].drop_duplicates()
print(len(DEG_biased))
# Cacluate a score
sc.tl.score_genes(sdata["gex"], 
                      DEG_biased,
                      score_name='PB-biased score')


In [ ]:
# Build and apply dictionaries using the clean key and annotate
for col, new_col in [('handle', 'handle'), ('Virus', 'Virus'), ('HLA', 'HLA'), ("Antigen", "Antigen"), ("Sequence", "Sequence")]:
    mlt_dic = Transfer_df.set_index('clone_handle').to_dict()[col]
    sdata["gex"].obs[new_col] = (
        sdata["gex"].obs['clone_handle']
        .map(mlt_dic)
    )


In [ ]:
# Get categories
cats = sdata["gex"].obs["Antigen"].drop_duplicates().tolist()
n_cats = len(cats)

# Get tab20 colormap
cmap = plt.get_cmap("tab20")

# Extract colors (wrap around if >20 categories)
colors = [cmap(i % 20) for i in range(n_cats)]

# Convert to hex (Scanpy prefers this)
colors = [plt.matplotlib.colors.to_hex(c) for c in colors]

# Assign to AnnData
sdata["gex"].uns["Antigen_colors"] = colors

sc.settings.set_figure_params(figsize=(4.3, 4))
fig_umap = sc.pl.umap(sdata["gex"], color=["Antigen"], size = 20, legend_fontweight= 'bold', frameon= False ,legend_fontoutline=2, show=False,  cmap="RdBu_r", return_fig=True)
plt.show()

In [ ]:
# Antigen of interest
Ag_int = "rep"

# Get categories
cats = sdata["gex"].obs["Antigen"].cat.categories

# Create a color list (default everything light gray)
colors = ["lightgrey"] * len(cats)

# Set EBNA1 to black
colors[list(cats).index(Ag_int)] = "black"

# Assign back to AnnData
sdata["gex"].uns["Antigen_colors"] = colors


sc.settings.set_figure_params(figsize=(4.3, 4))
fig_umap = sc.pl.umap(sdata["gex"], color=["Antigen"], groups=[Ag_int], size = 20, legend_fontweight= 'bold', frameon= False ,legend_fontoutline=2, show=False,  cmap="RdBu_r", return_fig=True)
plt.show()


## How many clones were matched?

I want to show the general overlap between CSF and PB for:
- All CD8 T cells
- Expanded CD8 T cells
- CD8 TEM cells
- EBV-specific
- CMV-specific
- FLU-specific

In [ ]:
# Number of resdicovered CD8 T cell clonotypes
obs_df = sdata["gex"].obs.copy()

# obs_df = obs_df[~obs_df["donor"].isin(["Pt4", "Pt5"])]
obs_df["Virus"] = obs_df["Virus"].cat.remove_unused_categories()
sum_df = obs_df.dropna(subset="Virus")[["Virus", "clone_handle"]].drop_duplicates()

# --- Ensure categorical (important for full combinations) ---
sum_df["Virus"] = sum_df["Virus"].astype("category")

# --- Groupby + complete combinations ---
sum_df = (
    sum_df
    .groupby(["Virus"], observed=False)  # key line
    .size()
    .reset_index(name="count")
)

# Order viruses by counts
order = sum_df.sort_values("count", ascending=False)["Virus"]

# Plot
fig, ax = plt.subplots(figsize=(4,4))

sb.barplot(data=sum_df, x="Virus", y="count", ax=ax, order=order)

# Add count labels on top of bars
for bar in ax.patches:
    height = bar.get_height()
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        height,
        f"{int(height)}",
        ha="center",
        va="bottom",
        fontsize=11
    )

plt.xticks(rotation=90)
plt.ylabel("Clonotype count")
plt.xlabel("Virus")
plt.title("Resdicovered clones")

# Move legend to the right
# ax.legend(title="Compartment", bbox_to_anchor=(1.05, 1), loc="upper left")

sb.despine()
ax.grid(False)

plt.tight_layout()
fig.savefig("../../05_plots/09_multimers/36_Count_ResdiscoveredClones_MS_NIC.pdf")
plt.show()

In [ ]:
# Number of resdicovered CD8 T cell clonotypes
obs_df = sdata["gex"].obs.copy()

obs_df = obs_df[~obs_df["donor"].isin(["Pt4", "Pt5"])]
obs_df["Virus"] = obs_df["Virus"].cat.remove_unused_categories()
sum_df = obs_df.dropna(subset="Virus")[["Compartment", "Virus", "clone_handle"]].drop_duplicates()

# --- Ensure categorical (important for full combinations) ---
sum_df["Compartment"] = sum_df["Compartment"].astype("category")
sum_df["Virus"] = sum_df["Virus"].astype("category")

# --- Groupby + complete combinations ---
sum_df = (
    sum_df
    .groupby(["Compartment", "Virus"], observed=False)  # key line
    .size()
    .reset_index(name="count")
)

# colors
compartment_palette = {
    "CSF": "#4B8AC1",   # blue
    "PB":  "#C06A6A"    # red
}

# Plot
fig, ax = plt.subplots(figsize=(6,4.5))
sb.barplot(data=sum_df, x="Virus", y="count", hue="Compartment", palette=compartment_palette, ax=ax)

plt.xticks(rotation=90)
plt.ylabel("Clonotype count")
plt.xlabel("Virus")
plt.title("Resdicovered clones")

# Move legend to the right
ax.legend(title="Compartment", bbox_to_anchor=(1.05, 1), loc="upper left")

sb.despine()
ax.grid(False)

plt.tight_layout()
fig.savefig("../../05_plots/09_multimers/33_Count_ResdiscoveredClones_MS_NIC.pdf")
plt.show()
print(sum_df)

# "Biased" is an indication whether a given clone is skewed towards CSF. 
### So are EBV-specific clones in the original dataset more biased?

## CSF-biased % for CMV and EBV-specific clones

In [ ]:
## We start by calculating the clone-wise % of cells in PB and CSF for relevant donors
obs_df = sdata["gex"].obs.copy()
obs_df = obs_df[~obs_df["donor"].isin(["Pt4", "Pt5"])]

## summarize per clone_handle
sum_df = obs_df[["clone_handle", "Compartment", "Virus", "donor", "catpat"]].value_counts().reset_index()

## Pivot
sum_df = sum_df.pivot(index=["clone_handle", "Virus", "donor", "catpat"], columns="Compartment", values="count").fillna(0)
sum_df["total"] = sum_df.sum(axis=1)

## CSF percentage
sum_df["CSF_pct"] = 100*sum_df["CSF"] / sum_df["total"]

## Summary data for each expansion bins
sum_df = sum_df.reset_index()
df0 = sum_df[sum_df["total"] > 0][["clone_handle", "donor", "Virus", "CSF_pct", "catpat"]]
df0["source"] = "0"

df2 = sum_df[sum_df["total"] > 2][["clone_handle", "donor", "Virus", "CSF_pct", "catpat"]]
df2["source"] = "2"

df4 = sum_df[sum_df["total"] > 4][["clone_handle", "donor", "Virus", "CSF_pct", "catpat"]]
df4["source"] = "4"

df8 = sum_df[sum_df["total"] > 8][["clone_handle", "donor", "Virus", "CSF_pct", "catpat"]]
df8["source"] = "8"

# df10 = sum_df[sum_df["total"] > 10][["clone_handle", "donor", "Virus", "CSF_pct", "catpat"]]
# df10["source"] = "10"

# df16 = sum_df[sum_df["total"] > 16][["clone_handle", "donor", "Virus", "CSF_pct", "catpat"]]
# df16["source"] = "16"

## Bind these dataframes together
df_all = pd.concat([df0, df2, df4, df8], ignore_index=True)

## CMV and EBV
plot_df = df_all[df_all["Virus"].isin(["EBV", "CMV"])]
plot_df["Virus"] = plot_df["Virus"].cat.remove_unused_categories()

## Make categories
plot_df["catexpvir"] = plot_df["Virus"].astype(str) + "_" + plot_df["catpat"].astype(str)

# Ensure correct ordering (important!)
plot_df["source"] = pd.Categorical(
    plot_df["source"],
    categories=["0","2","4", "8"],
    ordered=True
)

## Palette
virus_palette = {
    "CMV" : "#4C72B0", 
    "EBV" : "#DD8452", 
    "FLU-A" : "#55A868", 
    "SARS-COV-2" : "#BCBD22"
}

#Initialize
fig, ax = plt.subplots(figsize=(6,4))

# Boxplot (instead of barplot)
sb.boxplot(
    data=plot_df,
    x="source",
    y="CSF_pct",
    hue="Virus",
    palette=virus_palette,
    dodge=True,
    fliersize=0,   # hide outliers (since dots show them)
    ax=ax
)

# Overlay dots (still dodged, still black)
sb.stripplot(
    data=plot_df,
    x="source",
    y="CSF_pct",
    hue="Virus",
    dodge=True,
    color="black",
    size=7,
    jitter=True,
    alpha=0.8,
    ax=ax
)

# Fix duplicated legend
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles[0:2], labels[0:2], title="Virus",  bbox_to_anchor=(1.05, 1), loc="upper left")

# Move legend to the right
# ax.set_title("Of matched clones")
ax.set_ylabel("% in CSF of a given clone")
ax.set_xlabel("Expansion threshold\n(n > X cells)")
ax.grid(False)
sb.despine()

## Statistics
stat_results = []

for src in plot_df["source"].cat.categories:
    sub = plot_df[plot_df["source"] == src]

    cmv = sub[sub["Virus"] == "CMV"]["CSF_pct"]
    ebv = sub[sub["Virus"] == "EBV"]["CSF_pct"]

    if len(cmv) > 0 and len(ebv) > 0:
        stat, p = mannwhitneyu(cmv, ebv, alternative="two-sided")
        stat_results.append((src, p))

def p_to_stars(p):
    if p < 0.0001:
        return "****"
    elif p < 0.001:
        return "***"
    elif p < 0.01:
        return "**"
    elif p < 0.05:
        return "*"
    else:
        return "ns"

#Correction for multiple hypothesis testing
from statsmodels.stats.multitest import multipletests

pvals = [p for _, p in stat_results]
reject, pvals_corr, _, _ = multipletests(pvals, method='fdr_bh')

stat_results = [(src, p_corr) for (src, _), p_corr in zip(stat_results, pvals_corr)]
print(pvals)
print(pd.DataFrame(stat_results, columns=["source", "p_adj"]))

# Get max y per group for positioning
y_max = plot_df.groupby("source")["CSF_pct"].max()

for i, (src, p) in enumerate(stat_results):
    x = i  # categorical index

    y = y_max[src] + 5  # offset above max
    h = 3               # height of bracket

    # positions of CMV and EBV boxes
    x1 = x - 0.2
    x2 = x + 0.2

    # draw bracket
    ax.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.5, c="black")

    # add text
    ax.text(x, y + h + 1, p_to_stars(p), ha='center', va='bottom')

plt.tight_layout()
fig.savefig("../../05_plots/09_multimers/32_CSF_percentage_over_expansion.pdf", format="pdf", dpi=400)
plt.show()

# Of CSF-biased CD8 T cells, what what percentage of specificities can we explain?

In [ ]:
obs_df = sdata["gex"].obs.copy()
## Filter
obs_df = obs_df[~obs_df["donor"].isin(["Pt4", "Pt5"])]

## Filter by the CSF-biased clones
CSFbiased = pd.read_csv(r"..\..\06_csv_outputs\Clones\2025_6_CD8_CSF14_clonehandles_ForCSF.csv")["0"].tolist()
print(len(CSFbiased))
biased_clones = set(CSFbiased)
obs_df = obs_df[obs_df["clone_handle"].isin(biased_clones)]

## Summarize
sum_df = obs_df[["catpat","clone_handle", "Virus"]].drop_duplicates()
print(sum_df)
sum_df["Virus"] = np.where(sum_df["Virus"].astype(str) == "nan", "Unknown", sum_df["Virus"])
sum_df["Virus"] = pd.Categorical(sum_df["Virus"], categories=["CMV", "EBV", "FLU-A", "Unknown"], ordered=True)
sum_df = sum_df.groupby(["Virus","catpat"])["Virus"].size().reset_index(name="count")

# ## Percentage
sum_df = sum_df.pivot(index="catpat", columns="Virus", values="count")
sum_df["total"] = sum_df.sum(axis=1)
sum_df["EBV"] = sum_df["EBV"] / sum_df["total"]*100
sum_df["CMV"] = sum_df["CMV"] / sum_df["total"]*100
sum_df["FLU-A"] = sum_df["FLU-A"] / sum_df["total"]*100
sum_df["Unknown"] = sum_df["Unknown"] / sum_df["total"]*100

virus_palette = {
    "CMV" : "#4C72B0", 
    "EBV" : "#DD8452", 
    "FLU-A" : "#55A868", 
    "Unknown" : "#F5F5F5"
}

groups = sum_df.index  # Control, RRMS
viruses = ["CMV", "EBV", "FLU-A", "Unknown"]

x = np.arange(len(groups))
bottom = np.zeros(len(groups))

fig, ax = plt.subplots(figsize=(4, 4))

for virus in viruses:
    values = sum_df[virus].values

    plt.bar(
        x,
        values,
        bottom=bottom,
        label=virus,
        color=virus_palette[virus]   # 👈 key addition
    )

    bottom += values

plt.xticks(x, groups)
plt.ylabel("Percentage (%)")
plt.title("Virus distribution by group")
plt.ylim(0, 100)
plt.legend(title="Virus", bbox_to_anchor=(1.05, 1))
sb.despine()
plt.grid(False)
plt.tight_layout()
plt.savefig("../../05_plots/09_multimers/35_CEF_specificity_among_CSFbiased.pdf", format="pdf", dpi=400)
plt.show()

# B-LCL reactive clones

In [ ]:
#### annotate LCL reactive
LCL_list = pd.read_csv("../../06_csv_outputs/Clones/2026_4_BLCL_reactive_d19.csv")
LCL_list = LCL_list["Clone_handle"].drop_duplicates().tolist()

## Get obs
obs_df = sdata["gex"].obs.copy()

## annotate
obs_df["clone_handle2"] = obs_df["donor"].astype(str) + "_" + obs_df["IR_VDJ_1_junction_aa"].astype(str) 
obs_df["LCL_reactive"] = np.where(obs_df["clone_handle2"].isin(LCL_list), "LCL-reactive", "Non-reactive")

## Filter
obs_df = obs_df[~obs_df["donor"].isin(["Pt1", "Pt4", "Pt5", "Pt8", "Pt11", "Pt12"])]
CSFbiased = pd.read_csv(r"..\..\06_csv_outputs\Clones\2025_6_CD8_CSF14_clonehandles_ForCSF.csv")["0"].tolist()
PBbiased = pd.read_csv(r"..\..\06_csv_outputs\Clones\2025_6_CD8_PB14_clonehandles_ForCSF.csv")["0"].tolist()
# biased_clones = set(CSFbiased)
# obs_df = obs_df[obs_df["clone_handle"].isin(biased_clones)]

## Summarize with donor resolution
sum_df = obs_df[["catpat", "donor", "clone_handle", "LCL_reactive"]].drop_duplicates()
sum_df = sum_df.groupby(["LCL_reactive","donor", "catpat"], observed=True)["LCL_reactive"].size().reset_index(name="count")
sum_df = sum_df.pivot(index=["catpat", "donor"], columns="LCL_reactive", values="count").reset_index()
print(sum_df)


## Summarize
sum_df = obs_df[["catpat","clone_handle", "LCL_reactive"]].drop_duplicates()
sum_df = sum_df.groupby(["LCL_reactive","catpat"], observed=True)["LCL_reactive"].size().reset_index(name="count")
sum_df = sum_df.pivot(index="catpat", columns="LCL_reactive", values="count").reset_index()
print(sum_df)

## Convert to percentage
sum_df = sum_df.set_index("catpat")
sum_df["total"] = sum_df.sum(axis=1)
sum_df["LCL_pct"] = sum_df["LCL-reactive"] / sum_df["total"]*100
sum_df["NR"] = sum_df["Non-reactive"] / sum_df["total"]*100

# Plot
fig, ax = plt.subplots(figsize=(3, 5))

# X positions
groups = ["NIC", "MS"]
x = np.arange(len(groups))

# Bottom layer (reactive)
plt.bar(x, sum_df["LCL_pct"], label="LCL-reactive", color="grey")

# Labels and formatting
plt.xticks(x, groups)
plt.ylabel("% LCL-reactive clones")
# plt.title("Of CSF-biased clones")
sb.despine()
plt.grid(False)
plt.tight_layout()
fig.savefig("../../05_plots/09_multimers/34_LCL_reactive_in_main.pdf", format="pdf", dpi=400)
plt.show()

In [ ]:
#### annotate LCL reactive
LCL_list = pd.read_csv("../../06_csv_outputs/Clones/2026_4_BLCL_reactive_d19.csv")
LCL_list = LCL_list["Clone_handle"].drop_duplicates().tolist()

## Get obs
obs_df = sdata["gex"].obs.copy()

## annotate
obs_df["clone_handle2"] = obs_df["donor"].astype(str) + "_" + obs_df["IR_VDJ_1_junction_aa"].astype(str) 
obs_df["LCL_reactive"] = np.where(obs_df["clone_handle2"].isin(LCL_list), "LCL-reactive", "Non-reactive")

## Filter
obs_df = obs_df[~obs_df["donor"].isin(["Pt1", "Pt4", "Pt5", "Pt8", "Pt11", "Pt12"])]
CSFbiased = pd.read_csv(r"..\..\06_csv_outputs\Clones\2025_6_CD8_CSF14_clonehandles_ForCSF.csv")["0"].tolist()
PBbiased = pd.read_csv(r"..\..\06_csv_outputs\Clones\2025_6_CD8_PB14_clonehandles_ForCSF.csv")["0"].tolist()
obs_df["catbiased"] = np.where(obs_df["clone_handle"].isin(set(CSFbiased)), "CSF-biased", 
                               np.where(obs_df["clone_handle"].isin(set(PBbiased)), "PB-biased", "Unselected"))
obs_df = obs_df[obs_df["catbiased"].isin(["CSF-biased", "PB-biased"])]

## Summarize
sum_df = obs_df[["catpat","clone_handle", "LCL_reactive", "catbiased"]].drop_duplicates()
sum_df = sum_df.groupby(["LCL_reactive","catpat", "catbiased"])["LCL_reactive"].size().reset_index(name="count")
sum_df = sum_df.pivot(index=["catpat", "catbiased"], columns="LCL_reactive", values="count").reset_index()
print(sum_df)

## Convert to percentage
sum_df = sum_df.set_index(["catpat", "catbiased"])
sum_df["total"] = sum_df.sum(axis=1)
sum_df["LCL_pct"] = sum_df["LCL-reactive"] / sum_df["total"]*100
sum_df["NR"] = sum_df["Non-reactive"] / sum_df["total"]*100
print(sum_df)

In [ ]:
# colors
compartment_palette = {
    "Control": "#808080", #"#4B8AC1",   # blue
    "RRMS":  "#000000" #"#C06A6A"    # red
}

# Plot
fig, ax = plt.subplots(figsize=(5.5,5))
sb.barplot(data=sum_df, x="catbiased", y="LCL_pct", hue="catpat", palette=compartment_palette, ax=ax)

plt.xticks(rotation=0)
plt.ylabel("% LCL-reactive CD8 T-cell clones")
plt.xlabel(" ")
plt.ylim(0,40)
plt.title("NIC (n=3), MS (n=3)")

# Move legend to the right
ax.legend(title="Compartment", bbox_to_anchor=(1.05, 1), loc="upper left")

sb.despine()
ax.grid(False)

plt.tight_layout()
fig.savefig("../../05_plots/09_multimers/35_LCLreact_MSvsNIC.pdf")
plt.show()

In [ ]:
#### annotate LCL reactive
LCL_list = pd.read_csv("../../06_csv_outputs/Clones/2026_4_BLCL_reactive_d19.csv")
LCL_list = LCL_list["Clone_handle"].drop_duplicates().tolist()

## Get obs
obs_df = sdata["gex"].obs.copy()

## annotate
obs_df["clone_handle2"] = obs_df["donor"].astype(str) + "_" + obs_df["IR_VDJ_1_junction_aa"].astype(str) 
obs_df["LCL_reactive"] = np.where(obs_df["clone_handle2"].isin(LCL_list), "LCL-reactive", "Non-reactive")

## Filter
obs_df = obs_df[~obs_df["donor"].isin(["Pt1", "Pt4", "Pt5", "Pt8", "Pt11", "Pt12"])]
CSFbiased = pd.read_csv(r"..\..\06_csv_outputs\Clones\2025_6_CD8_CSF14_clonehandles_ForCSF.csv")["0"].tolist()
PBbiased = pd.read_csv(r"..\..\06_csv_outputs\Clones\2025_6_CD8_PB14_clonehandles_ForCSF.csv")["0"].tolist()
biased_clones = set(CSFbiased + PBbiased)
obs_df = obs_df[obs_df["clone_handle"].isin(biased_clones)]

obs_df[obs_df["LCL_reactive"] == "LCL-reactive"][["donor", "Compartment", "catpat", "clone_handle", "L3_ann", "IR_VJ_1_junction_aa", "IR_VDJ_1_junction_aa"]].value_counts().reset_index().sort_values("donor")

# Differentially expressed genes - EBV ctrl vs EBV MS

In [ ]:
sdata_filtered = sdata.copy()

# IMPORTANT TO SUBSET ONLY ON CELLS OF INTEREST- because the filter_rank_genes behaves odd if there are cells outside the groups (target and reference) that express a gene of interest. CCR7 kept getting filtered out because naive cells/other cells were part of the analysis
# mu.pp.filter_obs(sdata_pb_filtered["gex"], "top_vs_bot_clones", lambda x: (x != "unselected"))   #Going back to using memory cells as background for filtering

# Create a mask for genes to exclude
genes = sdata_filtered["gex"].var_names
exclude_mask = genes.str.startswith(("TRAV", "TRBV", "TRGV", "TRDV", "TRAJ", "TRBJ", "TRGJ", "TRDJ", "TRAC", "TRBC", "TRGC", "TRDC", "RPS", "RPL", "MT-"))

# Keep only genes not matching the exclusion
genes_to_keep = genes[~exclude_mask]
sdata_filtered = sdata_filtered["gex"][:, genes_to_keep]

In [ ]:
## Differential expression groups
sdata_filtered.obs["ag_catpat_compartment"] = sdata_filtered.obs["Virus"].astype(str) + "_" + sdata_filtered.obs["catpat"].astype(str) + "_" + sdata_filtered.obs["Compartment"].astype(str)

## EBV contrl vs EBV ms 
sdata_filtered.obs["ag_catpat_compartment"].value_counts()

## 
sdata_filtered.obs["ag_catpat_compartment"] = np.where(sdata_filtered.obs["ag_catpat_compartment"].isin(["CMV_RRMS_PB", "CMV_Control_PB"]), sdata_filtered.obs["ag_catpat_compartment"], "Unselected")

In [ ]:
# Perform naive differential gene expression
sc.tl.rank_genes_groups(sdata_filtered, groupby="ag_catpat_compartment", groups=["CMV_RRMS_PB"], reference="CMV_Control_PB", n_genes=50, key_added="CMV_MSvCtrl", method="wilcoxon")

In [ ]:
# Display initial results
sc.pl.rank_genes_groups_dotplot(
      sdata_filtered, groupby="ag_catpat_compartment", standard_scale="var", dendrogram=False, n_genes=50, key="CMV_MSvCtrl")

In [ ]:
## Differential expression groups
sdata_filtered.obs["ag_catpat_compartment"] = sdata_filtered.obs["Virus"].astype(str) + "_" + sdata_filtered.obs["catpat"].astype(str) + "_" + sdata_filtered.obs["Compartment"].astype(str)

## EBV contrl vs EBV ms 
sdata_filtered.obs["ag_catpat_compartment"].value_counts()

## 
sdata_filtered.obs["ag_catpat_compartment"] = np.where(sdata_filtered.obs["ag_catpat_compartment"].isin(["EBV_RRMS_PB", "EBV_Control_PB"]), sdata_filtered.obs["ag_catpat_compartment"], "Unselected")

In [ ]:
# Perform naive differential gene expression
sc.tl.rank_genes_groups(sdata_filtered, groupby="ag_catpat_compartment", groups=["EBV_RRMS_PB"], reference="EBV_Control_PB", n_genes=50, key_added="EBV_MSvCtrl", method="wilcoxon")
sc.tl.rank_genes_groups(sdata_filtered, groupby="ag_catpat_compartment", groups=["EBV_Control_PB"], reference="EBV_RRMS_PB", n_genes=50, key_added="EBV_CtrlvsMS", method="wilcoxon")

In [ ]:
# Display initial results
sc.pl.rank_genes_groups_dotplot(
      sdata_filtered, groupby="ag_catpat_compartment", standard_scale="var", dendrogram=False, n_genes=50, key="EBV_MSvCtrl")
sc.pl.rank_genes_groups_dotplot(
      sdata_filtered, groupby="ag_catpat_compartment", standard_scale="var", dendrogram=False, n_genes=50, key="EBV_CtrlvsMS")

In [ ]:
## Differential expression groups
sdata_filtered.obs["ag_catpat_compartment"] = sdata_filtered.obs["Virus"].astype(str) + "_" + sdata_filtered.obs["catpat"].astype(str) + "_" + sdata_filtered.obs["Compartment"].astype(str)

## EBV contrl vs EBV ms 
sdata_filtered.obs["ag_catpat_compartment"].value_counts()

## 
sdata_filtered.obs["ag_catpat_compartment"] = np.where(sdata_filtered.obs["ag_catpat_compartment"].isin(["EBV_RRMS_CSF", "EBV_Control_CSF"]), sdata_filtered.obs["ag_catpat_compartment"], "Unselected")

In [ ]:
# Perform naive differential gene expression
sc.tl.rank_genes_groups(sdata_filtered, groupby="ag_catpat_compartment", groups=["EBV_RRMS_CSF"], reference="EBV_Control_CSF", n_genes=50, key_added="EBV_MSvCtrl", method="wilcoxon")
sc.tl.rank_genes_groups(sdata_filtered, groupby="ag_catpat_compartment", groups=["EBV_Control_CSF"], reference="EBV_RRMS_CSF", n_genes=50, key_added="EBV_CtrlvsMS", method="wilcoxon")

In [ ]:
# Display initial results
sc.pl.rank_genes_groups_dotplot(
      sdata_filtered, groupby="ag_catpat_compartment", standard_scale="var", dendrogram=False, n_genes=50, key="EBV_MSvCtrl")
# Display initial results
sc.pl.rank_genes_groups_dotplot(
      sdata_filtered, groupby="ag_catpat_compartment", standard_scale="var", dendrogram=False, n_genes=50, key="EBV_CtrlvsMS")

# Number of clonotypes common between datasets (mult vs main)

In [ ]:
obs_df = sdata["gex"].obs.copy()
df = obs_df[["handle", "clone_handle", "Virus"]].value_counts().reset_index(name = "counts")
df = df[["Virus"]].value_counts().reset_index(name="counts")

# Order viruses by counts
order = df.sort_values("counts", ascending=False)["Virus"]


# Initialize the matplotlib figure
f, ax = plt.subplots(figsize=(3.2, 4))

# Plot
sb.set_color_codes("pastel")
sb.barplot(x="Virus", y="counts", data=df, order=order)

# Add count labels on top of bars
for bar in ax.patches:
    height = bar.get_height()
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        height,
        f"{int(height)}",
        ha="center",
        va="bottom",
        fontsize=11
    )


# Add a legend and informative axis label
ax.legend(ncol=1, loc="lower right", frameon=True)
ax.set(ylabel="Clonotypes in both dataset")
plt.xlabel("")
# plt.title("Clones present in both the main\n and multimer-sort datasets")
plt.xticks(rotation=90)
sb.despine()
plt.grid(False)
plt.tight_layout()
plt.savefig("../../05_plots/09_multimers/25_NumberOfMatchedClones.pdf")

## How many clones were present in CSF?

In [ ]:
obs_df = sdata["gex"].obs.copy()
obs_df = obs_df[obs_df["Compartment"] == "CSF"]
obs_df["Virus"] = obs_df["Virus"].cat.remove_unused_categories()
df = obs_df[["handle", "clone_handle", "Virus"]].value_counts().reset_index(name = "counts")
df = df[["Virus"]].value_counts().reset_index(name="counts")

# Order viruses by counts
order = df.sort_values("counts", ascending=False)["Virus"]

# Initialize the matplotlib figure
f, ax = plt.subplots(figsize=(3.2, 4))

# Plot
sb.set_color_codes("pastel")
sb.barplot(x="Virus", y="counts", data=df, order=order)

# Add count labels on top of bars
for bar in ax.patches:
    height = bar.get_height()
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        height,
        f"{int(height)}",
        ha="center",
        va="bottom",
        fontsize=11
    )


# Add a legend and informative axis label
ax.legend(ncol=1, loc="lower right", frameon=True)
ax.set(ylabel="Clonotypes in both dataset")
plt.xlabel("")
# plt.title("Clones present in both the main\n and multimer-sort datasets")
plt.xticks(rotation=90)
sb.despine()
plt.grid(False)
plt.tight_layout()
plt.savefig("../../05_plots/09_multimers/25_NumberOfMatchedClones_InCSF.pdf")

## Enrichment plot in CSF

In [ ]:
## We need all clonotypes matched with a multimer present in CSF, their CSF (%) and  their enrichment in CSF over PB (if they were event present in PB!? like how do you calculate enrichment for a 100% CSF clone?)
obs_df = sdata["gex"].obs.copy()

#Annotate catpat
patient_list = ["Pt11", "Pt12", "Pt14", "Pt17", "Pt20"]
control_list = ["Pt1", "Pt8", "Pt18", "Pt19", "Pt21"]

# Define RRMS vs Control
obs_df["catpat"] = pd.Categorical(
    np.where(
        obs_df["donor"].isin(patient_list), "RRMS",
        np.where(obs_df["donor"].isin(control_list), "Control", "IC")
    ),
    categories=["Control", "IC", "RRMS"],
    ordered=True
)

#First I need the total number of cells in each compartment for a given donor
obs_df["total_cells"] = obs_df.groupby(["donor", "Compartment"], observed=True)["Compartment"].transform("count")

## Make two dataframes with Compartment-wise percentages and join them
pb_df = obs_df[obs_df["Compartment"] == "PB"][["Virus","donor", "catpat", "clone_handle", "type_x_clone_count", "total_cells"]].drop_duplicates()
pb_df["PB_pct"] = 100*obs_df["type_x_clone_count"] / obs_df["total_cells"]
csf_df = obs_df[obs_df["Compartment"] == "CSF"][["Virus", "donor", "catpat", "clone_handle", "type_x_clone_count", "total_cells"]].drop_duplicates()
csf_df["CSF_pct"] = 100*obs_df["type_x_clone_count"] / obs_df["total_cells"]

## Merge these dataframes
mrg_df = csf_df.merge(pb_df[["donor", "clone_handle", "PB_pct"]], how="left")
mrg_df.sort_values("PB_pct", ascending=False)

## Save the donor-wise pct min and used something 10 times lower to signal that the clone was undetected in PB
#### First fill NA values with 0
mrg_df["PB_pct"] = mrg_df["PB_pct"].fillna(0)

### Find the minimum that is not 0
temp_min = mrg_df[mrg_df["PB_pct"] != 0][["donor", "PB_pct"]].drop_duplicates()
temp_min["PB_pct_min"] = temp_min.groupby("donor", observed=True)["PB_pct"].transform("min")
temp_min = temp_min[["donor", "PB_pct_min"]].drop_duplicates()

### Merge back
mrg_df = mrg_df.merge(temp_min, how="left")

### zoom in on multimer+ responses
mrg_df = mrg_df.dropna(subset="Virus")
mrg_df = mrg_df.dropna(subset="clone_handle")

## Calculate the %PB / %CSF fold enrichment. Adjust 0 for PB_pct with the minimum observed frequency in PB of any clone.
mrg_df["PB_pct"] = np.where(mrg_df["PB_pct"] == 0, mrg_df["PB_pct_min"], mrg_df["PB_pct"])
mrg_df["FC_CSF-PB"] = mrg_df["CSF_pct"] / mrg_df["PB_pct"]

fig, axes = plt.subplots(1, 2, figsize=(7, 3.5), sharex=True, sharey=True)

# Split data
df_rrms = mrg_df[mrg_df["catpat"] == "RRMS"]
df_ctrl = mrg_df[mrg_df["catpat"] == "IC"]

# --- RRMS ---
sb.scatterplot(
    data=df_rrms,
    x="CSF_pct",
    y="FC_CSF-PB",
    hue="Virus",
    alpha=0.8,
    s=100,
    ax=axes[0],
    legend=False   # 👈 prevent duplicate legend
)

axes[0].set_title("RRMS")

# --- Control ---
sb.scatterplot(
    data=df_ctrl,
    x="CSF_pct",
    y="FC_CSF-PB",
    hue="Virus",
    alpha=0.8,
    s=100,
    ax=axes[1],
    legend=True    # 👈 keep this one for global legend
)

axes[1].set_title("IC")

# --- Shared formatting ---
for ax in axes:
    ax.set_yscale("log")
    ax.set_xlabel("% of CSF")
    sb.despine(ax=ax)
    ax.grid(False)

axes[0].set_ylabel("Fold-change (%CSF / %PB)")
axes[1].set_ylabel("")

# --- Move single legend outside ---
sb.move_legend(axes[1], "upper left", bbox_to_anchor=(1, 1))


plt.savefig("../../05_plots/09_multimers/29_enrichment_plot_in_csf.pdf",
            format="pdf", dpi=400, bbox_inches="tight")

plt.show()

## Percentage of cells in CSF

In [ ]:
# -----------------------------
# Filter
# -----------------------------
obs_df = sdata["gex"].obs.dropna(subset=["Virus"])
obs_df = obs_df[~obs_df["donor"].isin(["Pt4", "Pt5"])]
obs_df = obs_df[obs_df["Virus"].isin(["CMV", "EBV"])]

# -----------------------------
# Create grouping variable
# -----------------------------
obs_df["x_cat"] = obs_df["Virus"].astype(str) + "_" + obs_df["catpat"].astype(str)

# -----------------------------
# Count cells per group
# -----------------------------
count_df = (
    obs_df
    .groupby(["x_cat", "Compartment"], observed=True)
    .size()
    .reset_index(name="count")
)

# Total per x_cat
count_df["total_cells"] = count_df.groupby("x_cat")["count"].transform("sum")

# Percentage
count_df["comp_pct"] = 100 * count_df["count"] / count_df["total_cells"]

# -----------------------------
# Pivot
# -----------------------------
plot_df = (
    count_df
    .pivot(index="x_cat", columns="Compartment", values="comp_pct")
    .fillna(0)
    .reset_index()
    .sort_values("x_cat")
)

# -----------------------------
# Plot
# -----------------------------
csf_color = "#4B8AC1"   # muted blue
pb_color  = "#C06A6A"   # muted orange
x = np.arange(len(plot_df))

fig, ax = plt.subplots(figsize=(6, 4))

# CSF (bottom)
ax.bar(
    x,
    plot_df["CSF"],
    color=csf_color,
    label="CSF"
)

# PB (correct stacking!)
ax.bar(
    x,
    plot_df["PB"],
    bottom=plot_df["CSF"],
    color=pb_color,
    label="PB"
)

ax.set_xticks(x)
ax.set_xticklabels(plot_df["x_cat"], rotation=45, ha="right")

ax.set_ylabel("% of cells")
ax.set_xlabel("Condition")
plt.title("% cells in each compartment")

ax.legend(frameon=False)
sb.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))

plt.tight_layout()
plt.show()

In [ ]:
# -----------------------------
# Filter
# -----------------------------
obs_df = sdata["gex"].obs.dropna(subset=["Virus"])
obs_df = obs_df[~obs_df["donor"].isin(["Pt4", "Pt5"])]
obs_df = obs_df[obs_df["Virus"].isin(["CMV", "EBV", "FLU-A"])]
obs_df["Virus"] = obs_df["Virus"].cat.remove_unused_categories()

# -----------------------------
# Count cells per group
# -----------------------------
count_df = (
    obs_df
    .groupby(["Virus", "donor", "Compartment"], observed=True)
    .size()
    .reset_index(name="count")
)

# Total per Virus
count_df["total_cells"] = count_df.groupby(["Virus","donor"], observed=True)["count"].transform("sum")

# Percentage
count_df["comp_pct"] = 100 * count_df["count"] / count_df["total_cells"]

# CSF percentage per donor
plot_df = count_df[count_df["Compartment"] == "CSF"]

fig, ax = plt.subplots(figsize=(2, 3.5))

sb.barplot(
    data=plot_df,
    x="Virus",
    y="comp_pct",
    errorbar="sd",   # <-- standard error
    capsize=0.2,
    order=["EBV", "CMV", "FLU-A"],
    ax=ax
)

sb.stripplot(
    data=plot_df,
    x="Virus",
    y="comp_pct",
    color="black",
    size=4,
    jitter=True,
    order=["EBV", "CMV", "FLU-A"],
    ax=ax
)

# Formatting
ax.set_xlabel(" ")
ax.set_ylabel("CSF (%)")
ax.set_title("CSF proportion per donor ± SD\namong cells from matched clonotypes")
sb.despine()
plt.grid(False)
plt.xticks(rotation=90)
plt.ylim(0,110)

plt.tight_layout()
plt.savefig("../../05_plots/09_multimers/19_percentage_csf_matched_cells_EBV_CMV.pdf")
plt.show()

In [ ]:
## Statistics - we need  to do it unpaired, because some donors did not have EBV responses, but had CMV repsonses ect. 

ebv = plot_df[plot_df["Virus"] == "EBV"]["comp_pct"]
cmv = plot_df[plot_df["Virus"] == "CMV"]["comp_pct"]

stat, pval = mannwhitneyu(ebv, cmv, alternative="two-sided")

print(pval)

In [ ]:
# -----------------------------
# Filter
# -----------------------------
obs_df = sdata["gex"].obs.dropna(subset=["Virus"])
obs_df = obs_df[~obs_df["donor"].isin(["Pt4", "Pt5"])]
obs_df = obs_df[obs_df["Virus"].isin(["CMV", "EBV", "FLU-A"])]

# -----------------------------
# Create grouping variable
# -----------------------------
obs_df["x_cat"] = obs_df["Virus"].astype(str) + "_" + obs_df["catpat"].astype(str)

# -----------------------------
# Count cells per group
# -----------------------------
count_df = (
    obs_df
    .groupby(["x_cat", "donor", "Compartment"], observed=True)
    .size()
    .reset_index(name="count")
)

# Total per x_cat
count_df["total_cells"] = count_df.groupby(["x_cat","donor"], observed=True)["count"].transform("sum")

# Percentage
count_df["comp_pct"] = 100 * count_df["count"] / count_df["total_cells"]

# CSF percentage per donor
plot_df = count_df[count_df["Compartment"] == "CSF"]

fig, ax = plt.subplots(figsize=(2, 3.5))

sb.barplot(
    data=plot_df,
    x="x_cat",
    y="comp_pct",
    errorbar="sd",   # <-- standard error
    capsize=0.2,
    order=["EBV_Control", "EBV_RRMS", "CMV_Control", "CMV_RRMS", "FLU-A_Control", "FLU-A_RRMS"], 
    ax=ax
)

sb.stripplot(
    data=plot_df,
    x="x_cat",
    y="comp_pct",
    color="black",
    size=3,
    jitter=True,
    ax=ax
)

# Formatting
ax.set_xlabel("Condition")
ax.set_ylabel("CSF (%)")
ax.set_title("CSF proportion per donor ± SD\namong cells from matched clonotypes")

# Despine and remove grid
sb.despine()
plt.grid(False)

# ticks
plt.xticks(rotation=90)
# Set specific y-axis ticks
ax.set_yticks([0, 50, 100])

plt.tight_layout()
plt.savefig("../../05_plots/09_multimers/20_MSvCtrl_CSF_percentage.pdf", format="pdf", dpi=400)
plt.show()

## % CSF per clone

In [ ]:
# -----------------------------
# Filter
# -----------------------------
obs_df = sdata["gex"].obs.dropna(subset=["Virus"])
obs_df = obs_df[~obs_df["donor"].isin(["Pt4", "Pt5"])]
obs_df = obs_df[obs_df["Virus"].isin(["CMV", "EBV"])]

# -----------------------------
# Create grouping variable
# -----------------------------
obs_df["x_cat"] = obs_df["Virus"].astype(str) + "_" + obs_df["catpat"].astype(str)

# -----------------------------
# Count cells per group
# -----------------------------
count_df = (
    obs_df
    .groupby(["x_cat", "donor", "Compartment", "clone_handle"], observed=True)
    .size()
    .reset_index(name="count")
)

# Total per x_cat
count_df["total_cells"] = count_df.groupby(["x_cat","donor", "clone_handle"], observed=True)["count"].transform("sum")

# Percentage
count_df["comp_pct"] = 100 * count_df["count"] / count_df["total_cells"]

# CSF percentage per donor
plot_df = count_df[count_df["Compartment"] == "CSF"]

fig, ax = plt.subplots(figsize=(1.5, 3.5))

sb.barplot(
    data=plot_df,
    x="x_cat",
    y="comp_pct",
    errorbar="sd",   # <-- standard error
    capsize=0.2,
    order=["EBV_Control", "EBV_RRMS", "CMV_Control", "CMV_RRMS"], 
    ax=ax
)

sb.stripplot(
    data=plot_df,
    x="x_cat",
    y="comp_pct",
    color="black",
    size=3,
    jitter=True,
    ax=ax
)

# Formatting
ax.set_xlabel("Condition")
ax.set_ylabel("CSF (%)")
ax.set_title("CSF proportion per donor ± SD\namong cells from matched clonotypes")
plt.ylim(0,100)

# Despine and remove grid
sb.despine()
plt.grid(False)

# ticks
plt.xticks(rotation=90)
# Set specific y-axis ticks
ax.set_yticks([0, 50, 100])

plt.tight_layout()
plt.savefig("../../05_plots/09_multimers/20_MSvCtrl_CSF_percentage_perclone.pdf", format="pdf", dpi=400)
plt.show()

## CSF-biased score per cell

In [ ]:
## Extract score values per cell and plot as a box plot per specificity
obs_df = mdata["gex"].obs[["Virus", "CSF-biased score"]].copy()
obs_df = obs_df.dropna(subset="Virus")

#plotting order: 
order = (
    obs_df
    .groupby("Virus")["CSF-biased score"]
    .median()
    .sort_values()
    .index
)

plt.figure(figsize=(4,4))
# Stripplot
sb.stripplot(
    data=obs_df,
    x="Virus",
    y="CSF-biased score",
    jitter=0.25,
    size=4,
    alpha=0.7,
    color="black",
    zorder=1,
    order=order
)

# Boxplot
sb.boxplot(
    data=obs_df,
    x="Virus",
    y="CSF-biased score",
    width=0.5,
    color="white",
    fliersize=0,
    linewidth=1,
    zorder=2,
    order=order
)
plt.title("Per cell CSF-biased score")
plt.ylabel("CSF-biased score")
plt.xlabel("Virus")
plt.xticks(rotation=90)
sb.despine()

plt.tight_layout()
plt.show()

In [ ]:
# Clean data
df_clean = obs_df.dropna(subset=['Virus', 'CSF-biased score'])

# Perform Dunn's test with FDR correction
dunn_results = sp.posthoc_dunn(
    df_clean, 
    val_col='CSF-biased score', 
    group_col='Virus', 
    p_adjust='fdr_bh'
)

# Convert p-values to -log10 for heatmap coloring
heatmap_data = -np.log10(dunn_results)
heatmap_data = heatmap_data.clip(upper=10)  # clip extreme values

# Function to convert p-value to stars
def pval_to_stars(p):
    if p <= 0.0001:
        return "****"
    elif p <= 0.001:
        return "***"
    elif p <= 0.01:
        return "**"
    elif p <= 0.05:
        return "*"
    else:
        return "ns"

# Apply function to create annotation matrix
stars_df = dunn_results.applymap(pval_to_stars)

# Mask upper triangle
mask = np.triu(np.ones_like(heatmap_data, dtype=bool))

# Plot heatmap with log10(p) coloring and stars
plt.figure(figsize=(8,6))
sb.heatmap(
    heatmap_data,
    mask=mask,
    annot=stars_df,
    fmt='',
    cmap="Reds",
    cbar_kws={'label': '-log10(p)'},
    linewidths=0.5,
    linecolor='gray'
)
plt.title("Dunn's Test Pairwise Significance")
plt.tight_layout()
plt.show()

## CSF-biased score per clone

In [ ]:
## Extract score values per cell and plot as a box plot per specificity
obs_df = mdata["gex"].obs[["Virus", "CSF-biased score", "clone_handle"]].copy()
obs_df = obs_df.dropna(subset="Virus")
obs_df = obs_df.groupby(["clone_handle", "Virus"], observed=True).mean("CSF-biased score").reset_index()

#plotting order: 
order = (
    obs_df
    .groupby("Virus")["CSF-biased score"]
    .median()
    .sort_values()
    .index
)

# Create the figure object
fig, ax = plt.subplots(figsize=(4,4))

# Stripplot (background dots)
sb.stripplot(
    data=obs_df,
    x="Virus",
    y="CSF-biased score",
    jitter=0.25,
    size=4,
    alpha=0.7,
    color="black",
    zorder=1,
    order=order,
    ax=ax
)

# Boxplot (foreground)
sb.boxplot(
    data=obs_df,
    x="Virus",
    y="CSF-biased score",
    width=0.5,
    color="white",
    fliersize=0,
    boxprops=dict(alpha=.5),
    linewidth=1,
    zorder=2,
    order=order,
    ax=ax
)

plt.ylabel("Mean CSF-biased score\nper clone")
plt.xlabel("Virus")
plt.xticks(rotation=90)
sb.despine()

plt.tight_layout()
fig.savefig("../../05_plots/09_multimers/7_mean_clone_CSF_biased_score.pdf", bbox_inches='tight')
plt.show()

In [ ]:
# Clean data
df_clean = obs_df.dropna(subset=['Virus', 'CSF-biased score'])

# Define the order
order_virus = ["NWV", "HMPV", "CMV", "HHV-1", "FLU-A", "SARS-COV-2", "HHV-6B", "HAdV-C", "EBV", "VZV"]

# Perform Dunn's test with FDR correction
dunn_results = sp.posthoc_dunn(
    df_clean, 
    val_col='CSF-biased score', 
    group_col='Virus', 
    p_adjust='fdr_bh'
)

# Reindex both rows and columns to match your desired order
dunn_results = dunn_results.reindex(index=order_virus, columns=order_virus)

# Convert p-values to -log10 for heatmap coloring
heatmap_data = -np.log10(dunn_results)
heatmap_data = heatmap_data.clip(upper=10)  # clip extreme values

# Function to convert p-value to stars
def pval_to_stars(p):
    if p <= 0.0001:
        return "****"
    elif p <= 0.001:
        return "***"
    elif p <= 0.01:
        return "**"
    elif p <= 0.05:
        return "*"
    else:
        return "ns"

# Apply function to create annotation matrix
stars_df = dunn_results.applymap(pval_to_stars)

# Mask upper triangle
mask = np.triu(np.ones_like(heatmap_data, dtype=bool))

# Plot heatmap with log10(p) coloring and stars
plt.figure(figsize=(10,8))
sb.heatmap(
    heatmap_data,
    mask=mask,
    annot=stars_df,
    fmt='',
    cmap="Reds",
    cbar_kws={'label': '-log10(p.adj)'},
    linewidths=0.5,
    linecolor='gray'
)
plt.title("Dunn's Test Pairwise Significance (Ordered Viruses)")
plt.tight_layout()
plt.show()

In [ ]:
### PB-biased score for completeness
## Extract score values per cell and plot as a box plot per specificity
obs_df = mdata["gex"].obs[["Virus", "PB-biased score", "clone_handle"]].copy()
obs_df = obs_df.dropna(subset="Virus")
obs_df = obs_df.groupby(["clone_handle", "Virus"], observed=True).mean("PB-biased score").reset_index()

#plotting order: 
order = (
    obs_df
    .groupby("Virus")["PB-biased score"]
    .median()
    .sort_values()
    .index
)

# Create the figure object
fig, ax = plt.subplots(figsize=(4.25,3.75))

# Stripplot (background dots)
sb.stripplot(
    data=obs_df,
    x="Virus",
    y="PB-biased score",
    jitter=0.25,
    size=4,
    alpha=0.7,
    color="black",
    zorder=1,
    order=order,
    ax=ax
)

# Boxplot (foreground)
sb.boxplot(
    data=obs_df,
    x="Virus",
    y="PB-biased score",
    width=0.5,
    color="white",
    fliersize=0,
    boxprops=dict(alpha=.5),
    linewidth=1,
    zorder=2,
    order=order,
    ax=ax
)

plt.ylabel("Mean PB-biased score\nper clone")
plt.xlabel("Virus")
plt.xticks(rotation=90)
sb.despine()

plt.tight_layout()
fig.savefig("../../05_plots/09_multimers/7_mean_clone_PB_biased_score.pdf", bbox_inches='tight')
plt.show()

In [ ]:
# Clean data
df_clean = obs_df.dropna(subset=['Virus', 'PB-biased score'])

# Define the order
order_virus = ["NWV", "HMPV", "CMV", "HHV-1", "FLU-A", "SARS-COV-2", "HHV-6B", "HAdV-C", "EBV", "VZV"]

# Perform Dunn's test with FDR correction
dunn_results = sp.posthoc_dunn(
    df_clean, 
    val_col='PB-biased score', 
    group_col='Virus', 
    p_adjust='fdr_bh'
)

# Reindex both rows and columns to match your desired order
dunn_results = dunn_results.reindex(index=order_virus, columns=order_virus)

# Convert p-values to -log10 for heatmap coloring
heatmap_data = -np.log10(dunn_results)
heatmap_data = heatmap_data.clip(upper=10)  # clip extreme values

# Function to convert p-value to stars
def pval_to_stars(p):
    if p <= 0.0001:
        return "****"
    elif p <= 0.001:
        return "***"
    elif p <= 0.01:
        return "**"
    elif p <= 0.05:
        return "*"
    else:
        return "ns"

# Apply function to create annotation matrix
stars_df = dunn_results.applymap(pval_to_stars)

# Mask upper triangle
mask = np.triu(np.ones_like(heatmap_data, dtype=bool))

# Plot heatmap with log10(p) coloring and stars
plt.figure(figsize=(10,8))
sb.heatmap(
    heatmap_data,
    mask=mask,
    annot=stars_df,
    fmt='',
    cmap="Reds",
    cbar_kws={'label': '-log10(p.adj)'},
    linewidths=0.5,
    linecolor='gray'
)
plt.title("Dunn's Test Pairwise Significance (Ordered Viruses)")
plt.tight_layout()
plt.show()

## Statistics - Ctrl vs MS - CSF biased score per donor (out of virus-specific cells discovered in that donor)

In [ ]:
# Group annotation
patient_list = ["Pt11", "Pt12", "Pt14", "Pt17", "Pt20"]
control_list = ["Pt1", "Pt8", "Pt18", "Pt19", "Pt21"]

## Extract score values per cell and plot as a box plot per specificity
obs_df = mdata["gex"].obs[["Virus", "total_clone_count", "donor", "catpat", "CSF-biased score", "clone_handle"]].copy()
obs_df = obs_df.dropna(subset="Virus")
obs_df = obs_df[~obs_df["donor"].isin(["Pt4", "Pt5"])] #Exclude IC
obs_df = obs_df[obs_df["Virus"].isin(["CMV", "EBV"])] 
obs_df["Virus"] = obs_df["Virus"].cat.remove_unused_categories()
    
# Define RRMS vs Control
obs_df["catpat"] = pd.Categorical(
    np.where(
        obs_df["donor"].isin(patient_list), "RRMS",
        np.where(obs_df["donor"].isin(control_list), "Control", "IC")
    ),
    categories=["Control", "RRMS"],
    ordered=True
)

palette = {
    "RRMS": "#000000",
    # "IC" : "grey", 
    "Control": "#FFFFFF",
}

#Calculate the mean per donor of all cells
obs_df = obs_df.groupby(["donor", "catpat", "Virus"], observed=True).mean("CSF-biased score").reset_index()

#plotting order: 
order = (
    obs_df
    .groupby("Virus")["CSF-biased score"]
    .median()
    .sort_values(ascending=False)
    .index
)

# Create the figure object
fig, ax = plt.subplots(figsize=(3.5,3.2))

# Boxplot (foreground)
sb.boxplot(
    data=obs_df,
    x="Virus",
    y="CSF-biased score",
    hue="catpat",
    width=0.75,
    color="white",
    showcaps=True,
    boxprops={'facecolor': 'white', "alpha":0.5, "linewidth":0.5},
    whiskerprops={'color': '#666666', 'linewidth': 0.5},
    showfliers=False,
    palette=palette,
    zorder=2,
    order=order,
    ax=ax
)

# Stripplot (background dots)
sb.stripplot(
    data=obs_df,
    x="Virus",
    y="CSF-biased score",
    hue="catpat",
    dodge=True,
    s=4,
    jitter=True,
    alpha=0.8,
    linewidth=0.5,
    zorder=1,
    edgecolor="black",
    palette=palette,
    order=order,
    ax=ax
)


plt.ylabel("CSF-biased score\nper donor")
plt.xlabel(" ")
# plt.ylim(-0.25, 0.25)
plt.xticks(rotation=90)

# Remove duplicate legend and move the legend
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles[2:4], labels[2:4], title="Group")
sb.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))

#Despine
sb.despine()

plt.tight_layout()
fig.savefig("../../05_plots/09_multimers/22_boxplot_donorlevel_CSF_biased_score.pdf", format='pdf', dpi=400)
plt.show()

results = []

for virus in ["CMV", "EBV"]:
    
    sub = obs_df[obs_df["Virus"] == virus]
    
    rrms = sub[sub["catpat"] == "RRMS"]["CSF-biased score"]
    ctrl = sub[sub["catpat"] == "Control"]["CSF-biased score"]
    
    # Mann–Whitney U test (two-sided)
    stat, pval = stats.mannwhitneyu(rrms, ctrl, alternative="two-sided")
    
    # Effect size: rank-biserial correlation
    n1, n2 = len(rrms), len(ctrl)
    rbc = 1 - (2 * stat) / (n1 * n2)
    
    results.append({
        "Virus": virus,
        "n_RRMS": n1,
        "n_Control": n2,
        "RRMS_mean": rrms.mean(),
        "Control_mean": ctrl.mean(),
        "RRMS_median": rrms.median(),
        "Control_median": ctrl.median(),
        "U_statistic": stat,
        "p_value": pval,
        "effect_size_rbc": rbc
    })

stats_df_donor = pd.DataFrame(results).sort_values("p_value")

stats_df_donor

## Statistics - Ctrl vs MS - mean CSF-biased clones (although these are discovered in heterogeneous donors)

In [ ]:
# Group annotation
patient_list = ["Pt11", "Pt12", "Pt14", "Pt17", "Pt20"]
control_list = ["Pt1", "Pt8", "Pt18", "Pt19", "Pt21"]

## Extract score values per cell and plot as a box plot per specificity
obs_df = mdata["gex"].obs[["Virus", "donor", "total_clone_count", "CSF-biased score", "clone_handle"]].copy()
obs_df = obs_df.dropna(subset="Virus")
obs_df = obs_df[~obs_df["donor"].isin(["Pt4", "Pt5"])] #Exclude IC
obs_df = obs_df[obs_df["Virus"].isin(["CMV", "EBV"])] #Include only CMV, EBV and FLU-A
obs_df["Virus"] = obs_df["Virus"].cat.remove_unused_categories()

# Define RRMS vs Control
obs_df["catpat"] = pd.Categorical(
    np.where(
        obs_df["donor"].isin(patient_list), "RRMS",
        np.where(obs_df["donor"].isin(control_list), "Control", "IC")
    ),
    categories=["Control", "RRMS"],
    ordered=True
)

palette = {
    "RRMS": "#000000",
    # "IC" : "grey", 
    "Control": "#FFFFFF",
}

#Calculate the mean per donor of all cells
obs_df = obs_df.groupby(["clone_handle", "catpat", "Virus"], observed=True).mean("CSF-biased score").reset_index()

#plotting order: 
order = (
    obs_df
    .groupby("Virus")["CSF-biased score"]
    .median()
    .sort_values(ascending=False)
    .index
)

# Create the figure object
fig, ax = plt.subplots(figsize=(3.5,3.2))

# Boxplot (foreground)
sb.boxplot(
    data=obs_df,
    x="Virus",
    y="CSF-biased score",
    hue="catpat",
    width=0.75,
    color="white",
    showcaps=True,
    zorder=2,
    boxprops={'facecolor': 'white', "alpha":0.5, "linewidth":0.5},
    whiskerprops={'color': '#666666', 'linewidth': 0.5},
    showfliers=False,
    palette=palette,
    order=order,
    ax=ax
)

# Stripplot (background dots)
sb.stripplot(
    data=obs_df,
    x="Virus",
    y="CSF-biased score",
    hue="catpat",
    dodge=True,
    s=4,
    zorder=1,
    jitter=True,
    alpha=0.8,
    linewidth=0.5,
    edgecolor="black",
    palette=palette,
    order=order,
    ax=ax
)


plt.ylabel("Mean CSF-biased score\nper clone")
plt.xlabel(" ")
plt.xticks(rotation=90)
# plt.ylim(-0.25, 0.25)

# Remove duplicate legend and move the legend
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles[2:4], labels[2:4], title="Group")
sb.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))

#Despine
sb.despine()
plt.grid(False)

plt.tight_layout()
fig.savefig("../../05_plots/09_multimers/21_boxplots_csfscore_ctrlvms.pdf", bbox_inches='tight')
plt.show()

results = []

for virus in ["CMV", "EBV", "FLU-A"]:
    
    sub = obs_df[obs_df["Virus"] == virus]
    
    rrms = sub[sub["catpat"] == "RRMS"]["CSF-biased score"]
    ctrl = sub[sub["catpat"] == "Control"]["CSF-biased score"]
    
    # Mann–Whitney U test (two-sided, no correction)
    stat, pval = stats.mannwhitneyu(rrms, ctrl, alternative="two-sided")
    
    results.append({
        "Virus": virus,
        "n_RRMS": len(rrms),
        "n_Control": len(ctrl),
        "RRMS_median": rrms.median(),
        "Control_median": ctrl.median(),
        "U_statistic": stat,
        "p_value": pval
    })

stats_df = pd.DataFrame(results)

# Optional: sort by p-value
stats_df = stats_df.sort_values("p_value")

stats_df

## Show for BZLF1 only MS/Controls

In [ ]:
mdata["gex"].obs[["clone_handle", "handle", "Virus", "HLA", "Antigen", "Sequence"]].dropna(subset="Sequence").drop_duplicates()[["Antigen"]].value_counts().head(5)
#BZLF1 had by far the largest number of clones

In [ ]:
# Group annotation
patient_list = ["Pt11", "Pt12", "Pt14", "Pt17", "Pt20"]
control_list = ["Pt1", "Pt8", "Pt18", "Pt19", "Pt21"]

## Extract score values per cell and plot as a box plot per specificity
obs_df = mdata["gex"].obs[["Virus", "donor", "Antigen", "total_clone_count", "CSF-biased score", "clone_handle"]].copy()
obs_df = obs_df.dropna(subset="Virus")

#Focus on BZLF1
obs_df = obs_df[obs_df["Antigen"].isin(["BZLF1", "pp65", "IE1", "MP", "EBNA1", "EBNA3"])]
obs_df = obs_df.dropna(subset="Antigen")
obs_df["Antigen"] = obs_df["Antigen"].cat.remove_unused_categories()

# Define RRMS vs Control
obs_df["catpat"] = pd.Categorical(
    np.where(
        obs_df["donor"].isin(patient_list), "RRMS",
        np.where(obs_df["donor"].isin(control_list), "Control", "IC")
    ),
    categories=["Control", "RRMS"],
    ordered=True
)

palette = {
    "RRMS": "#000000",
    # "IC" : "grey", 
    "Control": "#FFFFFF",
}

#Calculate the mean per donor of all cells
obs_df = obs_df.groupby(["clone_handle", "catpat", "Antigen"], observed=True).mean("CSF-biased score").reset_index()

#plotting order: 
order = [ "EBNA1", "EBNA3", "BZLF1", "MP", "pp65", "IE1"]

# Create the figure object
fig, ax = plt.subplots(figsize=(6,3.5))

# Boxplot (foreground)
sb.boxplot(
    data=obs_df,
    x="Antigen",
    y="CSF-biased score",
    hue="catpat",
    width=0.75,
    color="white",
    showcaps=True,
    boxprops={'facecolor': 'white', "alpha":0.5},
    showfliers=False,
    zorder=2,
    palette=palette,
    order=order,
    ax=ax
)

# Stripplot (background dots)
sb.stripplot(
    data=obs_df,
    x="Antigen",
    y="CSF-biased score",
    hue="catpat",
    dodge=True,
    s=5,
    jitter=True,
    alpha=0.8,
    linewidth=0.5,
    edgecolor="black",
    palette=palette,
    order=order,
    zorder=1,
    ax=ax
)


plt.ylabel("CSF-biased score per clone")
plt.xlabel(" ")
# plt.ylim(-0.25, 0.25)
# plt.xticks(rotation=90)

# Remove duplicate legend and move the legend
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles[2:4], labels[2:4], title="Group")
sb.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))

#Despine
sb.despine()
plt.grid(False)

plt.tight_layout()
fig.savefig("../../05_plots/09_multimers/24_BZLF1_clones_CSFbiased.pdf", bbox_inches='tight')
plt.show()

results = []

for virus in ["BZLF1", "pp65", "IE1", "EBNA1", "EBNA3", "MP"]:
    
    sub = obs_df[obs_df["Antigen"] == virus]
    
    rrms = sub[sub["catpat"] == "RRMS"]["CSF-biased score"]
    ctrl = sub[sub["catpat"] == "Control"]["CSF-biased score"]
    
    # Mann–Whitney U test (two-sided)
    stat, pval = stats.mannwhitneyu(rrms, ctrl, alternative="two-sided")
    
    # Effect size: rank-biserial correlation
    n1, n2 = len(rrms), len(ctrl)
    rbc = 1 - (2 * stat) / (n1 * n2)
    
    results.append({
        "Virus": virus,
        "n_RRMS": n1,
        "n_Control": n2,
        "RRMS_mean": rrms.mean(),
        "Control_mean": ctrl.mean(),
        "RRMS_median": rrms.median(),
        "Control_median": ctrl.median(),
        "U_statistic": stat,
        "p_value": pval,
        "effect_size_rbc": rbc
    })

stats_df_donor = pd.DataFrame(results).sort_values("p_value")

stats_df_donor

In [ ]:
# Group annotation
patient_list = ["Pt11", "Pt12", "Pt14", "Pt17", "Pt20"]
control_list = ["Pt1", "Pt8", "Pt18", "Pt19", "Pt21"]

## Extract score values per cell and plot as a box plot per specificity
obs_df = mdata["gex"].obs[["Virus", "donor", "Antigen", "total_clone_count", "CSF-biased score", "clone_handle"]].copy()
obs_df = obs_df.dropna(subset="Virus")

#Focus on BZLF1
obs_df = obs_df[obs_df["Antigen"].isin(["BZLF1", "pp65", "IE1", "MP", "EBNA1", "EBNA3"])]
obs_df = obs_df.dropna(subset="Antigen")
obs_df["Antigen"] = obs_df["Antigen"].cat.remove_unused_categories()

# Define RRMS vs Control
obs_df["catpat"] = pd.Categorical(
    np.where(
        obs_df["donor"].isin(patient_list), "RRMS",
        np.where(obs_df["donor"].isin(control_list), "Control", "IC")
    ),
    categories=["Control", "RRMS"],
    ordered=True
)

palette = {
    "RRMS": "#000000",
    # "IC" : "grey", 
    "Control": "#FFFFFF",
}

#Calculate the mean per donor of all cells
obs_df = obs_df.groupby(["donor", "catpat", "Antigen"], observed=True).mean("CSF-biased score").reset_index()

#plotting order: 
order = [ "EBNA1", "EBNA3", "BZLF1", "MP", "pp65", "IE1"]

# Create the figure object
fig, ax = plt.subplots(figsize=(5.5,3.5))

# Boxplot (foreground)
sb.boxplot(
    data=obs_df,
    x="Antigen",
    y="CSF-biased score",
    hue="catpat",
    width=0.75,
    color="white",
    showcaps=True,
    boxprops={'facecolor': 'white', "alpha":0.5},
    showfliers=False,
    zorder=2,
    palette=palette,
    order=order,
    ax=ax
)

# Stripplot (background dots)
sb.stripplot(
    data=obs_df,
    x="Antigen",
    y="CSF-biased score",
    hue="catpat",
    dodge=True,
    s=5,
    jitter=True,
    alpha=0.8,
    linewidth=0.5,
    edgecolor="black",
    palette=palette,
    order=order,
    zorder=1,
    ax=ax
)


plt.ylabel("CSF-biased score per clone")
plt.xlabel(" ")
# plt.ylim(-0.25, 0.25)
# plt.xticks(rotation=90)

# Remove duplicate legend and move the legend
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles[2:4], labels[2:4], title="Group")
sb.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))

#Despine
sb.despine()
plt.grid(False)

plt.tight_layout()
fig.savefig("../../05_plots/09_multimers/24_BZLF1_clones_CSFbiased_perdonor.pdf", bbox_inches='tight')
plt.show()

results = []

for virus in ["BZLF1", "pp65", "IE1", "EBNA1", "MP"]:
    
    sub = obs_df[obs_df["Antigen"] == virus]
    
    rrms = sub[sub["catpat"] == "RRMS"]["CSF-biased score"]
    ctrl = sub[sub["catpat"] == "Control"]["CSF-biased score"]
    
    # Mann–Whitney U test (two-sided)
    stat, pval = stats.mannwhitneyu(rrms, ctrl, alternative="two-sided")
    
    # Effect size: rank-biserial correlation
    n1, n2 = len(rrms), len(ctrl)
    rbc = 1 - (2 * stat) / (n1 * n2)
    
    results.append({
        "Virus": virus,
        "n_RRMS": n1,
        "n_Control": n2,
        "RRMS_mean": rrms.mean(),
        "Control_mean": ctrl.mean(),
        "RRMS_median": rrms.median(),
        "Control_median": ctrl.median(),
        "U_statistic": stat,
        "p_value": pval,
        "effect_size_rbc": rbc
    })

stats_df_donor = pd.DataFrame(results).sort_values("p_value")

stats_df_donor

# All antigens

In [ ]:
# Group annotation
patient_list = ["Pt11", "Pt12", "Pt14", "Pt17", "Pt20"]
control_list = ["Pt1", "Pt8", "Pt18", "Pt19", "Pt21"]

## Extract score values per cell and plot as a box plot per specificity
obs_df = mdata["gex"].obs[["Antigen", "Virus", "donor", "CSF-biased score", "clone_handle"]].copy()
obs_df = obs_df.dropna(subset="Virus")

List_ag = ["EBNA1", "EBNA3", "BZLF1", "LMP2", "S", "pp65", "rep", "MP", "IE1"]
obs_df = obs_df[obs_df["Antigen"].isin(List_ag)]
obs_df["Antigen"] = obs_df["Antigen"].cat.remove_unused_categories()

# Define RRMS vs Control
obs_df["catpat"] = pd.Categorical(
    np.where(
        obs_df["donor"].isin(patient_list), "RRMS",
        np.where(obs_df["donor"].isin(control_list), "Control", "IC")
    ),
    categories=["Control", "RRMS"],
    ordered=True
)

palette = {
    "RRMS": "#000000",
    # "IC" : "grey", 
    "Control": "#FFFFFF",
}

obs_df[obs_df["Antigen"] == "EBNA1"]

#Calculate the mean per donor of all cells
obs_df = obs_df.groupby(["clone_handle", "catpat", "Antigen"], observed=True).mean("CSF-biased score").reset_index()
len(obs_df[obs_df["Antigen"] == "BZLF1"])

#plotting order: 
order = (
    obs_df
    .groupby("Antigen")["CSF-biased score"]
    .median()
    .sort_values()
    .index
)

# Create the figure object
fig, ax = plt.subplots(figsize=(10,5))


# Stripplot (background dots)
sb.stripplot(
    data=obs_df,
    x="Antigen",
    y="CSF-biased score",
    hue="catpat",
    dodge=True,
    s=5,
    jitter=True,
    alpha=0.8,
    linewidth=0.5,
    zorder=1,
    edgecolor="black",
    palette=palette,
    order=order,
    ax=ax
)

# Boxplot (foreground)
sb.boxplot(
    data=obs_df,
    x="Antigen",
    y="CSF-biased score",
    hue="catpat",
    width=0.75,
    color="white",
    showcaps=True,
    boxprops={'facecolor': 'white', "alpha":0.5},
    zorder=2,
    showfliers=False,
    palette=palette,
    order=order,
    ax=ax
)


plt.ylabel("Mean CSF-biased score per clone")
plt.xlabel("Virus")
plt.xticks(rotation=90)
plt.ylim(-0.25, 0.25)

# Remove duplicate legend and move the legend
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles[2:4], labels[2:4], title="Group")
sb.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))

#Despine
sb.despine()

plt.tight_layout()
fig.savefig("../../05_plots/09_multimers/23_perantigen_clonal_CSFbiased.pdf", format="pdf", dpi=400)
plt.show()

results = []

for virus in order:
    
    sub = obs_df[obs_df["Antigen"] == virus]
    
    rrms = sub[sub["catpat"] == "RRMS"]["CSF-biased score"]
    ctrl = sub[sub["catpat"] == "Control"]["CSF-biased score"]
    
    # Mann–Whitney U test (two-sided)
    stat, pval = stats.mannwhitneyu(rrms, ctrl, alternative="two-sided")
    
    # Effect size: rank-biserial correlation
    n1, n2 = len(rrms), len(ctrl)
    rbc = 1 - (2 * stat) / (n1 * n2)
    
    results.append({
        "Virus": virus,
        "n_RRMS": n1,
        "n_Control": n2,
        "RRMS_mean": rrms.mean(),
        "Control_mean": ctrl.mean(),
        "RRMS_median": rrms.median(),
        "Control_median": ctrl.median(),
        "U_statistic": stat,
        "p_value": pval,
        "effect_size_rbc": rbc
    })

stats_df_donor = pd.DataFrame(results).sort_values("p_value")

stats_df_donor

In [ ]:
# Group annotation
patient_list = ["Pt11", "Pt12", "Pt14", "Pt17", "Pt20"]
control_list = ["Pt1", "Pt8", "Pt18", "Pt19", "Pt21"]

## Extract score values per cell and plot as a box plot per specificity
obs_df = mdata["gex"].obs[["Antigen", "donor", "CSF-biased score", "clone_handle"]].copy()
obs_df = obs_df.dropna(subset="Antigen")

List_ag = ["BZLF1"]
obs_df = obs_df[obs_df["Antigen"].isin(List_ag)]
obs_df["Antigen"] = obs_df["Antigen"].cat.remove_unused_categories()

# Define RRMS vs Control
obs_df["catpat"] = pd.Categorical(
    np.where(
        obs_df["donor"].isin(patient_list), "RRMS",
        np.where(obs_df["donor"].isin(control_list), "Control", "IC")
    ),
    categories=["Control", "IC", "RRMS"],
    ordered=True
)

palette = {
    "RRMS": "#000000",
    "IC" : "grey", 
    "Control": "#FFFFFF",
}

#Calculate the mean per donor of all cells
obs_df = obs_df.groupby(["donor", "catpat"], observed=True).mean("CSF-biased score").reset_index()

#plotting order: 
# order = (
#     obs_df
#     .groupby("Antigen")["CSF-biased score"]
#     .median()
#     .sort_values()
#     .index
# )

# Create the figure object
fig, ax = plt.subplots(figsize=(5,4))

# Boxplot (foreground)
sb.boxplot(
    data=obs_df,
    x="catpat",
    y="CSF-biased score",
    hue="catpat",
    width=0.75,
    color="white",
    showcaps=True,
    boxprops={'facecolor': 'white'},
    showfliers=False,
    palette=palette,
    # order=order,
    ax=ax
)

# Stripplot (background dots)
sb.stripplot(
    data=obs_df,
    x="catpat",
    y="CSF-biased score",
    hue="catpat",
    # dodge=True,
    s=7,
    jitter=True,
    alpha=0.8,
    linewidth=0.5,
    edgecolor="black",
    palette=palette,
    # order=order,
    ax=ax
)


plt.ylabel("CSF-biased score")
plt.xlabel(" ")
plt.title("BZLF1-specific CD8 T cells")
plt.ylim(0, 0.25)

# Remove duplicate legend and move the legend
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles[0:2], labels[0:2], title="Group")
sb.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))

#Despine
sb.despine()

plt.tight_layout()
# fig.savefig("../../05_plots/09_multimers/7_mean_clone_CSF_biased_score.pdf", bbox_inches='tight')
plt.show()

In [ ]:
# -----------------------------
# Filter
# -----------------------------
obs_df = sdata["gex"].obs.dropna(subset=["Virus"])
obs_df = obs_df[~obs_df["donor"].isin(["Pt4", "Pt5"])]
obs_df = obs_df[obs_df["Virus"].isin(["CMV", "EBV", "FLU-A"])]
obs_df["Virus"] = obs_df["Virus"].cat.remove_unused_categories()

# -----------------------------
# Count cells per group
# -----------------------------
count_df = (
    obs_df
    .groupby(["Virus", "donor", "Compartment"], observed=True)
    .size()
    .reset_index(name="count")
)

# Total cells per category
count_df["total_cells"] = count_df.groupby(["Virus", "donor"], observed=True)["count"].transform("sum")

# Percentage
count_df["comp_pct"] = 100 * count_df["count"] / count_df["total_cells"]

# CSF percentage per donor
plot_df = count_df[count_df["Compartment"] == "CSF"]

fig, ax = plt.subplots(figsize=(3, 4))

sb.barplot(
    data=plot_df,
    x="Virus",
    y="comp_pct",
    errorbar="sd",   # <-- standard error
    capsize=0.2,
    ax=ax
)

sb.stripplot(
    data=plot_df,
    x="Virus",
    y="comp_pct",
    color="black",
    size=5,
    jitter=True,
    ax=ax
)

# Formatting
ax.set_xlabel("Condition")
ax.set_ylabel("CSF (%)")
ax.set_title("CSF proportion per donor ± SD\namong cells from matched clonotypes")

plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

# Dunn's test (with multiple testing correction)
pvals = sp.posthoc_dunn(
    plot_df,
    val_col="comp_pct",
    group_col="Virus",
    p_adjust="fdr_bh"   # or "bonferroni", "fdr_bh"
)

print(pvals)

# Cummulative distribution to visualize the unique tendency for EBV-specific clones to have high CSF-biased score

In [ ]:
## Extract score values per cell and plot as a box plot per specificity
obs_df = mdata["gex"].obs[["Virus", "CSF-biased score", "clone_handle"]].copy()

# Short list antigens based on how many clones we observed. Only include antigens with more than 5 independent responses
temp_count = obs_df[["Virus", "clone_handle"]].dropna().drop_duplicates()[["Virus"]].value_counts().reset_index()
shortlist = temp_count[temp_count["count"] > 5]["Virus"].tolist()

#Filter to include only the most promising viruses
obs_df = obs_df[obs_df["Virus"].isin(shortlist)]

obs_df = obs_df.dropna(subset="Virus")
obs_df = obs_df.groupby(["clone_handle", "Virus"], observed=True).mean("CSF-biased score").reset_index()

# Goal: Calculate percentile ranks for each clone and display the cummulative % sum of clones belonging to each Virus category on the y-axis going from the top percentile to the lowest percentile on the x-axis. 

#Sort the dataframe first:
obs_df = obs_df.sort_values("CSF-biased score", ascending=False)

# Calculate percentile rank for each clone's CSF-biased score
# percentile_rank gives the percentage of scores <= current score
obs_df["percentile_rank"] = obs_df["CSF-biased score"].rank(pct=True) * 100


# # Compute cumulative % of clones per Virus group along the ranked list
virus_groups = obs_df["Virus"].unique()
virus_counts = obs_df["Virus"].value_counts()  # total clones per virus

fig, ax = plt.subplots(figsize=(3, 3))

for virus in virus_groups:
    # Boolean mask: is each ranked clone from this virus?
    is_virus = (obs_df["Virus"] == virus).astype(int)
    
    # Cumulative sum normalized by total clones of that virus → cumulative %
    cumulative_pct = is_virus.cumsum() / virus_counts[virus] * 100
    
    ax.plot(obs_df["percentile_rank"], cumulative_pct, label=virus)

ax.set_xlabel("CSF-biased Score Percentile (High → Low)")
ax.set_ylabel("Proportion of clones")
# ax.set_title("For viruses with more than 5 independent clones")
ax.set_title(" ")
ax.invert_xaxis()  # x-axis goes from top percentile (100) to lowest (0)
ax.legend()
plt.grid(False)

plt.tight_layout()
plt.savefig("../../05_plots/09_multimers/17_cummulative_EBV_distribution_over_clones.pdf", dpi=400, format="pdf")
plt.show()

In [ ]:
## Extract score values per cell and plot as a box plot per specificity
obs_df = mdata["gex"].obs[["Virus", "CSF-biased score", "catpat", "donor", "clone_handle"]].copy()
obs_df = obs_df[~obs_df["donor"].isin(["Pt4", "Pt5"])]

# Short list antigens based on how many clones we observed. Only include antigens with more than 5 independent responses
temp_count = obs_df[["Virus", "catpat","clone_handle"]].dropna().drop_duplicates()[["Virus"]].value_counts().reset_index()
shortlist = temp_count[temp_count["count"] > 5]["Virus"].tolist()

#Filter to include only the most promising viruses
obs_df = obs_df[obs_df["Virus"].isin(shortlist)]
obs_df = obs_df.dropna(subset="Virus")

## Now define catpat_virus categories
obs_df["catpat_virus"]= obs_df["catpat"].astype(str) + "_" + obs_df["Virus"].astype(str)

obs_df = obs_df.groupby(["clone_handle", "catpat_virus"], observed=True).mean("CSF-biased score").reset_index()

# Goal: Calculate percentile ranks for each clone and display the cummulative % sum of clones belonging to each Virus category on the y-axis going from the top percentile to the lowest percentile on the x-axis. 

#Sort the dataframe first:
obs_df = obs_df.sort_values("CSF-biased score", ascending=False)

# Calculate percentile rank for each clone's CSF-biased score
# percentile_rank gives the percentage of scores <= current score
obs_df["percentile_rank"] = obs_df["CSF-biased score"].rank(pct=True) * 100


# # Compute cumulative % of clones per Virus group along the ranked list
virus_groups = obs_df["catpat_virus"].unique()
virus_counts = obs_df["catpat_virus"].value_counts()  # total clones per virus

fig, ax = plt.subplots(figsize=(3, 3))

for virus in virus_groups:
    # Boolean mask: is each ranked clone from this virus?
    is_virus = (obs_df["catpat_virus"] == virus).astype(int)
    
    # Cumulative sum normalized by total clones of that virus → cumulative %
    cumulative_pct = is_virus.cumsum() / virus_counts[virus] * 100
    
    ax.plot(obs_df["percentile_rank"], cumulative_pct, label=virus)

ax.set_xlabel("CSF-biased Score Percentile (High → Low)")
ax.set_ylabel("Proportion of clones")
# ax.set_title("For viruses with more than 5 independent clones")
ax.set_title(" ")
ax.invert_xaxis()  # x-axis goes from top percentile (100) to lowest (0)
# ax.legend(bbox_to_anchor=(1.25, 0.75), loc="center")
plt.grid(False)

plt.tight_layout()
plt.show()

## Stats summary of antigen-wise cummulative distributions

In [ ]:
## Among EBV specificities, are there antigens with different kinetics?

## Extract score values per cell and plot as a box plot per specificity
obs_df = mdata["gex"].obs[["Virus", "CSF-biased score", "clone_handle", "Antigen"]].copy()

# Short list antigens based on how many clones we observed. Only include antigens with more than 5 independent responses
temp_ag_count = obs_df[obs_df["Virus"].isin(["EBV", "CMV", "SARS-COV-2", "FLU-A"])][["Antigen", "clone_handle"]].dropna().drop_duplicates()[["Antigen"]].value_counts().reset_index()
ag_shortlist = temp_ag_count[temp_ag_count["count"] > 5]["Antigen"].tolist()

#Filter to include only the most promising antigens
obs_df = obs_df[obs_df["Antigen"].isin(ag_shortlist)]
obs_df = obs_df.dropna(subset="Antigen")

#Calculate mean CSF-biased scores per clone
obs_df = obs_df.groupby(["clone_handle", "Antigen", "Virus"], observed=True).mean("CSF-biased score").reset_index()

# Goal: Calculate percentile ranks for each clone and display the cummulative % sum of clones belonging to each Antigen category on the y-axis going from the top percentile to the lowest percentile on the x-axis. 
#Sort the dataframe first:
obs_df = obs_df.sort_values("CSF-biased score", ascending=False)

# Calculate percentile rank for each clone's CSF-biased score
# percentile_rank gives the percentage of scores <= current score
obs_df["percentile_rank"] = obs_df["CSF-biased score"].rank(pct=True) * 100

# # Compute cumulative % of clones per Virus group along the ranked list
ag_groups = obs_df["Antigen"].unique()
ag_counts = obs_df["Antigen"].value_counts()  # total clones per virus

fig, ax = plt.subplots(figsize=(3, 3))

for ag in ag_groups:
    # Boolean mask: is each ranked clone from this virus?
    is_ag = (obs_df["Antigen"] == ag).astype(int)
    
    # Cumulative sum normalized by total clones of that virus → cumulative %
    cumulative_pct = is_ag.cumsum() / ag_counts[ag] * 100
    
    ax.plot(obs_df["percentile_rank"], cumulative_pct, label=ag)

ax.set_xlabel("CSF-biased Score Percentile (High → Low)")
ax.set_ylabel("Proportion of clones")
ax.set_title("For antigens with more than 5 independent clones")
ax.invert_xaxis()  # x-axis goes from top percentile (100) to lowest (0)
ax.legend()
plt.grid(False)

plt.tight_layout()
plt.savefig("../../05_plots/09_multimers/18_cummulative_over_clones_and_antigens.pdf", format="pdf", dpi=400)
plt.show()

In [ ]:
### obs_df from before
obs_df["Antigen"] = obs_df["Antigen"].cat.remove_unused_categories()

#plotting order: 
order = (
    obs_df
    .groupby("Antigen")["CSF-biased score"]
    .median()
    .sort_values()
    .index
)

# Create the figure object
fig, ax = plt.subplots(figsize=(3.55,3))

# Stripplot (background dots)
sb.stripplot(
    data=obs_df,
    x="Antigen",
    y="CSF-biased score",
    jitter=0.25,
    size=4,
    alpha=0.7,
    color="black",
    zorder=1,
    order=order,
    ax=ax
)

# Boxplot (foreground)
sb.boxplot(
    data=obs_df,
    x="Antigen",
    y="CSF-biased score",
    width=0.5,
    color="white",
    fliersize=0,
    boxprops=dict(alpha=.5),
    linewidth=1,
    zorder=2,
    order=order,
    ax=ax
)

plt.ylabel("Mean CSF-biased score\nper clone")
plt.xlabel("Antigen")
plt.xticks(rotation=90)
sb.despine()

plt.tight_layout()
fig.savefig("../../05_plots/09_multimers/12_mean_clone_CSF_biased_score_per_antigen.pdf", bbox_inches='tight')
plt.show()

# Clean data
df_clean = obs_df.dropna(subset=['Antigen', 'CSF-biased score'])

# Define the order
order_ag = order

# Perform Dunn's test with FDR correction
dunn_results = sp.posthoc_dunn(
    df_clean, 
    val_col='CSF-biased score', 
    group_col='Antigen', 
    p_adjust='fdr_bh'
)

# Reindex both rows and columns to match your desired order
dunn_results = dunn_results.reindex(index=order_ag, columns=order_ag)

# Convert p-values to -log10 for heatmap coloring
heatmap_data = -np.log10(dunn_results)
heatmap_data = heatmap_data.clip(upper=10)  # clip extreme values

# Function to convert p-value to stars
def pval_to_stars(p):
    if p <= 0.0001:
        return "****"
    elif p <= 0.001:
        return "***"
    elif p <= 0.01:
        return "**"
    elif p <= 0.05:
        return "*"
    else:
        return "ns"

# Apply function to create annotation matrix
stars_df = dunn_results.applymap(pval_to_stars)

# Mask upper triangle
mask = np.triu(np.ones_like(heatmap_data, dtype=bool))

# Plot heatmap with log10(p) coloring and stars
plt.figure(figsize=(10,8))
sb.heatmap(
    heatmap_data,
    mask=mask,
    annot=stars_df,
    fmt='',
    cmap="Reds",
    cbar_kws={'label': '-log10(p.adj)'},
    linewidths=0.5,
    linecolor='gray'
)
plt.title("Dunn's Test Pairwise Significance (Ordered antigens)")
sb.despine()
plt.tight_layout()
plt.show()

# Diversity metrics

In [ ]:
# --------------------------------------------------
# Diversity metric:
# Number of unique clones detected per multimer (antigen) per donor
# --------------------------------------------------
# Note: The dataframe is reduced to unique combinations of
# clone_handle and multimer recognition per donor

obs_df = (
    mdata["gex"].obs[["Antigen", "handle", "Virus", "donor", "clone_handle", "catpat"]]
    .drop_duplicates()
    .dropna(subset=["Virus"])
    .copy()
)

obs_df["Antigen"] = obs_df["Antigen"].cat.remove_unused_categories()

# --------------------------------------------------
# Count unique clones per antigen per donor
# --------------------------------------------------
count_df = (
    obs_df
    .groupby(["Antigen", "handle", "donor", "catpat"], observed=True)
    .size()
    .reset_index(name="count")
)

count_df = count_df.sort_values("count", ascending=False)

## Order of the x-axis on the plot: 
order = (
    count_df
    .groupby("Antigen")["count"]
    .median()
    .sort_values(ascending=False)
    .index
)

# --------------------------------------------------
# Plot
# --------------------------------------------------
fig, ax = plt.subplots(figsize=(7, 3))

# Background: individual donor points
sb.stripplot(
    data=count_df,
    x="Antigen",
    y="count",
    jitter=0.25,
    dodge=False,
    size=4,
    alpha=0.7,
    color="black",
    zorder=1,
    order=order, 
    ax=ax
)

# Foreground: distribution summary
sb.boxplot(
    data=count_df,
    x="Antigen",
    y="count",
    width=0.5,
    color="white",
    fliersize=0,
    boxprops={"alpha": 0.5},
    linewidth=1,
    zorder=2,
    order=order, 
    ax=ax
)

#Format
ax.tick_params(axis='x', rotation=90)
ax.set_ylabel("Number of clonotypes\nper multimer population")
sb.despine()
plt.grid(False)

#Export
plt.tight_layout()
plt.savefig("../../05_plots/09_multimers/26_NumberOfClonotypes_permultimer.pdf")
plt.show()

# Inverse Simpson index - per virus

In [ ]:
mdata_rep = mdata.copy()

#Transfer the Antigen, clone_handle and virus label to airr
mdata_rep["airr"].obs["clone_handle"] = mdata_rep["gex"].obs["clone_handle"].copy()
# mdata_rep["airr"].obs["Antigen"] = mdata_rep["gex"].obs["Antigen"].copy()
mdata_rep["airr"].obs["Virus"] = mdata_rep["gex"].obs["Virus"].copy()
mdata_rep["airr"].obs["donor"] = mdata_rep["gex"].obs["donor"].copy()

#Filter the data frame
# mu.pp.filter_obs(mdata_rep["gex"], "Antigen", lambda x: (np.isin(x,["EBNA1", "EBNA3", "BZLF1", "LMP2", "S", "pp65", "rep", "MP", "IE1"]))) 
# mdata.update()

#Update the dataframe ---------------------
#Make sure only the indexes recorded in GEX are used in the other modalities too
index_list = mdata_rep["gex"].obs.index.tolist()

## define a cell index
mdata_rep["airr"].obs["cell_index"] = mdata_rep["airr"].obs.index
mdata_rep["prot"].obs["cell_index"] = mdata_rep["prot"].obs.index

## ".isin" using indexes does not seem to work on a global level, but if I annotate a column and then filter in place, maybe then it works. 
mdata_rep["airr"].obs["filter_index"] = pd.Categorical(np.where(mdata_rep["airr"].obs["cell_index"].isin(index_list), "GEX_present", "GEX_absent"))
mdata_rep["prot"].obs["filter_index"] = pd.Categorical(np.where(mdata_rep["prot"].obs["cell_index"].isin(index_list), "GEX_present", "GEX_absent"))

#Enforce that mdata_rep["airr"].obs and mdata_rep["prot"].obs only contain information on those indexes that exists in GEX
mu.pp.filter_obs(mdata_rep["airr"], 'filter_index', lambda x: x == "GEX_present")
mu.pp.filter_obs(mdata_rep["prot"], 'filter_index', lambda x: x == "GEX_present")
mdata_rep.update()
# -----------------------------------------

## Create the donor_virus column and calculate the Gini index --------
mdata_rep["airr"].obs["donor_virus"] = mdata_rep["airr"].obs["donor"].astype(str) + "_" + mdata_rep["airr"].obs["Virus"].astype(str)

## %% Calculate the diversity metrics andsave them under a column key. 
ir.tl.alpha_diversity(mdata_rep["airr"], groupby="donor_virus", target_col='clone_handle', metric='inv_simpson', inplace=True, key_added="InverseSimpson_per_donor_per_virus", airr_mod='airr')

In [ ]:
# Get the obs_df
obs_df = mdata_rep["airr"].obs[["donor", "Virus", "InverseSimpson_per_donor_per_virus"]].drop_duplicates().reset_index()
obs_df = obs_df.dropna(subset="Virus")
obs_df["Virus"] = obs_df["Virus"].cat.remove_unused_categories()

fig, ax = plt.subplots(figsize=(4,4))

# Column plot (mean)
sb.barplot(
    data=obs_df,
    x="Virus",
    y="InverseSimpson_per_donor_per_virus",
    color="white",
    edgecolor="black",
    errorbar="se",   # <- adds error bars
    capsize=0.5,   # <- small caps look nice
    ax=ax
)

# Dots (individual donors)
sb.stripplot(
    data=obs_df,
    x="Virus",
    y="InverseSimpson_per_donor_per_virus",
    dodge=True,
    jitter=True,
    s=5,
    edgecolor="black",
    linewidth=0.5,
    ax=ax
)

plt.ylabel("Inverse Simpson (diversity) index")
plt.xlabel("Virus")
plt.xticks(rotation=90)
# plt.ylim(-0.025, 1)

# Remove duplicate legend
# handles, labels = ax.get_legend_handles_labels()
# ax.legend(handles[0:0], labels[0:0], title="Group")
# sb.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))

sb.despine()
plt.tight_layout()
plt.show()

In [ ]:
# Get the obs_df
obs_df = mdata_rep["airr"].obs[["donor", "Virus", "InverseSimpson_per_donor_per_virus"]].drop_duplicates().reset_index()
obs_df = obs_df.dropna(subset="Virus")
obs_df["Virus"] = obs_df["Virus"].cat.remove_unused_categories()

#Annotate catpat
patient_list = ["Pt11", "Pt12", "Pt14", "Pt17", "Pt20"]
control_list = ["Pt1", "Pt8", "Pt18", "Pt19", "Pt21"]

# Define RRMS vs Control
obs_df["catpat"] = pd.Categorical(
    np.where(
        obs_df["donor"].isin(patient_list), "RRMS",
        np.where(obs_df["donor"].isin(control_list), "Control", "IC")
    ),
    categories=["Control", "IC", "RRMS"],
    ordered=True
)


fig, ax = plt.subplots(figsize=(6,4))

palette = {
    "RRMS": "#000000",
    "IC": "grey",
    "Control": "#FFFFFF"
}

# Column plot (mean)
sb.barplot(
    data=obs_df,
    x="Virus",
    y="InverseSimpson_per_donor_per_virus",
    hue="catpat",
    color="white",
    palette={"RRMS":"white", "IC":"white", "Control":"white"},
    edgecolor="black",
    errorbar="se",   # <- adds error bars
    capsize=0.5,   # <- small caps look nice
    ax=ax
)

# Dots (individual donors)
sb.stripplot(
    data=obs_df,
    x="Virus",
    y="InverseSimpson_per_donor_per_virus",
    hue="catpat",
    dodge=True,
    jitter=True,
    s=5,
    edgecolor="black",
    linewidth=0.5,
    palette=palette,
    ax=ax
)

plt.ylabel("Inverse Simpson (diversity) index")
plt.xlabel("Virus")
plt.xticks(rotation=90)
# plt.ylim(-0.025, 1)

# Remove duplicate legend
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles[3:6], labels[3:6], title="Group")
sb.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))

sb.despine()
plt.tight_layout()
plt.show()

# Norm Shannon Entropy - per virus

In [ ]:
mdata_rep = mdata.copy()

#Transfer the Antigen, clone_handle and virus label to airr
mdata_rep["airr"].obs["clone_handle"] = mdata_rep["gex"].obs["clone_handle"].copy()
# mdata_rep["airr"].obs["Antigen"] = mdata_rep["gex"].obs["Antigen"].copy()
mdata_rep["airr"].obs["Virus"] = mdata_rep["gex"].obs["Virus"].copy()
mdata_rep["airr"].obs["donor"] = mdata_rep["gex"].obs["donor"].copy()

#Filter the data frame
# mu.pp.filter_obs(mdata_rep["gex"], "Antigen", lambda x: (np.isin(x,["EBNA1", "EBNA3", "BZLF1", "LMP2", "S", "pp65", "rep", "MP", "IE1"]))) 
# mdata.update()

#Update the dataframe ---------------------
#Make sure only the indexes recorded in GEX are used in the other modalities too
index_list = mdata_rep["gex"].obs.index.tolist()

## define a cell index
mdata_rep["airr"].obs["cell_index"] = mdata_rep["airr"].obs.index
mdata_rep["prot"].obs["cell_index"] = mdata_rep["prot"].obs.index

## ".isin" using indexes does not seem to work on a global level, but if I annotate a column and then filter in place, maybe then it works. 
mdata_rep["airr"].obs["filter_index"] = pd.Categorical(np.where(mdata_rep["airr"].obs["cell_index"].isin(index_list), "GEX_present", "GEX_absent"))
mdata_rep["prot"].obs["filter_index"] = pd.Categorical(np.where(mdata_rep["prot"].obs["cell_index"].isin(index_list), "GEX_present", "GEX_absent"))

#Enforce that mdata_rep["airr"].obs and mdata_rep["prot"].obs only contain information on those indexes that exists in GEX
mu.pp.filter_obs(mdata_rep["airr"], 'filter_index', lambda x: x == "GEX_present")
mu.pp.filter_obs(mdata_rep["prot"], 'filter_index', lambda x: x == "GEX_present")
mdata_rep.update()
# -----------------------------------------

## Create the donor_virus column and calculate the Gini index --------
mdata_rep["airr"].obs["donor_virus"] = mdata_rep["airr"].obs["donor"].astype(str) + "_" + mdata_rep["airr"].obs["Virus"].astype(str)

## %% Calculate the diversity metrics andsave them under a column key. 
ir.tl.alpha_diversity(mdata_rep["airr"], groupby="donor_virus", target_col='clone_handle', metric='normalized_shannon_entropy', inplace=True, key_added="Shannon_per_donor_per_virus", airr_mod='airr')

In [ ]:
# Get the obs_df
obs_df = mdata_rep["airr"].obs[["donor", "Virus", "Shannon_per_donor_per_virus"]].drop_duplicates().reset_index()
obs_df = obs_df.dropna(subset="Virus")
obs_df["Virus"] = obs_df["Virus"].cat.remove_unused_categories()

fig, ax = plt.subplots(figsize=(4,4))

# Column plot (mean)
sb.barplot(
    data=obs_df,
    x="Virus",
    y="Shannon_per_donor_per_virus",
    color="white",
    edgecolor="black",
    errorbar="se",   # <- adds error bars
    capsize=0.5,   # <- small caps look nice
    ax=ax
)

# Dots (individual donors)
sb.stripplot(
    data=obs_df,
    x="Virus",
    y="Shannon_per_donor_per_virus",
    dodge=True,
    jitter=True,
    s=5,
    edgecolor="black",
    linewidth=0.5,
    ax=ax
)

plt.ylabel("Normalized Shannon Entropy")
plt.xlabel("Virus")
plt.xticks(rotation=90)
# plt.ylim(-0.025, 1)

# Remove duplicate legend
# handles, labels = ax.get_legend_handles_labels()
# ax.legend(handles[0:0], labels[0:0], title="Group")
# sb.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))

sb.despine()
plt.tight_layout()
plt.show()

In [ ]:
# Get the obs_df
obs_df = mdata_rep["airr"].obs[["donor", "Virus", "Shannon_per_donor_per_virus"]].drop_duplicates().reset_index()
obs_df = obs_df.dropna(subset="Virus")
obs_df["Virus"] = obs_df["Virus"].cat.remove_unused_categories()

#Annotate catpat
patient_list = ["Pt11", "Pt12", "Pt14", "Pt17", "Pt20"]
control_list = ["Pt1", "Pt8", "Pt18", "Pt19", "Pt21"]

# Define RRMS vs Control
obs_df["catpat"] = pd.Categorical(
    np.where(
        obs_df["donor"].isin(patient_list), "RRMS",
        np.where(obs_df["donor"].isin(control_list), "Control", "IC")
    ),
    categories=["Control", "IC", "RRMS"],
    ordered=True
)


fig, ax = plt.subplots(figsize=(6,4))

palette = {
    "RRMS": "#000000",
    "IC": "grey",
    "Control": "#FFFFFF"
}

# Column plot (mean)
sb.barplot(
    data=obs_df,
    x="Virus",
    y="Shannon_per_donor_per_virus",
    hue="catpat",
    color="white",
    palette={"RRMS":"white", "IC":"white", "Control":"white"},
    edgecolor="black",
    errorbar="se",   # <- adds error bars
    capsize=0.5,   # <- small caps look nice
    ax=ax
)

# Dots (individual donors)
sb.stripplot(
    data=obs_df,
    x="Virus",
    y="Shannon_per_donor_per_virus",
    hue="catpat",
    dodge=True,
    jitter=True,
    s=5,
    edgecolor="black",
    linewidth=0.5,
    palette=palette,
    ax=ax
)

plt.ylabel("Normalized Shannon Entropy")
plt.xlabel("Virus")
plt.xticks(rotation=90)
# plt.ylim(-0.025, 1)

# Remove duplicate legend
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles[3:6], labels[3:6], title="Group")
sb.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))

sb.despine()
plt.tight_layout()
plt.show()

# Gini index - per virus

In [ ]:
mdata_rep = mdata.copy()

#Transfer the Antigen, clone_handle and virus label to airr
mdata_rep["airr"].obs["clone_handle"] = mdata_rep["gex"].obs["clone_handle"].copy()
# mdata_rep["airr"].obs["Antigen"] = mdata_rep["gex"].obs["Antigen"].copy()
mdata_rep["airr"].obs["Virus"] = mdata_rep["gex"].obs["Virus"].copy()
mdata_rep["airr"].obs["donor"] = mdata_rep["gex"].obs["donor"].copy()

#Filter the data frame
# mu.pp.filter_obs(mdata_rep["gex"], "Antigen", lambda x: (np.isin(x,["EBNA1", "EBNA3", "BZLF1", "LMP2", "S", "pp65", "rep", "MP", "IE1"]))) 
# mdata.update()

#Update the dataframe ---------------------
#Make sure only the indexes recorded in GEX are used in the other modalities too
index_list = mdata_rep["gex"].obs.index.tolist()

## define a cell index
mdata_rep["airr"].obs["cell_index"] = mdata_rep["airr"].obs.index
mdata_rep["prot"].obs["cell_index"] = mdata_rep["prot"].obs.index

## ".isin" using indexes does not seem to work on a global level, but if I annotate a column and then filter in place, maybe then it works. 
mdata_rep["airr"].obs["filter_index"] = pd.Categorical(np.where(mdata_rep["airr"].obs["cell_index"].isin(index_list), "GEX_present", "GEX_absent"))
mdata_rep["prot"].obs["filter_index"] = pd.Categorical(np.where(mdata_rep["prot"].obs["cell_index"].isin(index_list), "GEX_present", "GEX_absent"))

#Enforce that mdata_rep["airr"].obs and mdata_rep["prot"].obs only contain information on those indexes that exists in GEX
mu.pp.filter_obs(mdata_rep["airr"], 'filter_index', lambda x: x == "GEX_present")
mu.pp.filter_obs(mdata_rep["prot"], 'filter_index', lambda x: x == "GEX_present")
mdata_rep.update()
# -----------------------------------------

## Create the donor_virus column and calculate the Gini index --------
mdata_rep["airr"].obs["donor_virus"] = mdata_rep["airr"].obs["donor"].astype(str) + "_" + mdata_rep["airr"].obs["Virus"].astype(str)

## %% Calculate the diversity metrics andsave them under a column key. 
ir.tl.alpha_diversity(mdata_rep["airr"], groupby="donor_virus", target_col='clone_handle', metric='gini_index', inplace=True, key_added="Gini_per_donor_per_virus", airr_mod='airr')

In [ ]:
# Get the obs_df
obs_df = mdata_rep["airr"].obs[["donor", "Virus", "Gini_per_donor_per_virus"]].drop_duplicates().reset_index()
obs_df = obs_df.dropna(subset="Virus")
obs_df["Virus"] = obs_df["Virus"].cat.remove_unused_categories()

fig, ax = plt.subplots(figsize=(4,4))

# Column plot (mean)
sb.barplot(
    data=obs_df,
    x="Virus",
    y="Gini_per_donor_per_virus",
    color="white",
    edgecolor="black",
    errorbar="se",   # <- adds error bars
    capsize=0.5,   # <- small caps look nice
    ax=ax
)

# Dots (individual donors)
sb.stripplot(
    data=obs_df,
    x="Virus",
    y="Gini_per_donor_per_virus",
    dodge=True,
    jitter=True,
    s=5,
    edgecolor="black",
    linewidth=0.5,
    ax=ax
)

plt.ylabel("Gini index")
plt.xlabel("Virus")
plt.xticks(rotation=90)
# plt.ylim(-0.025, 1)

# Remove duplicate legend
# handles, labels = ax.get_legend_handles_labels()
# ax.legend(handles[0:0], labels[0:0], title="Group")
# sb.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))

sb.despine()
plt.tight_layout()
plt.show()

In [ ]:
# Get the obs_df
obs_df = mdata_rep["airr"].obs[["donor", "Virus", "Gini_per_donor_per_virus"]].drop_duplicates().reset_index()
obs_df = obs_df.dropna(subset="Virus")
obs_df["Virus"] = obs_df["Virus"].cat.remove_unused_categories()

#Annotate catpat
patient_list = ["Pt11", "Pt12", "Pt14", "Pt17", "Pt20"]
control_list = ["Pt1", "Pt8", "Pt18", "Pt19", "Pt21"]

# Define RRMS vs Control
obs_df["catpat"] = pd.Categorical(
    np.where(
        obs_df["donor"].isin(patient_list), "RRMS",
        np.where(obs_df["donor"].isin(control_list), "Control", "IC")
    ),
    categories=["Control", "IC", "RRMS"],
    ordered=True
)


fig, ax = plt.subplots(figsize=(6,4))

palette = {
    "RRMS": "#000000",
    "IC": "grey",
    "Control": "#FFFFFF"
}

# Column plot (mean)
sb.barplot(
    data=obs_df,
    x="Virus",
    y="Gini_per_donor_per_virus",
    hue="catpat",
    color="white",
    palette={"RRMS":"white", "IC":"white", "Control":"white"},
    edgecolor="black",
    errorbar="se",   # <- adds error bars
    capsize=0.5,   # <- small caps look nice
    ax=ax
)

# Dots (individual donors)
sb.stripplot(
    data=obs_df,
    x="Virus",
    y="Gini_per_donor_per_virus",
    hue="catpat",
    dodge=True,
    jitter=True,
    s=5,
    edgecolor="black",
    linewidth=0.5,
    palette=palette,
    ax=ax
)

plt.ylabel("Gini index")
plt.xlabel("Virus")
plt.xticks(rotation=90)
# plt.ylim(-0.025, 1)

# Remove duplicate legend
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles[3:6], labels[3:6], title="Group")
sb.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))

sb.despine()
plt.tight_layout()
plt.show()

# Inverse Simpson index - repertoire statistics per virus similar to Sturmlechner et al

In [ ]:
mdata_rep = mdata.copy()

#Transfer the Antigen, clone_handle and virus label to airr
mdata_rep["airr"].obs["clone_handle"] = mdata_rep["gex"].obs["clone_handle"].copy()
mdata_rep["airr"].obs["Antigen"] = mdata_rep["gex"].obs["Antigen"].copy()
mdata_rep["airr"].obs["Virus"] = mdata_rep["gex"].obs["Virus"].copy()
mdata_rep["airr"].obs["donor"] = mdata_rep["gex"].obs["donor"].copy()

#Filter the data frame
mu.pp.filter_obs(mdata_rep["gex"], "Antigen", lambda x: (np.isin(x,["EBNA1", "EBNA3", "BZLF1", "LMP2", "S", "pp65", "rep", "MP", "IE1"]))) 
mdata.update()

#Update the dataframe ---------------------
#Make sure only the indexes recorded in GEX are used in the other modalities too
index_list = mdata_rep["gex"].obs.index.tolist()

## define a cell index
mdata_rep["airr"].obs["cell_index"] = mdata_rep["airr"].obs.index
mdata_rep["prot"].obs["cell_index"] = mdata_rep["prot"].obs.index

## ".isin" using indexes does not seem to work on a global level, but if I annotate a column and then filter in place, maybe then it works. 
mdata_rep["airr"].obs["filter_index"] = pd.Categorical(np.where(mdata_rep["airr"].obs["cell_index"].isin(index_list), "GEX_present", "GEX_absent"))
mdata_rep["prot"].obs["filter_index"] = pd.Categorical(np.where(mdata_rep["prot"].obs["cell_index"].isin(index_list), "GEX_present", "GEX_absent"))

#Enforce that mdata_rep["airr"].obs and mdata_rep["prot"].obs only contain information on those indexes that exists in GEX
mu.pp.filter_obs(mdata_rep["airr"], 'filter_index', lambda x: x == "GEX_present")
mu.pp.filter_obs(mdata_rep["prot"], 'filter_index', lambda x: x == "GEX_present")
mdata_rep.update()
# -----------------------------------------

## Create the donor_antigen column and calculate the Gini index --------
mdata_rep["airr"].obs["donor_antigen"] = mdata_rep["airr"].obs["donor"].astype(str) + "_" + mdata_rep["airr"].obs["Antigen"].astype(str)

## %% Calculate the diversity metrics andsave them under a column key. 
ir.tl.alpha_diversity(mdata_rep["airr"], groupby="donor_antigen", target_col='clone_handle', metric='inv_simpson', inplace=True, key_added="InverseSimpson_per_donor_per_antigen", airr_mod='airr')

In [ ]:
# Get the obs_df
obs_df = mdata_rep["airr"].obs[["donor", "Antigen", "InverseSimpson_per_donor_per_antigen"]].drop_duplicates().reset_index()
obs_df = obs_df.dropna(subset="Antigen")
obs_df["Antigen"] = obs_df["Antigen"].cat.remove_unused_categories()

fig, ax = plt.subplots(figsize=(4,4))

# Column plot (mean)
sb.barplot(
    data=obs_df,
    x="Antigen",
    y="InverseSimpson_per_donor_per_antigen",
    color="white",
    edgecolor="black",
    errorbar="se",   # <- adds error bars
    capsize=0.5,   # <- small caps look nice
    ax=ax
)

# Dots (individual donors)
sb.stripplot(
    data=obs_df,
    x="Antigen",
    y="InverseSimpson_per_donor_per_antigen",
    dodge=True,
    jitter=True,
    s=5,
    edgecolor="black",
    linewidth=0.5,
    ax=ax
)

plt.ylabel("Inverse Simpson (diversity) index")
plt.xlabel("Antigen")
plt.xticks(rotation=90)
# plt.ylim(-0.025, 1)

# Remove duplicate legend
# handles, labels = ax.get_legend_handles_labels()
# ax.legend(handles[0:0], labels[0:0], title="Group")
# sb.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))

sb.despine()
plt.tight_layout()
plt.show()

In [ ]:
# Get the obs_df
obs_df = mdata_rep["airr"].obs[["donor", "Antigen", "InverseSimpson_per_donor_per_antigen"]].drop_duplicates().reset_index()
obs_df = obs_df.dropna(subset="Antigen")
obs_df["Antigen"] = obs_df["Antigen"].cat.remove_unused_categories()

#Annotate catpat
patient_list = ["Pt11", "Pt12", "Pt14", "Pt17", "Pt20"]
control_list = ["Pt1", "Pt8", "Pt18", "Pt19", "Pt21"]

# Define RRMS vs Control
obs_df["catpat"] = pd.Categorical(
    np.where(
        obs_df["donor"].isin(patient_list), "RRMS",
        np.where(obs_df["donor"].isin(control_list), "Control", "IC")
    ),
    categories=["Control", "IC", "RRMS"],
    ordered=True
)


fig, ax = plt.subplots(figsize=(5,4))

palette = {
    "RRMS": "#000000",
    "IC": "grey",
    "Control": "#FFFFFF"
}

# Column plot (mean)
sb.barplot(
    data=obs_df,
    x="Antigen",
    y="InverseSimpson_per_donor_per_antigen",
    hue="catpat",
    color="white",
    palette={"RRMS":"white", "IC":"white", "Control":"white"},
    edgecolor="black",
    errorbar="se",   # <- adds error bars
    capsize=0.5,   # <- small caps look nice
    ax=ax
)

# Dots (individual donors)
sb.stripplot(
    data=obs_df,
    x="Antigen",
    y="InverseSimpson_per_donor_per_antigen",
    hue="catpat",
    dodge=True,
    jitter=True,
    s=5,
    edgecolor="black",
    linewidth=0.5,
    palette=palette,
    ax=ax
)

plt.ylabel("Inverse Simpson (diversity) index")
plt.xlabel("Antigen")
plt.xticks(rotation=90)
# plt.ylim(-0.025, 1)

# Remove duplicate legend
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles[3:6], labels[3:6], title="Group")
sb.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))

sb.despine()
plt.tight_layout()
plt.show()

# Normalized Shannon entropy

In [ ]:
mdata_rep = mdata.copy()

#Transfer the Antigen, clone_handle and virus label to airr
mdata_rep["airr"].obs["clone_handle"] = mdata_rep["gex"].obs["clone_handle"].copy()
mdata_rep["airr"].obs["Antigen"] = mdata_rep["gex"].obs["Antigen"].copy()
mdata_rep["airr"].obs["Virus"] = mdata_rep["gex"].obs["Virus"].copy()
mdata_rep["airr"].obs["donor"] = mdata_rep["gex"].obs["donor"].copy()

#Filter the data frame
mu.pp.filter_obs(mdata_rep["gex"], "Antigen", lambda x: (np.isin(x,["EBNA1", "EBNA3", "BZLF1", "LMP2", "S", "pp65", "rep", "MP", "IE1"]))) 
mdata.update()

#Update the dataframe ---------------------
#Make sure only the indexes recorded in GEX are used in the other modalities too
index_list = mdata_rep["gex"].obs.index.tolist()

## define a cell index
mdata_rep["airr"].obs["cell_index"] = mdata_rep["airr"].obs.index
mdata_rep["prot"].obs["cell_index"] = mdata_rep["prot"].obs.index

## ".isin" using indexes does not seem to work on a global level, but if I annotate a column and then filter in place, maybe then it works. 
mdata_rep["airr"].obs["filter_index"] = pd.Categorical(np.where(mdata_rep["airr"].obs["cell_index"].isin(index_list), "GEX_present", "GEX_absent"))
mdata_rep["prot"].obs["filter_index"] = pd.Categorical(np.where(mdata_rep["prot"].obs["cell_index"].isin(index_list), "GEX_present", "GEX_absent"))

#Enforce that mdata_rep["airr"].obs and mdata_rep["prot"].obs only contain information on those indexes that exists in GEX
mu.pp.filter_obs(mdata_rep["airr"], 'filter_index', lambda x: x == "GEX_present")
mu.pp.filter_obs(mdata_rep["prot"], 'filter_index', lambda x: x == "GEX_present")
mdata_rep.update()
# -----------------------------------------

## Create the donor_antigen column and calculate the Gini index --------
mdata_rep["airr"].obs["donor_antigen"] = mdata_rep["airr"].obs["donor"].astype(str) + "_" + mdata_rep["airr"].obs["Antigen"].astype(str)

## %% Calculate the diversity metrics andsave them under a column key. 
ir.tl.alpha_diversity(mdata_rep["airr"], groupby="donor_antigen", target_col='clone_handle', metric='normalized_shannon_entropy', inplace=True, key_added="Entropy_per_donor_per_antigen", airr_mod='airr')

In [ ]:
# Get the obs_df
obs_df = mdata_rep["airr"].obs[["donor", "Antigen", "Entropy_per_donor_per_antigen"]].drop_duplicates().reset_index()
obs_df = obs_df.dropna(subset="Antigen")
obs_df["Antigen"] = obs_df["Antigen"].cat.remove_unused_categories()

fig, ax = plt.subplots(figsize=(4,4))

# Column plot (mean)
sb.barplot(
    data=obs_df,
    x="Antigen",
    y="Entropy_per_donor_per_antigen",
    color="white",
    edgecolor="black",
    errorbar="se",   # <- adds error bars
    capsize=0.5,   # <- small caps look nice
    ax=ax
)

# Dots (individual donors)
sb.stripplot(
    data=obs_df,
    x="Antigen",
    y="Entropy_per_donor_per_antigen",
    dodge=True,
    jitter=True,
    s=5,
    edgecolor="black",
    linewidth=0.5,
    ax=ax
)

plt.ylabel("Normalized Shannon entropy (diversity)")
plt.xlabel("Antigen")
plt.xticks(rotation=90)
# plt.ylim(-0.025, 1)

# Remove duplicate legend
# handles, labels = ax.get_legend_handles_labels()
# ax.legend(handles[0:0], labels[0:0], title="Group")
# sb.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))

sb.despine()
plt.tight_layout()
plt.show()

In [ ]:
# Get the obs_df
obs_df = mdata_rep["airr"].obs[["donor", "Antigen", "Entropy_per_donor_per_antigen"]].drop_duplicates().reset_index()
obs_df = obs_df.dropna(subset="Antigen")
obs_df["Antigen"] = obs_df["Antigen"].cat.remove_unused_categories()

#Annotate catpat
patient_list = ["Pt11", "Pt12", "Pt14", "Pt17", "Pt20"]
control_list = ["Pt1", "Pt8", "Pt18", "Pt19", "Pt21"]

# Define RRMS vs Control
obs_df["catpat"] = pd.Categorical(
    np.where(
        obs_df["donor"].isin(patient_list), "RRMS",
        np.where(obs_df["donor"].isin(control_list), "Control", "IC")
    ),
    categories=["Control", "IC", "RRMS"],
    ordered=True
)


fig, ax = plt.subplots(figsize=(5,4))

palette = {
    "RRMS": "#000000",
    "IC": "grey",
    "Control": "#FFFFFF"
}

# Column plot (mean)
sb.barplot(
    data=obs_df,
    x="Antigen",
    y="Entropy_per_donor_per_antigen",
    hue="catpat",
    color="white",
    palette={"RRMS":"white", "IC":"white", "Control":"white"},
    edgecolor="black",
    errorbar="se",   # <- adds error bars
    capsize=0.5,   # <- small caps look nice
    ax=ax
)

# Dots (individual donors)
sb.stripplot(
    data=obs_df,
    x="Antigen",
    y="Entropy_per_donor_per_antigen",
    hue="catpat",
    dodge=True,
    jitter=True,
    s=5,
    edgecolor="black",
    linewidth=0.5,
    palette=palette,
    ax=ax
)

plt.ylabel("Normalized Shannon entropy (diversity)")
plt.xlabel("Antigen")
plt.xticks(rotation=90)
# plt.ylim(-0.025, 1)

# Remove duplicate legend
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles[3:6], labels[3:6], title="Group")
sb.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))

sb.despine()
plt.tight_layout()
plt.show()

# Gini index - repertoire statistics per virus similar to Sturmlechner et al

In [ ]:
mdata_rep = mdata.copy()

#Transfer the Antigen, clone_handle and virus label to airr
mdata_rep["airr"].obs["clone_handle"] = mdata_rep["gex"].obs["clone_handle"].copy()
mdata_rep["airr"].obs["Antigen"] = mdata_rep["gex"].obs["Antigen"].copy()
mdata_rep["airr"].obs["Virus"] = mdata_rep["gex"].obs["Virus"].copy()
mdata_rep["airr"].obs["donor"] = mdata_rep["gex"].obs["donor"].copy()

#Filter the data frame
mu.pp.filter_obs(mdata_rep["gex"], "Antigen", lambda x: (np.isin(x,["EBNA1", "EBNA3", "BZLF1", "LMP2", "S", "pp65", "rep", "MP", "IE1"]))) 
mdata.update()

#Update the dataframe ---------------------
#Make sure only the indexes recorded in GEX are used in the other modalities too
index_list = mdata_rep["gex"].obs.index.tolist()

## define a cell index
mdata_rep["airr"].obs["cell_index"] = mdata_rep["airr"].obs.index
mdata_rep["prot"].obs["cell_index"] = mdata_rep["prot"].obs.index

## ".isin" using indexes does not seem to work on a global level, but if I annotate a column and then filter in place, maybe then it works. 
mdata_rep["airr"].obs["filter_index"] = pd.Categorical(np.where(mdata_rep["airr"].obs["cell_index"].isin(index_list), "GEX_present", "GEX_absent"))
mdata_rep["prot"].obs["filter_index"] = pd.Categorical(np.where(mdata_rep["prot"].obs["cell_index"].isin(index_list), "GEX_present", "GEX_absent"))

#Enforce that mdata_rep["airr"].obs and mdata_rep["prot"].obs only contain information on those indexes that exists in GEX
mu.pp.filter_obs(mdata_rep["airr"], 'filter_index', lambda x: x == "GEX_present")
mu.pp.filter_obs(mdata_rep["prot"], 'filter_index', lambda x: x == "GEX_present")
mdata_rep.update()
# -----------------------------------------

## Create the donor_antigen column and calculate the Gini index --------
mdata_rep["airr"].obs["donor_antigen"] = mdata_rep["airr"].obs["donor"].astype(str) + "_" + mdata_rep["airr"].obs["Antigen"].astype(str)

## %% Calculate the diversity metrics andsave them under a column key. 
ir.tl.alpha_diversity(mdata_rep["airr"], groupby="donor_antigen", target_col='clone_handle', metric='gini_index', inplace=True, key_added="gini_per_donor_per_antigen", airr_mod='airr')

In [ ]:
# Get the obs_df
obs_df = mdata_rep["airr"].obs[["donor", "Antigen", "gini_per_donor_per_antigen"]].drop_duplicates().reset_index()
obs_df = obs_df.dropna(subset="Antigen")
obs_df["Antigen"] = obs_df["Antigen"].cat.remove_unused_categories()


fig, ax = plt.subplots(figsize=(4,4))

# Column plot (mean)
sb.barplot(
    data=obs_df,
    x="Antigen",
    y="gini_per_donor_per_antigen",
    color="white",
    edgecolor="black",
    errorbar="se",   # <- adds error bars
    capsize=0.5,   # <- small caps look nice
    ax=ax
)

# Dots (individual donors)
sb.stripplot(
    data=obs_df,
    x="Antigen",
    y="gini_per_donor_per_antigen",
    dodge=True,
    jitter=True,
    s=5,
    edgecolor="black",
    linewidth=0.5,
    ax=ax
)

plt.ylabel("Gini index")
plt.xlabel("Antigen")
plt.xticks(rotation=90)
# plt.ylim(-0.025, 1)

# Remove duplicate legend
# handles, labels = ax.get_legend_handles_labels()
# ax.legend(handles[0:0], labels[0:0], title="Group")
# sb.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))

sb.despine()
plt.tight_layout()
plt.show()

In [ ]:
# Get the obs_df
obs_df = mdata_rep["airr"].obs[["donor", "Antigen", "gini_per_donor_per_antigen"]].drop_duplicates().reset_index()
obs_df = obs_df.dropna(subset="Antigen")
obs_df["Antigen"] = obs_df["Antigen"].cat.remove_unused_categories()

#Annotate catpat
patient_list = ["Pt11", "Pt12", "Pt14", "Pt17", "Pt20"]
control_list = ["Pt1", "Pt8", "Pt18", "Pt19", "Pt21"]

# Define RRMS vs Control
obs_df["catpat"] = pd.Categorical(
    np.where(
        obs_df["donor"].isin(patient_list), "RRMS",
        np.where(obs_df["donor"].isin(control_list), "Control", "IC")
    ),
    categories=["Control", "IC", "RRMS"],
    ordered=True
)


fig, ax = plt.subplots(figsize=(5,4))

palette = {
    "RRMS": "#000000",
    "IC": "grey",
    "Control": "#FFFFFF"
}

# Column plot (mean)
sb.barplot(
    data=obs_df,
    x="Antigen",
    y="gini_per_donor_per_antigen",
    hue="catpat",
    color="white",
    palette={"RRMS":"white", "IC":"white", "Control":"white"},
    edgecolor="black",
    errorbar="se",   # <- adds error bars
    capsize=0.5,   # <- small caps look nice
    ax=ax
)

# Dots (individual donors)
sb.stripplot(
    data=obs_df,
    x="Antigen",
    y="gini_per_donor_per_antigen",
    hue="catpat",
    dodge=True,
    jitter=True,
    s=5,
    edgecolor="black",
    linewidth=0.5,
    palette=palette,
    ax=ax
)

plt.ylabel("Gini index")
plt.xlabel("Antigen")
plt.xticks(rotation=90)
plt.ylim(-0.025, 1)

# Remove duplicate legend
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles[3:6], labels[3:6], title="Group")
sb.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))

sb.despine()
plt.tight_layout()
plt.show()

# CSF-biased score per multimer (and clonotype)

In [ ]:
### PB-biased score for completeness
## Extract score values per cell and plot as a box plot per specificity
obs_df = mdata["gex"].obs[["Virus", "handle", "CSF-biased score", "clone_handle"]].copy()
obs_df = obs_df.dropna(subset="Virus")
obs_df = obs_df.groupby(["clone_handle", "handle"], observed=True).mean("CSF-biased score").reset_index()

#plotting order: 
order = (
    obs_df
    .groupby(["handle"], observed=True)["CSF-biased score"]
    .median()
    .sort_values()
    .index
)

# Create the figure object
fig, ax = plt.subplots(figsize=(10,4))

# Stripplot (background dots)
sb.stripplot(
    data=obs_df,
    x="handle",
    y="CSF-biased score",
    jitter=0.25,
    size=4,
    alpha=0.7,
    color="black",
    zorder=1,
    order=order,
    ax=ax
)

# Boxplot (foreground)
sb.boxplot(
    data=obs_df,
    x="handle",
    y="CSF-biased score",
    width=0.5,
    color="white",
    fliersize=0,
    boxprops=dict(alpha=.5),
    linewidth=1,
    zorder=2,
    order=order,
    ax=ax
)

plt.ylabel("Mean CSF-biased score\nper clone")
plt.xlabel("Virus")
plt.xticks(rotation=90)

plt.tight_layout()
fig.savefig("../../05_plots/09_multimers/11_multimer_stratified_csf_score_clonotype.pdf", bbox_inches='tight')
plt.show()

# Graveyard

In [ ]:
IFN_ab = [
    "TANK", "PSMD14", "ADAR", "IFITM3", "IFITM2", "ADRM1", "CDC37", "USP18",
    "TREX1", "TRIM6", "CR2", "DCST1", "TRIM65", "TTLL12", "YTHDF3", "SAMHD1",
    "SIN3A", "GIGYF2", "LSM14A", "SETD2", "TBK1", "CNOT7", "UBE2K", "IFNE",
    "STING1", "IFI27", "IFIT1", "IFNA1", "IFNA2", "IFNA4", "IFNA5", "IFNA6",
    "IFNA7", "IFNA8", "IFNA10", "IFNA14", "IFNA16", "IFNA17", "IFNA21",
    "IFNAR1", "IFNAR2", "IFNB1", "IFNW1", "IRAK1", "IRF3", "IRF7", "JAK1",
    "USP27X", "SMIM30", "MIR21", "MMP12", "MX1", "MYD88", "OAS1", "OAS2",
    "OAS3", "YTHDF2", "RBM47", "SHFL", "METTL3", "PSMA1", "PSMA2", "IFNK",
    "PSMA3", "PSMA4", "PSMA5", "PSMA6", "PSMA7", "PSMB1", "PSMB2", "PSMB3",
    "PSMB4", "GPR108", "PSMB5", "PSMB6", "PSMB7", "PSMC1", "PSMC2", "PSMC3",
    "PSMC4", "PSMC5", "PSMC6", "PSMD1", "PSMD2", "PSMD3", "PSMD4", "PSMD7",
    "PSMD8", "PSMD11", "PSMD12", "PSMD13", "PSME1", "PSME2", "MAVS", "USP29",
    "PTPN1", "PTPN2", "PTPN6", "PTPN11", "CACTIN", "IFIH1", "AZI2", "SHMT2",
    "SMPD1", "SP100", "STAT1", "STAT2", "TRAF3", "TYK2", "WNT5A", "MUL1",
    "SEM1", "ZBP1", "TRIM56", "NLRC5", "IFITM1", "OASL", "FADD", "CH25H",
    "TRIM41", "RNF185", "EIF4E2", "ISG15", "IKBKE", "TBKBP1", "HDAC4", "PSMD6"
]

sc.tl.score_genes(mdata["gex"], 
                      IFN_ab,  
                      score_name='Response to type I IFN (GOBP)')

In [ ]:
sc.settings.set_figure_params(figsize=(5, 4))
fig = sc.pl.umap(mdata["gex"], color="Response to type I IFN (GOBP)", legend_fontweight= 'bold', frameon= False , size=30, vmin=0, legend_fontoutline=2, return_fig=True, show=False, cmap="RdBu_r")
plt.tight_layout()
# fig.savefig("../../05_plots/09_multimers/13_scANVI_labels_on_multimer_intergration.pdf",  format="pdf", dpi=400)
plt.show

In [ ]:
## Confusion matrix on specific phenotypes
obs_df = mdata["gex"].obs.copy()
obs_df = obs_df[["donor", "L3_ann_overview_scanvi"]]

conf_mat = pd.crosstab(
    obs_df["donor"],
    obs_df["L3_ann_overview_scanvi"]
)

print(conf_mat)

In [ ]:
sc.settings.set_figure_params(figsize=(4, 4))
fig = sc.pl.umap(mdata["gex"], color="C_scANVI", legend_fontweight= 'bold', palette=cluster_color_dict,  frameon= False , size=20, vmin=0, legend_fontoutline=2, return_fig=True, show=False)
plt.tight_layout()
fig.savefig("../../05_plots/09_multimers/13_scANVI_labels_on_multimer_intergration.pdf",  format="pdf", dpi=400)
plt.show

In [ ]:
sc.settings.set_figure_params(figsize=(6, 4))

# copy palette so you don't overwrite the original
palette = cluster_color_dict.copy()

# set the group you want to highlight to black
palette["25 TA (CD3Elo Ribolo)"] = "black"

fig = sc.pl.umap(mdata["gex"], color="C_scANVI", groups=["25 TA (CD3Elo Ribolo)"], palette=palette, legend_fontweight= 'bold', frameon= False , size=20, vmin=0, legend_fontoutline=2, return_fig=True, show=False)
plt.tight_layout()
# fig.savefig("../../05_plots/09_multimers/13_scANVI_labels_on_multimer_intergration.pdf",  format="pdf", dpi=400)
plt.show

In [ ]:
### Specific to activated and ribolo
marker_genes = {
    "IFN" : ["ISG15", "MX1"],
    "TN" : ["SELL", "CCR7", "BACH2"],
    "TCM" : ["TCF7", "LTB", "KLF2"], 
    "TEM" : ["CD27", "GZMK"], 
    "TE": ["GNLY", "GZMB", "NKG7", "PRF1", "GZMH",  "CX3CR1", "IKZF2"],
    "NK-like" : ["KIR3DL2", "KIR3DL1", "TYROBP", "KLRD1", "KLRC2"],
    "GD" : ["TRDV1"], 
    "CSF-TE" : ["GZMK", "CD74", "HLA-DRA", "HLA-DRB1", "IL7R", "ITGA4", "ITGB1"],
    "CSF-TR" : ["CD69", "ZNF683", "XCL1", "CXCR6"],
    "TA": ["NFKBIA", "BATF3", "JUNB", "FOS"],
    
}

# # Define a custom order:
# Cluster_order = ["0", "1", "2", "6", "15",  #Tn
#                 "24", "11", "7",   #Tcm
#                  "8","5",         #Tem
#                  "13", "14",       #Temra
#                  "10", "12", "9",      #Effector cells
#                  "18",                      #TRDV1+ cells
#                  "17","23", "20","21", "25", 
#                  "3", "4", "16", "19", "22" ]    #Tcsf]    #Activated

# #Implement order
# sdata["gex"].obs["leiden_1.9"] = sdata["gex"].obs["leiden_1.9"].cat.reorder_categories(
#     Cluster_order, ordered=True
# )

dp = sc.pl.dotplot(
    mdata["gex"],
    marker_genes,
    groupby="L3_ann_overview_scanvi",
    standard_scale="var",
    dendrogram=False,
    cmap="RdBu_r",
    swap_axes=False,
    show=False,
    return_fig=True,
)

dp.savefig(
    "../../05_plots/09_multimers/15_bubbleplot_general_cd8.pdf",
    bbox_inches="tight",
    dpi=400
)


In [ ]:
### Specific to activated and ribolo
marker_genes = {
    "IFN" : ["ISG15", "MX1"],
    "TN" : ["SELL", "CCR7", "BACH2"],
    "TCM" : ["TCF7", "LTB", "KLF2"], 
    "TEM" : ["CD27", "GZMK"], 
    "TE": ["GNLY", "GZMB", "NKG7", "PRF1", "GZMH",  "CX3CR1", "IKZF2"],
    "NK-like" : ["KIR3DL2", "KIR3DL1", "TYROBP", "KLRD1", "KLRC2"],
    "GD" : ["TRDV1"], 
    "CSF-TE" : ["GZMK", "CD74", "HLA-DRA", "HLA-DRB1", "IL7R", "ITGA4", "ITGB1"],
    "CSF-TR" : ["CD69", "ZNF683", "XCL1", "CXCR6"],
    "TA": ["NFKBIA", "BATF3", "JUNB", "FOS"],
    
}

# # Define a custom order:
# Cluster_order = ["0", "1", "2", "6", "15",  #Tn
#                 "24", "11", "7",   #Tcm
#                  "8","5",         #Tem
#                  "13", "14",       #Temra
#                  "10", "12", "9",      #Effector cells
#                  "18",                      #TRDV1+ cells
#                  "17","23", "20","21", "25", 
#                  "3", "4", "16", "19", "22" ]    #Tcsf]    #Activated

# #Implement order
# sdata["gex"].obs["leiden_1.9"] = sdata["gex"].obs["leiden_1.9"].cat.reorder_categories(
#     Cluster_order, ordered=True
# )

dp = sc.pl.dotplot(
    mdata["gex"],
    marker_genes,
    groupby="C_scANVI",
    standard_scale="var",
    dendrogram=False,
    cmap="RdBu_r",
    swap_axes=False,
    show=False,
    return_fig=True,
)

dp.savefig(
    "../../05_plots/09_multimers/15_bubbleplot_general_cd8.pdf",
    bbox_inches="tight",
    dpi=400
)


In [ ]:
## Confusion matrix on specific phenotypes
obs_df = mdata["gex"].obs.copy()
obs_df = obs_df[["donor", "C_scANVI"]]

conf_mat = pd.crosstab(
    obs_df["donor"],
    obs_df["C_scANVI"]
)

conf_mat

In [ ]:
# normalize columns
conf_mat_colnorm = conf_mat.div(conf_mat.sum(axis=0), axis=1) * 100

fig, ax = plt.subplots(figsize=(15, 8))

sb.heatmap(
    conf_mat_colnorm,
    annot=True,
    fmt=".1f",
    linewidths=0.5,
    cbar=True,
    ax=ax
)

ax.set_xlabel("C_scANVI")
ax.set_ylabel("Donor")
plt.title("% of each cluster")

plt.tight_layout()
plt.show()

In [ ]:
# normalize columns
conf_mat_colnorm = conf_mat.div(conf_mat.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(15, 8))

sb.heatmap(
    conf_mat_colnorm,
    annot=True,
    fmt=".1f",
    linewidths=0.5,
    cbar=True,
    ax=ax
)

ax.set_xlabel("C_scANVI")
ax.set_ylabel("Donor")
plt.title("% of each donor")

plt.tight_layout()
plt.show()

# Define HVG genes without batch information

In [ ]:
## Default highly variable genes
sc.pp.highly_variable_genes(mdata["gex"], 
                            n_top_genes=5000, #!! 5000 genes creates a spread out umap. 
                            flavor='seurat',  #This mimics select intergration features
                            # batch_key="batch_handle",   #If a batch was "batch" - then the CITEseq multiplexed instances would immediate mark donor-specific expression of AC004448.2 and OOEP fx. this is not intended.
                           )

In [ ]:
### Remove TCR genes

#Bring gene names from index into a column
mdata["gex"].var = mdata["gex"].var.reset_index()

# Optionally rename the new column
mdata["gex"].var = mdata["gex"].var.rename(columns={"index": "genes"})

# Rename the highly_varaible_intersection cells according to an ifelse statement
tcr_gene_prefixs = ['TRAV', 'TRAJ', 'TRBV', "TRBD", "TRBJ"] 

pattern = r"^(" + "|".join(tcr_gene_prefixs) + ")"
mask = mdata["gex"].var["genes"].str.contains(pattern, regex=True)
masked_genes = mdata["gex"].var[mask]["genes"].tolist() #Works - it's giving me the exact genes I want except the D-region genes (although there is only 2? I will ask Kilian about it. Lea says there is something like that with mice, that htey don't express it)

#Modify the initial list of highly variable genes to exclude TCR genes
mdata["gex"].var["highly_variable"] = np.where(mdata["gex"].var["genes"].isin(masked_genes), False, mdata["gex"].var["highly_variable"])

# Return the genes to index
mdata["gex"].var = mdata["gex"].var.set_index(["genes"], drop=True)


In [ ]:
sc.tl.pca(mdata["gex"])
sc.pp.neighbors(mdata["gex"])
sc.tl.umap(mdata["gex"])

In [ ]:
sc.pl.umap(mdata["gex"], color="batch", legend_loc="on data", legend_fontweight= 'bold', frameon= False , size=5, vmin=0, legend_fontoutline=2)

# Define HVG genes with batch information

In [ ]:
#Removes columns of the var table. This is to rebuild the HVG table. 
mdata["gex"].var = mdata["gex"].var[[]]

## What is a batch?
## We want a donor aware + batch aware intergration (donor + batch)
mdata["gex"].obs["batch_handle"] = mdata["gex"].obs["batch"].astype(str) + "_" + mdata["gex"].obs["donor"].astype(str)

## Default highly variable genes
sc.pp.highly_variable_genes(mdata["gex"], 
                            n_top_genes=5000, #!! 5000 genes creates a spread out umap. 7500 HVGs creates a more dense umap.
                            flavor='seurat',  #This mimics select intergration features
                            batch_key="batch_handle",   #If a batch was "batch" - then the CITEseq multiplexed instances would immediate mark donor-specific expression of AC004448.2 and OOEP fx. this is not intended.
                           )

In [ ]:
### Remove TCR genes

#Bring gene names from index into a column
mdata["gex"].var = mdata["gex"].var.reset_index()

# Optionally rename the new column
mdata["gex"].var = mdata["gex"].var.rename(columns={"index": "genes"})

# Rename the highly_varaible_intersection cells according to an ifelse statement
tcr_gene_prefixs = ['TRAV', 'TRAJ', 'TRBV', "TRBD", "TRBJ"] 

pattern = r"^(" + "|".join(tcr_gene_prefixs) + ")"
mask = mdata["gex"].var["genes"].str.contains(pattern, regex=True)
masked_genes = mdata["gex"].var[mask]["genes"].tolist() #Works - it's giving me the exact genes I want except the D-region genes (although there is only 2? I will ask Kilian about it. Lea says there is something like that with mice, that htey don't express it)

#Modify the initial list of highly variable genes to exclude TCR genes
mdata["gex"].var["highly_variable"] = np.where(mdata["gex"].var["genes"].isin(masked_genes), False, mdata["gex"].var["highly_variable"])

# Return the genes to index
mdata["gex"].var = mdata["gex"].var.set_index(["genes"], drop=True)


In [ ]:
## PCA
sc.tl.pca(mdata["gex"], n_comps=50)
sc.pl.pca_variance_ratio(mdata["gex"], n_pcs=40, log=True) #Log tranformeD. 


In [ ]:
sc.pp.neighbors(mdata["gex"], n_pcs=30)
sc.tl.umap(mdata["gex"])

In [ ]:
sc.pl.umap(mdata["gex"], color="batch", legend_loc="on data", legend_fontweight= 'bold', frameon= False , size=5, vmin=0, legend_fontoutline=2)

In [ ]:
sc.pl.umap(mdata["gex"], color="donor", legend_loc="on data", legend_fontweight= 'bold', frameon= False , size=5, vmin=0, legend_fontoutline=2)

In [ ]:
#All naive and memory markers
sc.tl.rank_genes_groups(mdata["gex"], groupby="batch", groups=["NK13_1"], reference="NK13_2", key_added="upX")
sc.tl.rank_genes_groups(mdata["gex"], groupby="batch",  groups=["NK13_2"], reference="NK13_1", key_added="upY")

In [ ]:
sc.settings.set_figure_params(figsize=(6, 5))
sc.pl.rank_genes_groups(mdata["gex"], key="upX", n_genes=20, fontsize=15)
sc.pl.rank_genes_groups(mdata["gex"], key="upY", n_genes=20, fontsize=15)

In [ ]:
sc.pl.umap(mdata["gex"], color=["HLA-A"], cmap="RdBu_r", legend_fontweight= 'bold', frameon= False , size=5, vmin=0, legend_fontoutline=2)

## Of matched CD8 T cells - show the CSF-biased sharing

In [ ]:
obs_df = sdata["gex"].obs.dropna(subset="Virus")
obs_df = obs_df[~obs_df["donor"].isin(["Pt4", "Pt5"])]

obs_df = obs_df[["Virus", "clone_handle", "Compartment", "donor", "catpat"]].value_counts().reset_index()
obs_df["total_cells"] = obs_df.groupby(["donor", "catpat", "clone_handle"], observed=True)["count"].transform("sum")
obs_df["comp_pct"]= 100*obs_df["count"]/obs_df["total_cells"]

# Generate an average for EBV, FLU-A and CMV
obs_df = obs_df[obs_df["Virus"].isin(["CMV", "EBV", "FLU-A"])]
obs_df = obs_df.pivot_table(index=["Virus", "donor", "clone_handle", "catpat"], columns="Compartment", values="comp_pct", fill_value=0).reset_index()
obs_df["x_cat"]= obs_df["Virus"].astype(str) + "_" + obs_df["catpat"].astype(str)


In [ ]:
#Sort
plot_df = obs_df.sort_values(["x_cat", "CSF"], ascending=[True, False]).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(10, 3))

x = range(len(plot_df))

# Bottom layer: CSF
ax.bar(
    x,
    plot_df["CSF"],
    width=1.0,
    label="CSF"
)

# Top layer: PB stacked on CSF
ax.bar(
    x,
    plot_df["PB"],
    bottom=plot_df["CSF"],
    width=1.0,
    label="PB"
)

ax.set_xticks([])
ax.set_xlabel("clone_handle (sorted by x_cat)")
ax.set_ylabel("Percentage")

ax.legend(frameon=False)
sb.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))

plt.tight_layout()
plt.show()

In [ ]:
plot_df["global_PB"] = plot_df.groupby("x_cat", observed=True)["PB"].transform("mean")
plot_df["global_CSF"] = plot_df.groupby("x_cat", observed=True)["CSF"].transform("mean")

In [ ]:
# plot_df2 = plot_df[["x_cat", "global_PB", "global_CSF"]].drop_duplicates()
# plot_df2 = plot_df2.copy()

# x = np.arange(len(plot_df2))
# csf_color = "#4B8AC1"   # muted blue
# pb_color  = "#C06A6A"   # muted orange

# fig, ax = plt.subplots(figsize=(6, 4))

# # CSF (bottom)
# ax.bar(
#     x,
#     plot_df2["global_CSF"],
#     label="CSF",
#     color=csf_color
# )

# # PB (stacked on top)
# ax.bar(
#     x,
#     plot_df2["global_PB"],
#     bottom=plot_df2["global_CSF"],
#     label="PB",
#     color=pb_color
# )

# ax.set_xticks(x)
# ax.set_xticklabels(plot_df2["x_cat"], rotation=45, ha="right")

# ax.set_ylabel("Percentage")
# ax.set_xlabel("Condition")
# plt.title("Mean % CSF of matched virus-specific clones")

# ax.legend(frameon=False)
# sb.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))

# plt.tight_layout()
# plt.show()

#### Conclusion: Batch effects seem to be centered on the unequal expression of HLA, ribosomal and mitochondrial genes. 

In [ ]:
def build_freq_df(mdata, thresholds):
    
    # Group annotation
    patient_list = ["Pt11", "Pt12", "Pt14", "Pt17", "Pt20"]
    control_list = ["Pt1", "Pt8", "Pt18", "Pt19", "Pt21"]
    
    out_list = []

    for thr in thresholds:

        ## Extract Virus and handle per cell
        obs_df = mdata["gex"].obs[
            ["donor", "total_clone_count", "Virus", "handle",
             "catsort_A2", "catsort_virus", "total_agspecific_cells",
             "catagspecifc"]
        ].copy()

        ## Remove non-specific cells
        obs_df = obs_df[obs_df["catagspecifc"] == "agspecific"]

        ## Filter by clone size threshold
        obs_df = obs_df[obs_df["total_clone_count"] >= thr]

        ## Multimer frequency among antigen-specific cells
        obs_df["multimer_cellcout"] = (
            obs_df.groupby(["donor", "handle"], observed=True)
            .transform("size")
        )

        obs_df["mult_frac_agsp"] = (
            obs_df["multimer_cellcout"] /
            obs_df["total_agspecific_cells"]
        )

        ## Load flow frequencies
        excel_df = pd.read_excel(
            # "../../03_annotation/11_multimer_experiment_ann/2026_03 - FlowFreq_FromSortGates.xlsx"
            "../../03_annotation/11_multimer_experiment_ann/2026_03 - FlowFreq_FromRegate.xlsx"
        )

        excel_df = pd.melt(
            excel_df,
            id_vars=["donor", "catsort_A2"],
            var_name="catsort_virus",
            value_vars=["EBV_sort_gate_freq", "Non_EBV_sort_gate_freq"],
            value_name="sort_freq"
        )

        excel_df["catsort_virus"] = (
            excel_df["catsort_virus"]
            .str.replace("_gate_freq", "", regex=False)
            .str.strip()
        )

        excel_df["catsort_A2"] = excel_df["catsort_A2"].str.strip()

        ## Merge
        obs_df = obs_df.merge(
            excel_df,
            how="left",
            on=["donor", "catsort_A2", "catsort_virus"]
        )

        ## Estimated frequency
        obs_df["Est_freq"] = obs_df["sort_freq"] * obs_df["mult_frac_agsp"]

        ## Patient groups
        obs_df["catpat"] = pd.Categorical(
            np.where(
                obs_df["donor"].isin(patient_list), "RRMS",
                np.where(obs_df["donor"].isin(control_list), "Control", "IC")
            ),
            categories=["Control", "IC", "RRMS"],
            ordered=True
        )

        ## Cleanup
        obs_df = (
            obs_df[["donor", "Virus", "catpat", "Est_freq", "mult_frac_agsp"]]
            .dropna(subset="mult_frac_agsp")
            .drop_duplicates()
        )

        ## Filter EBV
        obs_df = obs_df[obs_df["Virus"] == "EBV"]
        obs_df["Virus"] = obs_df["Virus"].cat.remove_unused_categories()

        ## Remove IC
        obs_df = obs_df[obs_df["catpat"] != "IC"]
        obs_df["catpat"] = obs_df["catpat"].cat.remove_unused_categories()

        ## Store threshold
        obs_df["clone_threshold"] = thr

        out_list.append(obs_df)

    ## Concatenate all thresholds
    final_df = pd.concat(out_list, ignore_index=True)

    return final_df

In [ ]:
plot_df = build_freq_df(mdata, thresholds=range(0,20))

#Plot Est_freq
# Create the figure object
fig, ax = plt.subplots(figsize=(10,4))

# Boxplot (foreground)
sb.boxplot(
    data=plot_df,
    x="clone_threshold",
    y="Est_freq",
    hue="catpat",
    width=0.75,
    color="white",
    showcaps=True,
    boxprops={'facecolor': 'white'},
    showfliers=False,
    palette=palette,
    ax=ax
)

# Stripplot (background dots)
sb.stripplot(
    data=plot_df,
    x="clone_threshold",
    y="Est_freq",
    hue="catpat",
    dodge=True,
    s=5,
    jitter=True,
    alpha=0.8,
    linewidth=0.5,
    edgecolor="black",
    palette=palette,
    ax=ax
)

ax.set_yscale("log")
plt.ylabel("Est. frequency of EBV-specific cells (%)")
plt.xlabel("Expansion thresholds")
plt.xticks(rotation=90)
plt.title("Each dot is one multimer+ population")

# Remove duplicate legend and move the legend
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles[2:4], labels[2:4], title="Group")
sb.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))

#Despine
sb.despine()

plt.tight_layout()
# fig.savefig("../../05_plots/09_multimers/7_mean_clone_CSF_biased_score.pdf", bbox_inches='tight')
plt.show()

In [ ]:
plot_df = build_freq_df(mdata, thresholds=range(0,20))

#Aggregate responses from the same donor
plot_df = (
    plot_df
    .groupby(["donor", "Virus", "catpat", "clone_threshold" ], observed=True, as_index=False)["Est_freq"]
    .mean()
)

#Plot Est_freq
# Create the figure object
fig, ax = plt.subplots(figsize=(10,4))

# Boxplot (foreground)
sb.boxplot(
    data=plot_df,
    x="clone_threshold",
    y="Est_freq",
    hue="catpat",
    width=0.75,
    color="white",
    showcaps=True,
    boxprops={'facecolor': 'white'},
    showfliers=False,
    palette=palette,
    ax=ax
)

# Stripplot (background dots)
sb.stripplot(
    data=plot_df,
    x="clone_threshold",
    y="Est_freq",
    hue="catpat",
    dodge=True,
    s=5,
    jitter=True,
    alpha=0.8,
    linewidth=0.5,
    edgecolor="black",
    palette=palette,
    ax=ax
)

ax.set_yscale("log")
plt.ylabel("Est. frequency of EBV-specific cells (%)")
plt.xlabel("Expansion thresholds")
plt.xticks(rotation=90)
plt.title("Each dot is one donor")

# Remove duplicate legend and move the legend
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles[2:4], labels[2:4], title="Group")
sb.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))

#Despine
sb.despine()

plt.tight_layout()
# fig.savefig("../../05_plots/09_multimers/7_mean_clone_CSF_biased_score.pdf", bbox_inches='tight')
plt.show()

In [ ]:
## Extract Virus and handle per cell
obs_df = mdata["gex"].obs[["donor", "Virus", "Antigen", "handle","catsort_A2", "catsort_virus",  "total_agspecific_cells", "catagspecifc"]].copy()

## Remove all non-specific cells
obs_df = obs_df[obs_df["catagspecifc"] == "agspecific"]

## Calculate multimer frequency among antigen-specific cells for each donor
obs_df["multimer_cellcout"] = obs_df.groupby(["donor", "handle"], observed=True).transform("size")
obs_df["mult_frac_agsp"]= obs_df["multimer_cellcout"] / obs_df["total_agspecific_cells"]

## Load the total freq. of multimer+ cells
excel_df = pd.read_excel("../../03_annotation/11_multimer_experiment_ann/2026_03 - FlowFreq_FromSortGates.xlsx")
# excel_df = pd.read_excel("../../03_annotation/11_multimer_experiment_ann/2026_03 - FlowFreq_FromRegate.xlsx")
excel_df = pd.melt(excel_df, id_vars=["donor", "catsort_A2"], var_name="catsort_virus", value_vars=["EBV_sort_gate_freq", "Non_EBV_sort_gate_freq"], value_name="sort_freq")

#Remove  "_gate_freq" from EBV_sort and Non_EBV_sort column. 
excel_df["catsort_virus"] = excel_df["catsort_virus"].str.replace("_gate_freq", "", regex=False)

## Remove hidden whitespace
excel_df["catsort_virus"] = excel_df["catsort_virus"].str.strip()
excel_df["catsort_A2"] = excel_df["catsort_A2"].str.strip()

## Merge the two data frames
obs_df = obs_df.merge(excel_df, how="left", on=["donor", "catsort_A2", "catsort_virus"])

## Take the fraction (mult_frac_agspc) of sort_freq to extrapolate the estimated frequency
obs_df["Est_freq"] = obs_df["sort_freq"]*obs_df["mult_frac_agsp"]

## Drop na - these likely belongs to FRC responses and other excluded CMV-responses
obs_df = obs_df.dropna(subset="mult_frac_agsp").drop_duplicates()

#Filter on EBV
obs_df = obs_df[obs_df["Virus"] == "EBV"]
obs_df["Virus"] = obs_df["Virus"].cat.remove_unused_categories()


## Catpat
# Group annotation
patient_list = ["Pt11", "Pt12", "Pt14", "Pt17", "Pt20"]
control_list = ["Pt1", "Pt8", "Pt18", "Pt19", "Pt21"]

# Define RRMS vs Control
obs_df["catpat"] = pd.Categorical(
    np.where(
        obs_df["donor"].isin(patient_list), "RRMS",
        np.where(obs_df["donor"].isin(control_list), "Control", "IC")
    ),
    categories=["Control", "IC", "RRMS"],
    ordered=True
)

#Remove ICs to be consistent
obs_df = obs_df[obs_df["catpat"] != "IC"]
obs_df["catpat"] = obs_df["catpat"].cat.remove_unused_categories()

#Aggregate responses from the same donor
obs_df = (
    obs_df
    .groupby(["donor", "Virus", "catpat"], observed=True, as_index=False)["Est_freq"]
    .mean()
)

#Plot Est_freq
# Create the figure object
fig, ax = plt.subplots(figsize=(5,4))

# Boxplot (foreground)
sb.boxplot(
    data=obs_df,
    x="Virus",
    y="Est_freq",
    hue="catpat",
    width=0.75,
    color="white",
    showcaps=True,
    boxprops={'facecolor': 'white'},
    showfliers=False,
    palette=palette,
    ax=ax
)

# Stripplot (background dots)
sb.stripplot(
    data=obs_df,
    x="Virus",
    y="Est_freq",
    hue="catpat",
    dodge=True,
    s=5,
    jitter=True,
    alpha=0.8,
    linewidth=0.5,
    edgecolor="black",
    palette=palette,
    ax=ax
)

ax.set_yscale("log")
plt.ylabel("Average EBV-specific est. frequency (%)")
plt.xlabel("Virus")
plt.xticks(rotation=90)
plt.title("Each dot is one donor")

# Remove duplicate legend and move the legend
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles[3:6], labels[3:6], title="Group")
sb.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))

#Despine
sb.despine()

plt.tight_layout()
# fig.savefig("../../05_plots/09_multimers/7_mean_clone_CSF_biased_score.pdf", bbox_inches='tight')
plt.show()

# # Kruskal–Wallis global test
# groups = [
#     g["Est_freq"].values
#     for _, g in obs_df.groupby("catpat")
# ]

# H, p_kw = stats.kruskal(*groups)
# print("Kruskal-Wallis p =", p_kw)

# # Dunn's post-hoc test
# dunn_df = sp.posthoc_dunn(
#     obs_df,
#     val_col="Est_freq",
#     group_col="catpat",
#     p_adjust="fdr_bh"
# )

# print(dunn_df)

# Extract values
ctrl_vals = obs_df.loc[obs_df["catpat"] == "Control", "Est_freq"]
rrms_vals = obs_df.loc[obs_df["catpat"] == "RRMS", "Est_freq"]

# Mann–Whitney test
stat, pval = mannwhitneyu(ctrl_vals, rrms_vals, alternative="two-sided")

print("Mann–Whitney p-value:", pval)

In [ ]:
plot_df = build_freq_df(mdata, thresholds=range(0,20))

#Aggregate responses from the same donor
plot_df = (
    plot_df
    .groupby(["donor", "Virus", "catpat", "clone_threshold"], observed=True, as_index=False)["Est_freq"]
    .mean()
)

#Plot Est_freq
# Create the figure object
fig, ax = plt.subplots(figsize=(10,4))

# Boxplot (foreground)
sb.boxplot(
    data=plot_df,
    x="clone_threshold",
    y="Est_freq",
    hue="catpat",
    width=0.75,
    color="white",
    showcaps=True,
    boxprops={'facecolor': 'white'},
    showfliers=False,
    palette=palette,
    ax=ax
)

# Stripplot (background dots)
sb.stripplot(
    data=plot_df,
    x="clone_threshold",
    y="Est_freq",
    hue="catpat",
    dodge=True,
    s=5,
    jitter=True,
    alpha=0.8,
    linewidth=0.5,
    edgecolor="black",
    palette=palette,
    ax=ax
)

ax.set_yscale("log")
plt.ylabel("Estimated frequency (%)")
plt.xlabel("Expansion thresholds")
plt.xticks(rotation=90)
plt.title("Each dot is one donor")

# Remove duplicate legend and move the legend
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles[3:6], labels[3:6], title="Group")
sb.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))

#Despine
sb.despine()

plt.tight_layout()
# fig.savefig("../../05_plots/09_multimers/7_mean_clone_CSF_biased_score.pdf", bbox_inches='tight')
plt.show()

In [ ]:
## Extract Virus and handle per cell
obs_df = mdata["gex"].obs[["donor", "Virus", "Antigen", "handle","catsort_A2", "catsort_virus",  "total_agspecific_cells", "catagspecifc"]].copy()

## Remove all non-specific cells
obs_df = obs_df[obs_df["catagspecifc"] == "agspecific"]

## Calculate multimer frequency among antigen-specific cells for each donor
obs_df["multimer_cellcout"] = obs_df.groupby(["donor", "handle"], observed=True).transform("size")
obs_df["mult_frac_agsp"]= obs_df["multimer_cellcout"] / obs_df["total_agspecific_cells"]

## Load the total freq. of multimer+ cells
# excel_df = pd.read_excel("../../03_annotation/11_multimer_experiment_ann/2026_03 - FlowFreq_FromSortGates.xlsx")
excel_df = pd.read_excel("../../03_annotation/11_multimer_experiment_ann/2026_03 - FlowFreq_FromRegate.xlsx")
excel_df = pd.melt(excel_df, id_vars=["donor", "catsort_A2"], var_name="catsort_virus", value_vars=["EBV_sort_gate_freq", "Non_EBV_sort_gate_freq"], value_name="sort_freq")

#Remove  "_gate_freq" from EBV_sort and Non_EBV_sort column. 
excel_df["catsort_virus"] = excel_df["catsort_virus"].str.replace("_gate_freq", "", regex=False)

## Remove hidden whitespace
excel_df["catsort_virus"] = excel_df["catsort_virus"].str.strip()
excel_df["catsort_A2"] = excel_df["catsort_A2"].str.strip()

## Merge the two data frames
obs_df = obs_df.merge(excel_df, how="left", on=["donor", "catsort_A2", "catsort_virus"])

## Take the fraction (mult_frac_agspc) of sort_freq to extrapolate the estimated frequency
obs_df["Est_freq"] = obs_df["sort_freq"]*obs_df["mult_frac_agsp"]

## Drop na - these likely belongs to FRC responses and other excluded CMV-responses
obs_df = obs_df.dropna(subset="mult_frac_agsp").drop_duplicates()

#Filter on EBV
obs_df = obs_df[obs_df["Virus"] == "EBV"]
obs_df["Virus"] = obs_df["Virus"].cat.remove_unused_categories()

#Filter on Antigen
obs_df = obs_df[obs_df["Antigen"].isin(["EBNA1", "EBNA3", "LMP2", "BZLF1"])]
obs_df["Virus"] = obs_df["Antigen"].cat.remove_unused_categories()  #Overwrite virus here, as it creates the analysis you are interested in

## Catpat
# Group annotation
patient_list = ["Pt11", "Pt12", "Pt14", "Pt17", "Pt20"]
control_list = ["Pt1", "Pt8", "Pt18", "Pt19", "Pt21"]

# Define RRMS vs Control
obs_df["catpat"] = pd.Categorical(
    np.where(
        obs_df["donor"].isin(patient_list), "RRMS",
        np.where(obs_df["donor"].isin(control_list), "Control", "IC")
    ),
    categories=["Control", "IC", "RRMS"],
    ordered=True
)

#Remove ICs to be consistent
obs_df = obs_df[obs_df["catpat"] != "IC"]
obs_df["catpat"] = obs_df["catpat"].cat.remove_unused_categories()

#Aggregate responses from the same donor
obs_df = (
    obs_df
    .groupby(["donor", "Virus", "catpat"], observed=True, as_index=False)["Est_freq"]
    .mean()
)

#Plot Est_freq
# Create the figure object
fig, ax = plt.subplots(figsize=(5,4))

# Boxplot (foreground)
sb.boxplot(
    data=obs_df,
    x="Virus",
    y="Est_freq",
    hue="catpat",
    width=0.75,
    color="white",
    showcaps=True,
    boxprops={'facecolor': 'white'},
    showfliers=False,
    palette=palette,
    ax=ax
)

# Stripplot (background dots)
sb.stripplot(
    data=obs_df,
    x="Virus",
    y="Est_freq",
    hue="catpat",
    dodge=True,
    s=5,
    jitter=True,
    alpha=0.8,
    linewidth=0.5,
    edgecolor="black",
    palette=palette,
    ax=ax
)

ax.set_yscale("log")
plt.ylabel("Average EBV-specific est. frequency (%)")
plt.xlabel("Virus")
plt.xticks(rotation=90)
plt.title("Each dot is one donor")

# Remove duplicate legend and move the legend
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles[3:6], labels[3:6], title="Group")
sb.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))

#Despine
sb.despine()

plt.tight_layout()
# fig.savefig("../../05_plots/09_multimers/7_mean_clone_CSF_biased_score.pdf", bbox_inches='tight')
plt.show()

# # Kruskal–Wallis global test
# groups = [
#     g["Est_freq"].values
#     for _, g in obs_df.groupby("catpat")
# ]

# H, p_kw = stats.kruskal(*groups)
# print("Kruskal-Wallis p =", p_kw)

# # Dunn's post-hoc test
# dunn_df = sp.posthoc_dunn(
#     obs_df,
#     val_col="Est_freq",
#     group_col="catpat",
#     p_adjust="fdr_bh"
# )

# print(dunn_df)

# Extract values
ctrl_vals = obs_df.loc[obs_df["catpat"] == "Control", "Est_freq"]
rrms_vals = obs_df.loc[obs_df["catpat"] == "RRMS", "Est_freq"]

# Mann–Whitney test
stat, pval = mannwhitneyu(ctrl_vals, rrms_vals, alternative="two-sided")

print("Mann–Whitney p-value:", pval)

In [ ]:
# # -----------------------------
# # Filter
# # -----------------------------
# obs_df = sdata["gex"].obs.dropna(subset=["Virus"])
# obs_df = obs_df[~obs_df["donor"].isin(["Pt4", "Pt5"])]
# obs_df = obs_df[obs_df["Virus"].isin(["CMV", "EBV", "FLU-A"])]

# # -----------------------------
# # Create grouping variable
# # -----------------------------
# obs_df["x_cat"] = obs_df["Virus"].astype(str) + "_" + obs_df["catpat"].astype(str)

# # -----------------------------
# # Count cells per group
# # -----------------------------
# count_df = (
#     obs_df
#     .groupby(["x_cat", "clone_handle", "Compartment"], observed=True)
#     .size()
#     .reset_index(name="count")
# )

# # Total per x_cat
# count_df["total_cells"] = count_df.groupby(["x_cat", "clone_handle"], observed=True)["count"].transform("sum")

# # Percentage
# count_df["comp_pct"] = 100 * count_df["count"] / count_df["total_cells"]

# # CSF percentage per donor
# plot_df = count_df[count_df["Compartment"] == "CSF"]

# fig, ax = plt.subplots(figsize=(5, 4))

# sb.barplot(
#     data=plot_df,
#     x="x_cat",
#     y="comp_pct",
#     errorbar="sd",   # <-- standard error
#     capsize=0.2,
#     ax=ax
# )

# sb.stripplot(
#     data=plot_df,
#     x="x_cat",
#     y="comp_pct",
#     color="black",
#     size=5,
#     jitter=True,
#     ax=ax
# )

# # Formatting
# ax.set_xlabel("Condition")
# ax.set_ylabel("CSF (%)")
# ax.set_title("CSF proportion per clone ± SD\namong cells from matched clonotypes")

# plt.xticks(rotation=45, ha="right")
# plt.tight_layout()
# plt.show()

# Heatmap of virus-specificities vs CSF-biased gene score

In [ ]:
DEG_biased = pd.read_csv("../../06_csv_outputs/DEG/CD8_CSF_biased_filtered_DEGs.csv")["names"].drop_duplicates().tolist()
print(len(DEG_biased))

In [ ]:
DEG_biased = [str(g) for g in DEG_biased]  # avoids np.str_ issues
valid_genes = list(set(DEG_biased).intersection(mdata["gex"].var_names))

len(valid_genes)

In [ ]:
missing = sorted(set(DEG_biased) - set(valid_genes))
print("Missing genes:", missing)

In [ ]:
# Work with clones of interest (those with an annotated antigen-specificity)
mtemp = mdata.copy()
print(len(mtemp["gex"]))
mu.pp.filter_obs(mtemp["gex"], "Virus", lambda x: (~x.isna()))
mu.pp.filter_obs(mtemp["gex"], "donor", lambda x: (~x.isin(["Pt4", "Pt5"])))
print(len(mtemp["gex"]))

# Remove unused categories
for col in mtemp["gex"].obs.select_dtypes(include="category").columns:
    mtemp["gex"].obs[col] = (
        mtemp["gex"].obs[col].cat.remove_unused_categories()
    )

# Extract expression for the genes of interest
expr_df = mtemp["gex"][:, valid_genes].to_df()

# Add metadata
expr_df = pd.concat(
    [
        mtemp["gex"].obs[["clone_handle", "catpat", "Virus"]].reset_index(drop=True),
        expr_df.reset_index(drop=True),
    ],
    axis=1,
)

# Average expression per clone
clone_expr_df = (
    expr_df
    .groupby(["clone_handle", "catpat", "Virus"], observed=True)
    .mean()
    .reset_index()
)

print(len(clone_expr_df))
clone_expr_df.head(3)

In [ ]:
## zscore transformation
# copy to avoid modifying original
clone_expr_z = clone_expr_df.copy()

# select gene columns (exclude metadata)
gene_cols = valid_genes

# Z-score per gene across clones
clone_expr_z[gene_cols] = clone_expr_z[gene_cols].apply(
    lambda x: zscore(x, nan_policy="omit"),
    axis=0
)

clone_expr_z.head(2)

In [ ]:
## Logistic regression - coeefficient indicates positive association with EBV-specific clonotypes
X = clone_expr_z[gene_cols]
y = (clone_expr_z["Virus"] == "EBV").astype(int)

model = LogisticRegression(max_iter=1000)
model.fit(X, y)

coef = pd.Series(model.coef_[0], index=gene_cols)

# coef.sort_values(ascending=False).head(20)
heatmap_order = coef.reset_index(name="coef").sort_values("coef", ascending=False)["index"].tolist()

In [ ]:
clone_expr_z["Virus"].value_counts()

In [ ]:
# # Filter to X specificity
# X_specificity = "EBV"
# ebv_df = clone_expr_z[clone_expr_z["Virus"] == X_specificity]

## Copy the z-scores percloe
vir_df = clone_expr_z.copy()

# Ensure gene columns
gene_cols = valid_genes

# Compute mean Z-scores per virus irrespective of catpat
vir_summary = (
    vir_df
    .groupby("Virus")[gene_cols]
    .mean()
    .loc[["EBV", "CMV", "FLU-A", "SARS-COV-2", "HHV-1", "NWV", "VZV", "HAdV-C", "HHV-2", "HMPV"]]  # enforce order
)

# Ensure correct order (important)
heatmap_df = vir_summary.copy()

# Optional: sort genes by strongest difference
gene_order = res_df.sort_values("p_adj")["gene"].tolist()
global_order = gene_order

heatmap_df = heatmap_df[heatmap_order]

# Plot
plt.figure(figsize=(max(10, len(heatmap_order) * 0.35), 4))

sb.heatmap(
    heatmap_df,
    cmap="coolwarm",
    vmin=-2,
    vmax=2,
    center=0,
    linewidths=0.3,
    cbar_kws={"label": "Mean Z-score"}
)

plt.title(f"Genes positively associated with EBV-specific CD8 T-cell clonotypes\nRanked by logistic regression")
plt.xlabel("")
plt.ylabel("")
plt.yticks(rotation=0)
plt.xticks(rotation=90)

plt.tight_layout()
plt.savefig(f"../../05_plots/09_multimers/2026_7_CD8T_clonotype_EBV_genes.pdf")
plt.show()

# Heatmap - clone level differencecs - CSF-biased score

In [ ]:
DEG_biased = pd.read_csv("../../06_csv_outputs/DEG/CD8_CSF_biased_filtered_DEGs.csv")["names"].drop_duplicates().tolist()
print(len(DEG_biased))

In [ ]:
DEG_biased = [str(g) for g in DEG_biased]  # avoids np.str_ issues
valid_genes = list(set(DEG_biased).intersection(mdata["gex"].var_names))

len(valid_genes)

In [ ]:
missing = sorted(set(DEG_biased) - set(valid_genes))
print("Missing genes:", missing)

In [ ]:
# Work with clones of interest (those with an annotated antigen-specificity)
mtemp = mdata.copy()
print(len(mtemp["gex"]))
mu.pp.filter_obs(mtemp["gex"], "Virus", lambda x: (~x.isna()))
mu.pp.filter_obs(mtemp["gex"], "donor", lambda x: (~x.isin(["Pt4", "Pt5"])))
print(len(mtemp["gex"]))

# Remove unused categories
for col in mtemp["gex"].obs.select_dtypes(include="category").columns:
    mtemp["gex"].obs[col] = (
        mtemp["gex"].obs[col].cat.remove_unused_categories()
    )

# Extract expression for the genes of interest
expr_df = mtemp["gex"][:, valid_genes].to_df()

# Add metadata
expr_df = pd.concat(
    [
        mtemp["gex"].obs[["clone_handle", "catpat", "Virus"]].reset_index(drop=True),
        expr_df.reset_index(drop=True),
    ],
    axis=1,
)

# Average expression per clone
clone_expr_df = (
    expr_df
    .groupby(["clone_handle", "catpat", "Virus"], observed=True)
    .mean()
    .reset_index()
)

print(len(clone_expr_df))
clone_expr_df.head(3)

In [ ]:
from scipy.stats import zscore

# copy to avoid modifying original
clone_expr_z = clone_expr_df.copy()

# select gene columns (exclude metadata)
gene_cols = valid_genes

# Z-score per gene across clones
clone_expr_z[gene_cols] = clone_expr_z[gene_cols].apply(
    lambda x: zscore(x, nan_policy="omit"),
    axis=0
)

clone_expr_z.head(2)

In [ ]:
### Calculate statistics using a non-parametric test
gene_cols = valid_genes

results = []

# Loop over Virus groups
for virus, sub in clone_expr_z.groupby("Virus"):

    ctrl = sub[sub["catpat"] == "NIC"]
    rrms = sub[sub["catpat"] == "MS"]

    # Loop over genes
    for gene in gene_cols:

        x = ctrl[gene].dropna()
        y = rrms[gene].dropna()

        # skip if too few observations
        if len(x) < 2 or len(y) < 2:
            continue

        stat, p = mannwhitneyu(
            x, y,
            alternative="two-sided",
            method="exact"
        )

        results.append({
            "Virus": virus,
            "gene": gene,
            "U": stat,
            "p": p,
            "n_Control": len(x),
            "n_RRMS": len(y)
        })

# Build dataframe
res_df = pd.DataFrame(results)

# BH correction within each Virus
res_df["p_adj"] = (
    res_df.groupby("Virus")["p"]
    .transform(lambda p: multipletests(p, method="fdr_bh")[1])
)

# Sort output
res_df = res_df.sort_values(["Virus", "p_adj"]).reset_index(drop=True)

res_df.sort_values("p_adj").head(20)

In [ ]:
# Filter to X specificity
X_specificity = "EBV"
ebv_df = clone_expr_z[clone_expr_z["Virus"] == X_specificity]

# Ensure gene columns
gene_cols = valid_genes

# Compute mean Z-scores per catpat (Control vs RRMS)
ebv_summary = (
    ebv_df
    .groupby("catpat")[gene_cols]
    .mean()
    .loc[["NIC", "MS"]]  # enforce order
)

# Ensure correct order (important)
heatmap_df = ebv_summary.copy()

# Optional: sort genes by strongest difference
gene_order = res_df[res_df["Virus"] == X_specificity].sort_values("p_adj")["gene"].tolist()
global_order = gene_order

heatmap_df = heatmap_df[gene_order]

# Plot
plt.figure(figsize=(max(10, len(gene_order) * 0.25), 2))

sb.heatmap(
    heatmap_df,
    cmap="coolwarm",
    vmin=-1,
    vmax=1,
    center=0,
    linewidths=0.3,
    cbar_kws={"label": "Mean Z-score"}
)

plt.title(f"{X_specificity}-specific cells: Gene expression per clonotype")
plt.xlabel("")
plt.ylabel("")
plt.yticks(rotation=0)
plt.xticks(rotation=90)

plt.tight_layout()
plt.savefig(f"../../05_plots/09_multimers/2026_7_{X_specificity}_patient_specific_expression.pdf")
plt.show()

In [ ]:
# Filter to X specificity
X_specificity = "CMV"
ebv_df = clone_expr_z[clone_expr_z["Virus"] == X_specificity]

# Ensure gene columns
gene_cols = valid_genes

# Compute mean Z-scores per catpat (Control vs RRMS)
ebv_summary = (
    ebv_df
    .groupby("catpat")[gene_cols]
    .mean()
    .loc[["NIC", "MS"]]  # enforce order
)

# Ensure correct order (important)
heatmap_df = ebv_summary.copy()

# Optional: sort genes by strongest difference
# gene_order = res_df[res_df["Virus"] == X_specificity].sort_values("p_adj")["gene"].tolist()
gene_order = global_order

heatmap_df = heatmap_df[gene_order]

# Plot
plt.figure(figsize=(max(10, len(gene_order) * 0.25), 2))

sb.heatmap(
    heatmap_df,
    cmap="coolwarm",
    center=0,
    vmin=-1,
    vmax=1,
    linewidths=0.3,
    cbar_kws={"label": "Mean Z-score"}
)

plt.title(f"{X_specificity}-specific cells: Gene expression per clonotype")
plt.xlabel("")
plt.ylabel("")
plt.yticks(rotation=0)
plt.xticks(rotation=90)

plt.tight_layout()
plt.savefig(f"../../05_plots/09_multimers/2026_7_{X_specificity}_patient_specific_expression.pdf")
plt.show()

In [ ]:
# Filter to X specificity
X_specificity = "FLU-A"
ebv_df = clone_expr_z[clone_expr_z["Virus"] == X_specificity]

# Ensure gene columns
gene_cols = valid_genes

# Compute mean Z-scores per catpat (Control vs RRMS)
ebv_summary = (
    ebv_df
    .groupby("catpat")[gene_cols]
    .mean()
    .loc[["NIC", "MS"]]  # enforce order
)

# Ensure correct order (important)
heatmap_df = ebv_summary.copy()

# Optional: sort genes by strongest difference
# gene_order = res_df[res_df["Virus"] == X_specificity].sort_values("p_adj")["gene"].tolist()
gene_order = global_order

heatmap_df = heatmap_df[gene_order]

# Plot
plt.figure(figsize=(max(10, len(gene_order) * 0.25), 2))

sb.heatmap(
    heatmap_df,
    cmap="coolwarm",
    center=0,
    vmin=-1,
    vmax=1,
    linewidths=0.3,
    cbar_kws={"label": "Mean Z-score"}
)

plt.title(f"{X_specificity}-specific cells: Gene expression per clonotype")
plt.xlabel("")
plt.ylabel("")
plt.yticks(rotation=0)
plt.xticks(rotation=90)

plt.tight_layout()
plt.savefig(f"../../05_plots/09_multimers/2026_7_{X_specificity}_patient_specific_expression.pdf")
plt.show()

In [ ]:
# Filter to X specificity
X_specificity = "SARS-COV-2"
ebv_df = clone_expr_z[clone_expr_z["Virus"] == X_specificity]

# Ensure gene columns
gene_cols = valid_genes

# Compute mean Z-scores per catpat (Control vs RRMS)
ebv_summary = (
    ebv_df
    .groupby("catpat")[gene_cols]
    .mean()
    .loc[["NIC", "MS"]]  # enforce order
)

# Ensure correct order (important)
heatmap_df = ebv_summary.copy()

# Optional: sort genes by strongest difference
# gene_order = res_df[res_df["Virus"] == X_specificity].sort_values("p_adj")["gene"].tolist()
gene_order = global_order

heatmap_df = heatmap_df[gene_order]

# Plot
plt.figure(figsize=(max(10, len(gene_order) * 0.25), 2))

sb.heatmap(
    heatmap_df,
    cmap="coolwarm",
    vmin=-1,
    vmax=1,
    center=0,
    linewidths=0.3,
    cbar_kws={"label": "Mean Z-score"}
)

plt.title(f"{X_specificity}-specific cells: Gene expression per clonotype")
plt.xlabel("")
plt.ylabel("")
plt.yticks(rotation=0)
plt.xticks(rotation=90)

plt.tight_layout()
plt.savefig(f"../../05_plots/09_multimers/2026_7_{X_specificity}_patient_specific_expression.pdf")
plt.show()